**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# AIMS - Pipeline reale e diagnostica Sim2Real

Notebook riordinato per la predizione dell'errore di puntamento del Sardinia Radio Telescope.

Obiettivi pratici:
- usare i dati reali in modo coerente con la loro natura: serie termica densa + pochi cross-scan supervisionati;
- evitare leakage temporale e finestre che attraversano buchi;
- confrontare baseline robuste prima di addestrare reti neurali;
- ridurre la dimensionalita' della sequenza prima della GRU, perche' 502 target sono pochi rispetto a `60 x 295` input;
- lasciare il FEM come diagnostica/prior, non come predittore zero-shot affidabile.


In [ ]:
from pathlib import Path
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV, HuberRegressor, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SEQ_LEN = 60                 # 60 two-minute samples span approximately two hours.
EXPECTED_STEP_SEC = 120      
MIN_STEP_SEC = 60             
MAX_GAP_SEC = 180            
MAX_SENSOR_RATE_C = 5.0       
USE_RESAMPLED_WINDOWS = True  
TARGET = "XEl Offset"        
SPEC_ARCSEC = 5.0             # Historical reference threshold; receiver-specific requirements differ.
INCLUDE_CURRENT_TEMP = True  


PROJECT_SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents, Path("/content/metrologyml"), Path.home() / "Downloads"]
DATA_SUBDIRS = [Path("."), Path("data/raw/real"), Path("data/raw/real/optional"), Path("data/raw/control"), Path("data/raw/fem"), Path("data/diagnostics/control"), Path("data/diagnostics/fem"), Path("data/diagnostics/history")]
DATA_DIRS = list(dict.fromkeys(root / subdir for root in PROJECT_SEARCH_ROOTS for subdir in DATA_SUBDIRS))
REAL_CANDIDATES = ["EffectiveDataset_Tseries_with_azimuth.csv", "EffectiveDataset_Tseries_total.csv", "EffectiveDataset_Tseries (2).xlsx", "EffectiveDataset_Tseries.xlsx", "EffectiveDataset_Tseries.csv"]
FEM_CANDIDATES = ["Dataset (1).xlsx", "dataset_multistep_jul_aug.csv", "Dataset.xlsx"]
POINTING_CANDIDATES = ["pointing_offsets_real_azimuth.csv"]
USE_POINTING_TARGETS_WITH_AZIMUTH = False  
MAX_ABS_XEL_FOR_TRAIN = 60.0    
MAX_ABS_EL_FOR_TRAIN = 180.0    
FILTER_MAINTENANCE_BY_NAME = False  # Maintenance scans are not rejected solely on their project label.
POINTING_CLEANING_MODE = "extreme_only"  
STABLE_JUMP_WINDOW_MIN = 180
STABLE_DXEL_JUMP_ARCSEC = 15.0
STABLE_DEL_JUMP_ARCSEC = 20.0

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


In [ ]:
def find_data_file(candidates, data_dirs=DATA_DIRS):
    for d in data_dirs:
        for name in candidates:
            p = d / name
            if p.exists():
                return p
    raise FileNotFoundError(f"No file found among: {candidates}")


def read_table(path: Path) -> pd.DataFrame:
    """Read CSV/XLSX, with a direct openpyxl fallback for version conflicts."""
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xlsm", ".xls"]:
        try:
            return pd.read_excel(path, engine="openpyxl")
        except ImportError:
            from openpyxl import load_workbook
            wb = load_workbook(path, read_only=True, data_only=True)
            ws = wb.active
            rows = ws.iter_rows(values_only=True)
            header = list(next(rows))
            data = list(rows)
            wb.close()
            return pd.DataFrame(data, columns=header)
    raise ValueError(f"Unsupported format: {path}")


def normalize_sensor_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize sensor-column names."""
    mapping = {}
    for col in df.columns:
        m = re.match(r"^Sensors_\s*(\d+)$", str(col))
        if m:
            mapping[col] = f"Sensors_{int(m.group(1)):03d}"
    return df.rename(columns=mapping)


def get_sensor_cols(df: pd.DataFrame):
    cols = [c for c in df.columns if re.match(r"^Sensors_\d{3}$", str(c))]
    return sorted(cols, key=lambda c: int(c.split("_")[1]))


def ensure_datetime(df, col):
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def xel_from_fem(df):
    """Project Accuracy_1 to XEL using cos(Elevation)."""
    out = df.copy()
    out["XEL"] = out["Accuracy_1"] * np.cos(np.deg2rad(out["Elevation"]))
    return out


In [ ]:
real_path = find_data_file(REAL_CANDIDATES)
print('Real dataset:', real_path)

real_raw = read_table(real_path)
real_raw = normalize_sensor_columns(real_raw)
real_raw = ensure_datetime(real_raw, "Timestamp")
real_raw = real_raw.sort_values("Timestamp").reset_index(drop=True)

sensor_cols = get_sensor_cols(real_raw)
print(f"Real rows: {len(real_raw):,} | Sensors: {len(sensor_cols)}")
print('Time range:', real_raw["Timestamp"].min(), "->", real_raw["Timestamp"].max())
print('Valid XEL targets:', real_raw["XEl Offset"].notna().sum(), "| EL:", real_raw["El Offset"].notna().sum())
print('Valid elevations:', real_raw["SRT Elevation Ange[DEG]"].notna().sum())
if "Azimuth" in real_raw.columns:
    print('Azimuth available:', real_raw["Azimuth"].notna().sum(), "| range:", real_raw["Azimuth"].min(), "->", real_raw["Azimuth"].max())
else:
    print('Azimuth is not present in the real dataset.')

real_raw[["SRT Elevation Ange[DEG]", "El Offset", "XEl Offset"]].describe()


In [ ]:
pointing_targets = None
if USE_POINTING_TARGETS_WITH_AZIMUTH:
    try:
        pointing_path = find_data_file(POINTING_CANDIDATES)
        print('Pointing and azimuth dataset:', pointing_path)
        pointing_raw = read_table(pointing_path)
        pointing_raw = pointing_raw.rename(columns={
            "Time": "Timestamp",
            "Elevation": "PointingElevation",
            "Offset_XEL_arcsec": "XEl Offset",
            "Offset_EL_arcsec": "El Offset",
        })
        pointing_raw["Timestamp"] = pd.to_datetime(pointing_raw["Timestamp"], errors="coerce")
        pointing_raw = pointing_raw.dropna(subset=["Timestamp", "Azimuth", "PointingElevation", "XEl Offset"]).copy()
        pointing_raw = pointing_raw.sort_values("Timestamp").reset_index(drop=True)


        tmin, tmax = real_raw["Timestamp"].min(), real_raw["Timestamp"].max()
        pointing_raw = pointing_raw[(pointing_raw["Timestamp"] >= tmin) & (pointing_raw["Timestamp"] <= tmax)].copy()

        pointing_raw["abs_xel"] = pointing_raw["XEl Offset"].abs()
        pointing_raw["abs_el"] = pointing_raw["El Offset"].abs()
        pointing_raw["flag_huge_xel"] = pointing_raw["abs_xel"] > MAX_ABS_XEL_FOR_TRAIN
        pointing_raw["flag_huge_el"] = pointing_raw["abs_el"] > MAX_ABS_EL_FOR_TRAIN
        pointing_raw["flag_maintenance"] = pointing_raw["Filename"].astype(str).str.contains("Maintenance", case=False, na=False)
        pointing_raw["flag_extreme_outlier"] = pointing_raw["flag_huge_xel"] | pointing_raw["flag_huge_el"]
        pointing_raw["flag_suspicious"] = pointing_raw["flag_extreme_outlier"] | (pointing_raw["flag_maintenance"] & FILTER_MAINTENANCE_BY_NAME)

        print('Pointing rows within the telemetry period:', len(pointing_raw))
        print('Extreme outliers above the threshold:', int(pointing_raw["flag_extreme_outlier"].sum()))
        print('Suspect/procedural events excluded:', int(pointing_raw["flag_suspicious"].sum()))
        print(" - huge XEL:", int(pointing_raw["flag_huge_xel"].sum()))
        print(" - huge EL :", int(pointing_raw["flag_huge_el"].sum()))
        print(' - maintenance (diagnostic only):', int(pointing_raw["flag_maintenance"].sum()))
        display(pointing_raw[["Azimuth", "PointingElevation", "XEl Offset", "El Offset", "abs_xel", "abs_el"]].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

        print('Largest suspect XEL offsets')
        display(pointing_raw.sort_values("abs_xel", ascending=False).head(20)[
            ["Filename", "Timestamp", "Azimuth", "PointingElevation", "XEl Offset", "El Offset", "flag_suspicious"]
        ])


        # extreme_only removes only threshold-defined extreme outliers.
        # stable_only also removes nearby jumps; it does not reproduce the original manual selection.
        pointing_clean = pointing_raw[~pointing_raw["flag_suspicious"]].copy().sort_values("Timestamp").reset_index(drop=True)
        if POINTING_CLEANING_MODE == "stable_only":
            pointing_clean["dt_prev_min"] = pointing_clean["Timestamp"].diff().dt.total_seconds() / 60
            pointing_clean["dt_next_min"] = (pointing_clean["Timestamp"].shift(-1) - pointing_clean["Timestamp"]).dt.total_seconds() / 60
            pointing_clean["dprev_xel"] = pointing_clean["XEl Offset"].diff().abs()
            pointing_clean["dnext_xel"] = pointing_clean["XEl Offset"].diff(-1).abs()
            pointing_clean["dprev_el"] = pointing_clean["El Offset"].diff().abs()
            pointing_clean["dnext_el"] = pointing_clean["El Offset"].diff(-1).abs()
            near_prev = pointing_clean["dt_prev_min"].fillna(np.inf) <= STABLE_JUMP_WINDOW_MIN
            near_next = pointing_clean["dt_next_min"].fillna(np.inf) <= STABLE_JUMP_WINDOW_MIN
            stable_jump_flag = (
                (near_prev & ((pointing_clean["dprev_xel"] > STABLE_DXEL_JUMP_ARCSEC) | (pointing_clean["dprev_el"] > STABLE_DEL_JUMP_ARCSEC))) |
                (near_next & ((pointing_clean["dnext_xel"] > STABLE_DXEL_JUMP_ARCSEC) | (pointing_clean["dnext_el"] > STABLE_DEL_JUMP_ARCSEC)))
            )
            print('Stable-only: local jumps removed:', int(stable_jump_flag.sum()))
            pointing_clean = pointing_clean[~stable_jump_flag].copy()
        elif POINTING_CLEANING_MODE != "extreme_only":
            raise ValueError(f"Unrecognized POINTING_CLEANING_MODE: {POINTING_CLEANING_MODE}")

        pointing_targets = pointing_clean.rename(columns={"PointingElevation": "Elevation"})
        pointing_targets = pointing_targets[["Timestamp", "Azimuth", "Elevation", "XEl Offset", "El Offset", "Filename", "abs_xel", "abs_el"]]
        print("Cleaning mode:", POINTING_CLEANING_MODE)
        print('Clean pointing targets available:', len(pointing_targets))
    except FileNotFoundError:
        print('Pointing/azimuth file not found; using targets embedded in the thermal dataset.')
        pointing_targets = None


In [ ]:
if "pointing_raw" in globals():
    after_extreme = pointing_raw[~pointing_raw["flag_extreme_outlier"]].copy().sort_values("Timestamp").reset_index(drop=True)
    print("Totale:", len(pointing_raw))
    print('After extreme-outlier removal:', len(after_extreme), '(approximately 1485 expected)')
    print('Removals needed to reach 1166:', len(after_extreme) - 1166)

    q = after_extreme.copy()
    q["dt_prev_min"] = q["Timestamp"].diff().dt.total_seconds() / 60
    q["dt_next_min"] = (q["Timestamp"].shift(-1) - q["Timestamp"]).dt.total_seconds() / 60
    for col in ["XEl Offset", "El Offset"]:
        q[f"dprev_{col}"] = q[col].diff().abs()
        q[f"dnext_{col}"] = q[col].diff(-1).abs()


    candidate_rows = []
    for x_abs in [15, 20, 25, 30, 35, 40, 45, 50]:
        for el_abs in [30, 40, 50, 60, 80, 100, 120]:
            flag = (q["XEl Offset"].abs() > x_abs) | (q["El Offset"].abs() > el_abs)
            candidate_rows.append({"criterion": f"abs: |XEL|>{x_abs} or |EL|>{el_abs}", "removed": int(flag.sum()), "remaining": int((~flag).sum())})

    for x_jump in [15, 20, 25, 30, 40, 50]:
        for el_jump in [20, 30, 40, 50, 60, 80]:
            near_prev = q["dt_prev_min"].fillna(np.inf) <= 180
            near_next = q["dt_next_min"].fillna(np.inf) <= 180
            flag_prev = near_prev & ((q["dprev_XEl Offset"] > x_jump) | (q["dprev_El Offset"] > el_jump))
            flag_next = near_next & ((q["dnext_XEl Offset"] > x_jump) | (q["dnext_El Offset"] > el_jump))
            flag = flag_prev | flag_next
            candidate_rows.append({"criterion": f"local jump<=3h: dXEL>{x_jump} or dEL>{el_jump}", "removed": int(flag.sum()), "remaining": int((~flag).sum())})

    candidates_df = pd.DataFrame(candidate_rows)
    candidates_df["distance_from_319"] = (candidates_df["removed"] - 319).abs()
    print('Candidate criteria closest to the 319 historical jumps/residuals')
    display(candidates_df.sort_values("distance_from_319").head(25))

    print('Distribution after removing extreme outliers only')
    display(after_extreme[["Azimuth", "PointingElevation", "XEl Offset", "El Offset", "abs_xel", "abs_el"]].describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]))


    APPLY_APPROX_RESIDUAL_FILTER = False
    if APPLY_APPROX_RESIDUAL_FILTER:

        residual_flag = (after_extreme["XEl Offset"].abs() > 30) | (after_extreme["El Offset"].abs() > 60)
        pointing_targets = after_extreme[~residual_flag].rename(columns={"PointingElevation": "Elevation"})[
            ["Timestamp", "Azimuth", "Elevation", "XEl Offset", "El Offset", "Filename", "abs_xel", "abs_el"]
        ].copy()
        print('Pointing targets after the approximate residual filter:', len(pointing_targets))
else:
    print('pointing_raw is unavailable; run cell 2.b first')


In [ ]:
def audit_real_dataset(df, sensor_cols):
    print('Missing sensors:', int(df[sensor_cols].isna().sum().sum()))
    dt = df["Timestamp"].diff().dt.total_seconds().dropna()
    print('Temperature sampling interval [s]')
    print(dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    target_df = df[df[TARGET].notna()].copy()
    target_dt = target_df["Timestamp"].diff().dt.total_seconds().dropna()
    print('\nValid cross-scan targets:', len(target_df))
    if len(target_dt):
        print('Time between targets [s]')
        print(target_dt.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    flat = df[sensor_cols].to_numpy(dtype="float32").ravel()
    flat = flat[np.isfinite(flat)]
    print('\nReal temperatures: mean/std/min/max', np.mean(flat), np.std(flat), np.min(flat), np.max(flat))

audit_real_dataset(real_raw, sensor_cols)


In [ ]:
def _impute_sensor_frame(sensor_df, impute_limit=3):
    # Limited backward filling is offline-only and needs an online availability audit.
    return sensor_df.astype("float32").ffill(limit=impute_limit).bfill(limit=1)


def build_supervised_windows_raw(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    min_step_sec=MIN_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build consecutive-row windows, rejecting duplicate timestamps."""
    df = df.sort_values("Timestamp").reset_index(drop=True).copy()
    df[sensor_cols] = _impute_sensor_frame(df[sensor_cols], impute_limit=impute_limit)

    if target_table is not None:
        print('Azimuth in target_table is fully supported by resampled_grid; raw_rows uses targets from the thermal dataset.')
    target_mask = df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()
    target_positions = np.flatnonzero(target_mask.to_numpy())

    X, elev, y, times = [], [], [], []
    skip = {"too_early": 0, "time_gap": 0, "too_close": 0, "nan": 0, "thermal_jump": 0}

    for pos in target_positions:
        end = pos + 1 if include_current else pos
        start = end - seq_len
        if start < 0:
            skip["too_early"] += 1
            continue

        window = df.iloc[start:end]
        dts = window["Timestamp"].diff().dt.total_seconds().dropna()
        if len(dts) != seq_len - 1 or (dts > max_gap_sec).any():
            skip["time_gap"] += 1
            continue
        if (dts < min_step_sec).any():
            skip["too_close"] += 1
            continue

        values = window[sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(df.loc[pos, "SRT Elevation Ange[DEG]"]))
        y.append(float(df.loc[pos, target_col]))
        times.append(df.loc[pos, "Timestamp"])

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Target": y})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows_resampled(
    df,
    sensor_cols,
    target_col=TARGET,
    target_table=None,
    seq_len=SEQ_LEN,
    expected_step_sec=EXPECTED_STEP_SEC,
    max_gap_sec=MAX_GAP_SEC,
    max_sensor_rate_c=MAX_SENSOR_RATE_C,
    include_current=INCLUDE_CURRENT_TEMP,
    impute_limit=3,
):
    """Build two-minute windows, aggregating duplicate rows and checking gaps and thermal jumps."""
    df = df.sort_values("Timestamp").copy()
    if target_table is not None:
        target_df = target_table.rename(columns={"Elevation": "SRT Elevation Ange[DEG]"}).copy()
    else:
        target_df = df[df[target_col].notna() & df["SRT Elevation Ange[DEG]"].notna()].copy()


    sensor_ts = df[["Timestamp"] + sensor_cols].dropna(subset=["Timestamp"]).copy()
    sensor_ts["GridTime"] = sensor_ts["Timestamp"].dt.floor(f"{expected_step_sec}s")
    grid = sensor_ts.groupby("GridTime", sort=True)[sensor_cols].mean()
    grid = _impute_sensor_frame(grid, impute_limit=impute_limit)

    X, elev, azim, y, times, filenames = [], [], [], [], [], []
    skip = {"too_early": 0, "missing_grid": 0, "nan": 0, "thermal_jump": 0}
    step = pd.Timedelta(seconds=expected_step_sec)

    for _, row in target_df.iterrows():
        target_time = row["Timestamp"]
        end_time = target_time.floor(f"{expected_step_sec}s") if include_current else (target_time.floor(f"{expected_step_sec}s") - step)
        expected_index = pd.date_range(end=end_time, periods=seq_len, freq=step)

        if len(grid) == 0 or expected_index[0] < grid.index.min():
            skip["too_early"] += 1
            continue

        # Reject incomplete grids rather than filling long resampling gaps.
        if not expected_index.isin(grid.index).all():
            skip["missing_grid"] += 1
            continue

        values = grid.loc[expected_index, sensor_cols].to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan"] += 1
            continue
        if np.abs(np.diff(values, axis=0)).max() > max_sensor_rate_c:
            skip["thermal_jump"] += 1
            continue

        X.append(values)
        elev.append(float(row["SRT Elevation Ange[DEG]"]))
        azim.append(float(row["Azimuth"]) if "Azimuth" in row and pd.notna(row["Azimuth"]) else np.nan)
        y.append(float(row[target_col]))
        times.append(target_time)
        filenames.append(row["Filename"] if "Filename" in row else None)

    meta = pd.DataFrame({"Timestamp": times, "Elevation": elev, "Azimuth": azim, "Target": y, "Filename": filenames})
    return np.asarray(X, dtype="float32"), np.asarray(elev, dtype="float32").reshape(-1, 1), np.asarray(y, dtype="float32"), meta, skip


def build_supervised_windows(df, sensor_cols, **kwargs):
    if USE_RESAMPLED_WINDOWS:
        return build_supervised_windows_resampled(df, sensor_cols, **kwargs)
    return build_supervised_windows_raw(df, sensor_cols, **kwargs)

target_source = pointing_targets if USE_POINTING_TARGETS_WITH_AZIMUTH and "pointing_targets" in globals() and pointing_targets is not None else None
X_seq, X_elev, y, meta, skip = build_supervised_windows(real_raw, sensor_cols, target_table=target_source)
print('Window mode:', "resampled_grid" if USE_RESAMPLED_WINDOWS else "raw_rows")
print('Target source:', "pointing_offsets_real_azimuth" if target_source is not None else 'thermal_dataset')
print('Windows with azimuth:', int(meta["Azimuth"].notna().sum()) if "Azimuth" in meta else 0)
print('Valid supervised samples:', len(y))
print('Rejected by timing/jump checks:', skip)
print("Shape X_seq:", X_seq.shape, "X_elev:", X_elev.shape, "y:", y.shape)
meta.describe(include="all")


In [ ]:
def window_quality_report(X, meta):
    if len(X) == 0:
        print('No valid windows.')
        return pd.DataFrame()
    diffs = np.diff(X, axis=1)
    q = pd.DataFrame({
        "Timestamp": meta["Timestamp"].to_numpy(),
        "Elevation": meta["Elevation"].to_numpy(),
        "Target": meta["Target"].to_numpy(),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "current_mean": X[:, -1, :].mean(axis=1),
        "trend_mean": (X[:, -1, :] - X[:, 0, :]).mean(axis=1),
        "abs_rate_mean": np.abs(diffs).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diffs).max(axis=(1, 2)),
    })
    print(q[["window_mean", "window_std", "current_mean", "trend_mean", "abs_rate_mean", "abs_rate_max"]].describe().to_string())
    suspect = q[(q["abs_rate_max"] > MAX_SENSOR_RATE_C) | (q["abs_rate_mean"] > 0.5)]
    print('Remaining suspect windows:', len(suspect))
    if len(suspect):
        display(suspect.sort_values("abs_rate_max", ascending=False).head(20))
    return q

quality_df = window_quality_report(X_seq, meta)

In [ ]:
def chronological_split(meta, train_frac=0.60, val_frac=0.20):
    order = np.argsort(meta["Timestamp"].to_numpy())
    n = len(order)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_idx = order[:n_train]
    val_idx = order[n_train:n_train + n_val]
    test_idx = order[n_train + n_val:]
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = chronological_split(meta)
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))
print("Train:", meta.iloc[train_idx]["Timestamp"].min(), "->", meta.iloc[train_idx]["Timestamp"].max())
print("Val  :", meta.iloc[val_idx]["Timestamp"].min(), "->", meta.iloc[val_idx]["Timestamp"].max())
print("Test :", meta.iloc[test_idx]["Timestamp"].min(), "->", meta.iloc[test_idx]["Timestamp"].max())

plt.figure(figsize=(10, 4))
plt.plot(meta["Timestamp"].to_numpy(), meta["Target"].to_numpy(), ".", alpha=0.75)
plt.axvspan(meta.iloc[train_idx]["Timestamp"].min(), meta.iloc[train_idx]["Timestamp"].max(), alpha=0.08, label="train")
plt.axvspan(meta.iloc[val_idx]["Timestamp"].min(), meta.iloc[val_idx]["Timestamp"].max(), alpha=0.08, label="val")
plt.axvspan(meta.iloc[test_idx]["Timestamp"].min(), meta.iloc[test_idx]["Timestamp"].max(), alpha=0.08, label="test")
plt.title(f"Observed target {TARGET} and chronological split")
plt.ylabel("arcsec")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
def scalar_physical_features(X, elev):
    current = X[:, -1, :]
    start = X[:, 0, :]
    mean_time = X.mean(axis=1)
    std_time = X.std(axis=1)
    diff = np.diff(X, axis=1)
    trend = current - start

    feats = pd.DataFrame({
        "elevation": elev.ravel(),
        "sin_el": np.sin(np.deg2rad(elev.ravel())),
        "cos_el": np.cos(np.deg2rad(elev.ravel())),
        "current_mean": current.mean(axis=1),
        "current_std_spatial": current.std(axis=1),
        "current_min": current.min(axis=1),
        "current_max": current.max(axis=1),
        "current_range": current.max(axis=1) - current.min(axis=1),
        "window_mean": X.mean(axis=(1, 2)),
        "window_std": X.std(axis=(1, 2)),
        "mean_sensor_std_time": std_time.mean(axis=1),
        "trend_mean": trend.mean(axis=1),
        "trend_std": trend.std(axis=1),
        "trend_range": trend.max(axis=1) - trend.min(axis=1),
        "abs_rate_mean": np.abs(diff).mean(axis=(1, 2)),
        "abs_rate_max": np.abs(diff).max(axis=(1, 2)),
    })
    return feats

def temporal_context_features(meta_df):
    """Build cyclic time features as seasonal and diurnal proxies."""
    ts = pd.to_datetime(meta_df["Timestamp"])
    hour = ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0
    doy = ts.dt.dayofyear.astype(float)
    out = pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour / 24.0),
        "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25),
        "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        "month": ts.dt.month.astype(float),
    }).reset_index(drop=True)
    if "Azimuth" in meta_df.columns and meta_df["Azimuth"].notna().any():
        az = pd.to_numeric(meta_df["Azimuth"], errors="coerce").fillna(meta_df["Azimuth"].median())
        out["az_sin"] = np.sin(np.deg2rad(az))
        out["az_cos"] = np.cos(np.deg2rad(az))
    return out

class FeatureBuilder:
    """Fit scalar features and current-state/trend PCA on training data only."""
    def __init__(self, n_pca_current=8, n_pca_trend=8):
        self.n_pca_current = n_pca_current
        self.n_pca_trend = n_pca_trend
        self.scaler_current = StandardScaler()
        self.scaler_trend = StandardScaler()
        self.pca_current = PCA(n_components=n_pca_current, random_state=RANDOM_SEED)
        self.pca_trend = PCA(n_components=n_pca_trend, random_state=RANDOM_SEED)

    def fit(self, X, elev):
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        self.pca_current.fit(self.scaler_current.fit_transform(current))
        self.pca_trend.fit(self.scaler_trend.fit_transform(trend))
        return self

    def transform(self, X, elev, meta_df=None):
        base = scalar_physical_features(X, elev).reset_index(drop=True)
        if meta_df is not None:
            base = pd.concat([base, temporal_context_features(meta_df)], axis=1)
        current = X[:, -1, :]
        trend = X[:, -1, :] - X[:, 0, :]
        pc_cur = self.pca_current.transform(self.scaler_current.transform(current))
        pc_tr = self.pca_trend.transform(self.scaler_trend.transform(trend))
        for i in range(pc_cur.shape[1]):
            base[f"pc_current_{i+1:02d}"] = pc_cur[:, i]
        for i in range(pc_tr.shape[1]):
            base[f"pc_trend_{i+1:02d}"] = pc_tr[:, i]
        return base

fb = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq[train_idx], X_elev[train_idx])
F_train = fb.transform(X_seq[train_idx], X_elev[train_idx], meta.iloc[train_idx])
F_val = fb.transform(X_seq[val_idx], X_elev[val_idx], meta.iloc[val_idx])
F_test = fb.transform(X_seq[test_idx], X_elev[test_idx], meta.iloc[test_idx])

y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(F_train.shape)
F_train.head()


nan_report = pd.DataFrame({
    "train_nan": F_train.isna().sum(),
    "val_nan": F_val.isna().sum(),
    "test_nan": F_test.isna().sum(),
})
nan_report = nan_report[(nan_report["train_nan"] > 0) | (nan_report["val_nan"] > 0) | (nan_report["test_nan"] > 0)]
if len(nan_report):
    print('Features containing NaN before imputation:')
    display(nan_report.sort_values(["val_nan", "test_nan", "train_nan"], ascending=False))
else:
    print('No NaN values in tabular features.')


In [ ]:
def eval_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

models = {
    "median_baseline": DummyRegressor(strategy="median"),
    "ridge_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))
    ]),
    "huber_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=2000))
    ]),
    "elasticnet_physical": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("model", ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.8, 0.95], alphas=np.logspace(-3, 2, 30), cv=5, random_state=RANDOM_SEED, max_iter=20000))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1
        )),
    ]),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.03, l2_regularization=0.1, min_samples_leaf=12, random_state=RANDOM_SEED
    ),
}

results = []
val_predictions = {}
test_predictions = {}

for name, model in models.items():
    m = clone(model)
    m.fit(F_train, y_train)
    pred_val = m.predict(F_val)
    pred_test = m.predict(F_test)
    val_predictions[name] = pred_val
    test_predictions[name] = pred_test
    row = eval_predictions(name + " / val", y_val, pred_val)
    row_test = eval_predictions(name + " / test", y_test, pred_test)
    results.extend([row, row_test])

results_df = pd.DataFrame(results).sort_values(["MAE"])
results_df


In [ ]:
az_cols = [c for c in F_train.columns if c.startswith("az_")]
if not az_cols:
    print("No azimuth features found; check that the dataset contains 'Azimuth' and meta retains it.")
else:
    print('Azimuth features:', az_cols)

    ablation_models = {
        "median_baseline": DummyRegressor(strategy="median"),
        "ridge_physical": Pipeline([
            ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
            ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))
        ]),
        "huber_physical": Pipeline([
            ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
            ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=2000))
        ]),
        "elasticnet_physical": Pipeline([
            ("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
            ("model", ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.8, 0.95], alphas=np.logspace(-3, 2, 30), cv=5, random_state=RANDOM_SEED, max_iter=20000))
        ]),
        "random_forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1
        ),
    }

    F_train_noaz = F_train.drop(columns=az_cols)
    F_val_noaz = F_val.drop(columns=az_cols)
    F_test_noaz = F_test.drop(columns=az_cols)

    rows = []
    for feature_set, Xtr, Xv, Xte in [
        ("with_azimuth", F_train, F_val, F_test),
        ("without_azimuth", F_train_noaz, F_val_noaz, F_test_noaz),
    ]:
        for name, model in ablation_models.items():
            m = clone(model)
            m.fit(Xtr, y_train)
            pv = m.predict(Xv)
            pt = m.predict(Xte)
            rv = eval_predictions(f"{name} / val", y_val, pv)
            rt = eval_predictions(f"{name} / test", y_test, pt)
            rv["feature_set"] = feature_set
            rt["feature_set"] = feature_set
            rows.extend([rv, rt])

    azimuth_ablation_df = pd.DataFrame(rows)
    display(azimuth_ablation_df.sort_values(["model", "feature_set"]))

    # Positive delta MAE indicates an azimuth-feature improvement.
    test_ab = azimuth_ablation_df[azimuth_ablation_df["model"].str.endswith("/ test")].copy()
    pivot = test_ab.pivot(index="model", columns="feature_set", values="MAE")
    if {"with_azimuth", "without_azimuth"}.issubset(pivot.columns):
        pivot["delta_MAE_without_minus_with"] = pivot["without_azimuth"] - pivot["with_azimuth"]
        display(pivot.sort_values("delta_MAE_without_minus_with", ascending=False))


In [ ]:
best_name = results_df[results_df["model"].str.endswith("/ val")].iloc[0]["model"].replace(" / val", "")
print('Best model on validation:', best_name)

pred_test = test_predictions[best_name]
resid = y_test - pred_test

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].plot(meta.iloc[test_idx]["Timestamp"].to_numpy(), y_test, label='observed', marker="o")
axs[0].plot(meta.iloc[test_idx]["Timestamp"].to_numpy(), pred_test, label='predicted', marker="x")
axs[0].set_title(f"Chronological test - {best_name}")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].scatter(y_test, pred_test, alpha=0.8)
lo = min(y_test.min(), pred_test.min())
hi = max(y_test.max(), pred_test.max())
axs[1].plot([lo, hi], [lo, hi], "k--")
axs[1].set_xlabel('observed [arcsec]')
axs[1].set_ylabel('predicted [arcsec]')
axs[1].set_title('Predicted versus observed')
axs[1].grid(True, alpha=0.3)

axs[2].hist(resid, bins=20, alpha=0.75)
axs[2].axvline(0, color="k", linestyle="--")
axs[2].set_title(f"Residuals | MAE={mean_absolute_error(y_test, pred_test):.2f} arcsec")
axs[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def local_repeatability_check(meta, max_dt_hours=6, max_delev_deg=3.0):
    rows = []
    m = meta.sort_values("Timestamp").reset_index(drop=True)
    for i in range(len(m)):
        t0 = m.loc[i, "Timestamp"]
        e0 = m.loc[i, "Elevation"]
        y0 = m.loc[i, "Target"]
        later = m.iloc[i+1:]
        if later.empty:
            continue
        dt_h = (later["Timestamp"] - t0).dt.total_seconds() / 3600
        cand = later[(dt_h > 0) & (dt_h <= max_dt_hours) & ((later["Elevation"] - e0).abs() <= max_delev_deg)]
        for _, r in cand.iterrows():
            rows.append({
                "dt_h": (r["Timestamp"] - t0).total_seconds() / 3600,
                "delev": abs(r["Elevation"] - e0),
                "abs_dtarget": abs(r["Target"] - y0),
            })
    return pd.DataFrame(rows)

repeat_df = local_repeatability_check(meta)
print('Nearby pairs found:', len(repeat_df))
if len(repeat_df):
    display(repeat_df.describe(percentiles=[0.5, 0.75, 0.9, 0.95]))
    plt.figure(figsize=(6,4))
    plt.scatter(repeat_df["dt_h"], repeat_df["abs_dtarget"], alpha=0.6)
    plt.xlabel('Time difference [h]')
    plt.ylabel("|Delta target| [arcsec]")
    plt.title('Local repeatability of the real target')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Few nearby scans; consecutive calibrations are needed to estimate the noise floor reliably.')


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
except Exception as e:
    TF_AVAILABLE = False
    print('TensorFlow is unavailable:', e)

class SequencePCA:
    def __init__(self, n_components=16):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=RANDOM_SEED)

    def fit(self, X):
        flat = X.reshape(-1, X.shape[-1])
        self.pca.fit(self.scaler.fit_transform(flat))
        return self

    def transform(self, X):
        n, t, f = X.shape
        flat = X.reshape(-1, f)
        z = self.pca.transform(self.scaler.transform(flat))
        return z.reshape(n, t, self.n_components).astype("float32")

if TF_AVAILABLE:
    seq_pca = SequencePCA(n_components=min(16, len(train_idx)-1)).fit(X_seq[train_idx])
    Z_train = seq_pca.transform(X_seq[train_idx])
    Z_val = seq_pca.transform(X_seq[val_idx])
    Z_test = seq_pca.transform(X_seq[test_idx])

    elev_scaler = StandardScaler().fit(X_elev[train_idx])
    E_train = elev_scaler.transform(X_elev[train_idx]).astype("float32")
    E_val = elev_scaler.transform(X_elev[val_idx]).astype("float32")
    E_test = elev_scaler.transform(X_elev[test_idx]).astype("float32")

    y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    ys_train = y_scaler.transform(y_train.reshape(-1, 1)).astype("float32")
    ys_val = y_scaler.transform(y_val.reshape(-1, 1)).astype("float32")

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_SEED)

    seq_in = keras.Input(shape=(SEQ_LEN, Z_train.shape[-1]), name="thermal_pca_history")
    elev_in = keras.Input(shape=(1,), name="elevation")

    x = layers.GRU(32, return_sequences=False, dropout=0.25, recurrent_dropout=0.0)(seq_in)
    x = layers.Dense(16, activation="gelu")(x)
    e = layers.Dense(8, activation="gelu")(elev_in)
    h = layers.Concatenate()([x, e])
    h = layers.Dropout(0.35)(h)
    h = layers.Dense(16, activation="gelu")(h)
    out = layers.Dense(1, dtype="float32")(h)

    gru_model = keras.Model([seq_in, elev_in], out)
    gru_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-3), loss="mae", metrics=["mae"])
    gru_model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-6),
    ]

    hist = gru_model.fit(
        [Z_train, E_train], ys_train,
        validation_data=([Z_val, E_val], ys_val),
        epochs=300,
        batch_size=16,
        callbacks=callbacks,
        verbose=0,
    )

    pred_test_s = gru_model.predict([Z_test, E_test], verbose=0)
    pred_test_gru = y_scaler.inverse_transform(pred_test_s).ravel()

    print(pd.DataFrame([
        eval_predictions("compact_pca_gru / test", y_test, pred_test_gru),
        eval_predictions(best_name + " / test", y_test, test_predictions[best_name]),
    ]))

    plt.figure(figsize=(8, 4))
    plt.plot(hist.history["loss"], label="train")
    plt.plot(hist.history["val_loss"], label="val")
    plt.title('Compact GRU - scaled loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print('Skipping GRU training.')


In [ ]:
# This block diagnoses domain shift; it does not train the final model.

def load_fem_sample(max_rows=50000):
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        print('FEM dataset not found.')
        return None
    print("Dataset FEM:", fem_path)
    if fem_path.suffix.lower() == ".csv":
        fem = pd.read_csv(fem_path, nrows=max_rows)
    else:

        try:
            fem = pd.read_excel(fem_path, engine="openpyxl", nrows=max_rows)
        except Exception:
            from openpyxl import load_workbook
            wb = load_workbook(fem_path, read_only=True, data_only=True)
            ws = wb.active
            rows = ws.iter_rows(values_only=True)
            header = list(next(rows))
            data = []
            for i, row in enumerate(rows):
                if i >= max_rows:
                    break
                data.append(row)
            wb.close()
            fem = pd.DataFrame(data, columns=header)
    fem = normalize_sensor_columns(fem)
    return fem

fem_sample = load_fem_sample(max_rows=20000)
if fem_sample is not None:
    fem_sensor_cols = get_sensor_cols(fem_sample)
    real_flat = real_raw[sensor_cols].to_numpy(dtype="float32").ravel()
    fem_flat = fem_sample[fem_sensor_cols].to_numpy(dtype="float32").ravel()
    real_flat = real_flat[np.isfinite(real_flat)]
    fem_flat = fem_flat[np.isfinite(fem_flat)]
    print("Real temp mean/std:", real_flat.mean(), real_flat.std())
    print("FEM  temp mean/std:", fem_flat.mean(), fem_flat.std())

    plt.figure(figsize=(8, 4))
    plt.hist(fem_flat[np.random.choice(len(fem_flat), min(100000, len(fem_flat)), replace=False)], bins=80, alpha=0.5, density=True, label="FEM")
    plt.hist(real_flat[np.random.choice(len(real_flat), min(100000, len(real_flat)), replace=False)], bins=80, alpha=0.5, density=True, label='Real')
    plt.xlabel('Temperature [C]')
    plt.ylabel("densita'")
    plt.title('Temperature covariate shift')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Note operative

Interpretazione dei risultati:

- Se le baseline fisiche compatte restano vicine alla baseline mediana, il target reale contiene poca informazione predicibile dai soli termometri.
- Se una baseline tabellare batte la GRU, il problema non richiede una rete piu' grande: richiede piu' dati, feature fisiche migliori o variabili mancanti.
- Se la GRU compatta batte chiaramente le baseline su test temporale blind, allora ha senso raffinare architettura e iperparametri.
- Il valore intorno a 7 arcsec va trattato come noise floor apparente finche' non viene misurata la ripetibilita' con cross-scan ravvicinati e condizioni simili.

Prossimi dati utili per scendere sotto quel limite:
- vento e meteo locale;
- inclinometri / stato meccanico;
- cross-scan consecutivi per stimare rumore di misura;
- piu' stagioni FEM o FEM calibrato su distribuzioni termiche reali;
- coordinate/adiacenze fisiche dei sensori per GNN o interpolazione spaziale rigorosa.


In [ ]:
fb_all = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq, X_elev)
F_all = fb_all.transform(X_seq, X_elev, meta)
y_all = y.copy()
meta_all = meta.sort_values("Timestamp").reset_index(drop=True)
order = np.argsort(meta["Timestamp"].to_numpy())
F_all = F_all.iloc[order].reset_index(drop=True)
y_all = y_all[order]

cv_models = {
    "median": DummyRegressor(strategy="median"),
    "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))]),
    "elasticnet": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.8, 0.95], alphas=np.logspace(-3, 2, 30), cv=3, random_state=RANDOM_SEED, max_iter=20000))]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1)),
    ]),
    "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, l2_regularization=0.1, min_samples_leaf=12, random_state=RANDOM_SEED),
}


n = len(y_all)
fold_bounds = []
for train_end_frac in [0.50, 0.625, 0.75, 0.875]:
    train_end = int(n * train_end_frac)
    test_end = min(n, train_end + max(20, int(n * 0.125)))
    if test_end > train_end and train_end >= 50:
        fold_bounds.append((train_end, test_end))

cv_rows = []
for fold, (train_end, test_end) in enumerate(fold_bounds, start=1):
    tr = np.arange(0, train_end)
    te = np.arange(train_end, test_end)
    for name, model in cv_models.items():
        m = clone(model)
        m.fit(F_all.iloc[tr], y_all[tr])
        pred = m.predict(F_all.iloc[te])
        cv_rows.append({
            "fold": fold,
            "model": name,
            "n_train": len(tr),
            "n_test": len(te),
            "test_start": meta_all.iloc[te[0]]["Timestamp"],
            "test_end": meta_all.iloc[te[-1]]["Timestamp"],
            "MAE": mean_absolute_error(y_all[te], pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_all[te], pred))),
            "R2": r2_score(y_all[te], pred) if len(np.unique(y_all[te])) > 1 else np.nan,
        })

cv_df = pd.DataFrame(cv_rows)
display(cv_df)
summary_cv = cv_df.groupby("model")[["MAE", "RMSE", "R2"]].agg(["mean", "std", "min", "max"])
display(summary_cv)


In [ ]:
err_df = meta.iloc[test_idx].copy().reset_index(drop=True)
err_df["y_true"] = y_test
err_df["y_pred"] = test_predictions[best_name]
err_df["error"] = err_df["y_true"] - err_df["y_pred"]
err_df["abs_error"] = err_df["error"].abs()
err_df["target_abs"] = err_df["y_true"].abs()
err_df["elev_band"] = pd.cut(err_df["Elevation"], bins=[0, 25, 40, 55, 70, 90], include_lowest=True)
err_df["target_band"] = pd.cut(err_df["target_abs"], bins=[0, 3, 7, 15, 30, 100], include_lowest=True)

print('Overall test MAE:', err_df["abs_error"].mean())
print('\nMAE by elevation band')
display(err_df.groupby("elev_band")["abs_error"].agg(["count", "mean", "median", "max"]))
print('\nMAE by target magnitude')
display(err_df.groupby("target_band")["abs_error"].agg(["count", "mean", "median", "max"]))
print("\nWorst 15")
display(err_df.sort_values("abs_error", ascending=False).head(15))

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].scatter(err_df["Elevation"], err_df["error"], alpha=0.8)
axs[0].axhline(0, color="k", linestyle="--")
axs[0].set_xlabel('Elevation [deg]')
axs[0].set_ylabel('Signed residual [arcsec]')
axs[0].set_title('Residuals versus elevation')
axs[0].grid(True, alpha=0.3)

axs[1].scatter(err_df["y_true"], err_df["error"], alpha=0.8)
axs[1].axhline(0, color="k", linestyle="--")
axs[1].set_xlabel('Observed target [arcsec]')
axs[1].set_title('Residuals versus target')
axs[1].grid(True, alpha=0.3)

axs[2].plot(err_df["Timestamp"].to_numpy(), err_df["abs_error"].to_numpy(), marker="o")
axs[2].set_ylabel('Absolute error [arcsec]')
axs[2].set_title('Absolute error over time')
axs[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
thresholds = [3, 7, 15, 30, 100]
rows = []
for thr in thresholds:
    mask = err_df["target_abs"] <= thr
    if mask.sum() == 0:
        continue
    rows.append({
        "subset": f"|target| <= {thr}",
        "count": int(mask.sum()),
        "coverage": float(mask.mean()),
        "MAE_model": float(err_df.loc[mask, "abs_error"].mean()),
        "median_abs_target": float(err_df.loc[mask, "target_abs"].median()),
        "max_abs_error": float(err_df.loc[mask, "abs_error"].max()),
    })

for thr in [7, 15, 30]:
    mask = err_df["target_abs"] > thr
    if mask.sum() == 0:
        continue
    rows.append({
        "subset": f"|target| > {thr}",
        "count": int(mask.sum()),
        "coverage": float(mask.mean()),
        "MAE_model": float(err_df.loc[mask, "abs_error"].mean()),
        "median_abs_target": float(err_df.loc[mask, "target_abs"].median()),
        "max_abs_error": float(err_df.loc[mask, "abs_error"].max()),
    })

regime_df = pd.DataFrame(rows)
display(regime_df)


worst_sorted = err_df.sort_values("abs_error", ascending=False).reset_index(drop=True)
impact_rows = []
for k in [1, 2, 3, 5, 10, 15]:
    if k >= len(worst_sorted):
        continue
    kept = worst_sorted.iloc[k:]
    impact_rows.append({
        "removed_worst_k": k,
        "remaining_count": len(kept),
        "MAE_remaining": float(kept["abs_error"].mean()),
        "removed_share_of_total_abs_error": float(worst_sorted.iloc[:k]["abs_error"].sum() / worst_sorted["abs_error"].sum()),
    })
impact_df = pd.DataFrame(impact_rows)
display(impact_df)

plt.figure(figsize=(7, 4))
plt.scatter(err_df["target_abs"], err_df["abs_error"], alpha=0.8)
plt.xlabel('|Observed target| [arcsec]')
plt.ylabel('Model absolute error [arcsec]')
plt.title('Target tails dominate the error')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
def spec_metrics(y_true, y_pred, spec=SPEC_ARCSEC, name="model"):
    err = np.asarray(y_true) - np.asarray(y_pred)
    abs_err = np.abs(err)
    return {
        "model": name,
        "MAE": float(abs_err.mean()),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "within_spec_%": float((abs_err <= spec).mean() * 100),
        "p50_abs_error": float(np.percentile(abs_err, 50)),
        "p75_abs_error": float(np.percentile(abs_err, 75)),
        "p90_abs_error": float(np.percentile(abs_err, 90)),
        "p95_abs_error": float(np.percentile(abs_err, 95)),
        "max_abs_error": float(abs_err.max()),
    }

spec_rows = []
for name, pred in test_predictions.items():
    spec_rows.append(spec_metrics(y_test, pred, name=name))
if "pred_test_gru" in globals():
    spec_rows.append(spec_metrics(y_test, pred_test_gru, name="compact_pca_gru"))

spec_df = pd.DataFrame(spec_rows).sort_values(["within_spec_%", "MAE"], ascending=[False, True])
display(spec_df)


if "err_df" in globals():
    rows = []
    for label, mask in {
        "all_test": np.ones(len(err_df), dtype=bool),
        "|target| <= 7": err_df["target_abs"] <= 7,
        "|target| <= 15": err_df["target_abs"] <= 15,
        "|target| > 15": err_df["target_abs"] > 15,
    }.items():
        if mask.sum() == 0:
            continue
        sub = err_df.loc[mask]
        rows.append({
            "subset": label,
            "count": int(mask.sum()),
            "coverage_%": float(mask.mean() * 100),
            "MAE": float(sub["abs_error"].mean()),
            "within_5arcsec_%": float((sub["abs_error"] <= SPEC_ARCSEC).mean() * 100),
            "p90_abs_error": float(np.percentile(sub["abs_error"], 90)),
            "max_abs_error": float(sub["abs_error"].max()),
        })
    display(pd.DataFrame(rows))


In [ ]:
if "F_all" not in globals():
    fb_all = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq, X_elev)
    F_all = fb_all.transform(X_seq, X_elev, meta)
    order = np.argsort(meta["Timestamp"].to_numpy())
    F_all = F_all.iloc[order].reset_index(drop=True)
    y_all = y[order]
else:
    y_all = y_all

holdout_frac = 0.20
n = len(y_all)
holdout_start = int(n * (1 - holdout_frac))
X_holdout = F_all.iloc[holdout_start:]
y_holdout = y_all[holdout_start:]

train_sizes = [40, 80, 120, 160, 220, holdout_start]
train_sizes = sorted(set([s for s in train_sizes if 20 <= s <= holdout_start]))

lc_rows = []
for s in train_sizes:
    tr = np.arange(0, s)
    m = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1)),
    ])
    m.fit(F_all.iloc[tr], y_all[tr])
    pred = m.predict(X_holdout)
    abs_err = np.abs(y_holdout - pred)
    lc_rows.append({
        "n_train": s,
        "n_holdout": len(y_holdout),
        "MAE": float(abs_err.mean()),
        "within_5arcsec_%": float((abs_err <= SPEC_ARCSEC).mean() * 100),
        "p90_abs_error": float(np.percentile(abs_err, 90)),
    })

learning_curve_df = pd.DataFrame(lc_rows)
display(learning_curve_df)

plt.figure(figsize=(8, 4))
plt.plot(learning_curve_df["n_train"], learning_curve_df["MAE"], marker="o", label="MAE")
plt.ylabel("MAE [arcsec]")
plt.xlabel('Training samples')
plt.grid(True, alpha=0.3)
plt.twinx()
plt.plot(learning_curve_df["n_train"], learning_curve_df["within_5arcsec_%"], marker="s", color="tab:green", label='within 5 arcsec')
plt.ylabel('Within 5 arcsec [%]')
plt.title('Operational learning curve')
plt.show()


In [ ]:
OP_MIN_ELEV = 20.0
OP_MAX_ELEV = 80.0
EXTREME_TARGET_ARCSEC = 15.0
EXTREME_ERROR_ARCSEC = 15.0

if "err_df" not in globals():
    raise RuntimeError('Run cell 13 first to create err_df.')

op_mask = err_df["Elevation"].between(OP_MIN_ELEV, OP_MAX_ELEV)
extreme_target = err_df["target_abs"] > EXTREME_TARGET_ARCSEC
extreme_error = err_df["abs_error"] > EXTREME_ERROR_ARCSEC

summary = pd.DataFrame([
    {"subset": "all_test", "count": len(err_df), "extreme_target_count": int(extreme_target.sum()), "extreme_target_%": float(extreme_target.mean()*100), "extreme_error_count": int(extreme_error.sum()), "extreme_error_%": float(extreme_error.mean()*100), "MAE": float(err_df["abs_error"].mean()), "within_5_%": float((err_df["abs_error"] <= SPEC_ARCSEC).mean()*100)},
    {"subset": f"operational_elev_{OP_MIN_ELEV:.0f}_{OP_MAX_ELEV:.0f}", "count": int(op_mask.sum()), "extreme_target_count": int((op_mask & extreme_target).sum()), "extreme_target_%": float((op_mask & extreme_target).sum()/max(op_mask.sum(),1)*100), "extreme_error_count": int((op_mask & extreme_error).sum()), "extreme_error_%": float((op_mask & extreme_error).sum()/max(op_mask.sum(),1)*100), "MAE": float(err_df.loc[op_mask, "abs_error"].mean()), "within_5_%": float((err_df.loc[op_mask, "abs_error"] <= SPEC_ARCSEC).mean()*100)},
    {"subset": f"outside_operational_elev", "count": int((~op_mask).sum()), "extreme_target_count": int(((~op_mask) & extreme_target).sum()), "extreme_target_%": float(((~op_mask) & extreme_target).sum()/max((~op_mask).sum(),1)*100), "extreme_error_count": int(((~op_mask) & extreme_error).sum()), "extreme_error_%": float(((~op_mask) & extreme_error).sum()/max((~op_mask).sum(),1)*100), "MAE": float(err_df.loc[~op_mask, "abs_error"].mean()) if (~op_mask).sum() else np.nan, "within_5_%": float((err_df.loc[~op_mask, "abs_error"] <= SPEC_ARCSEC).mean()*100) if (~op_mask).sum() else np.nan},
])
display(summary)

print('Events with |target| >', EXTREME_TARGET_ARCSEC)
cols = ["Timestamp", "Elevation", "Target", "y_pred", "error", "abs_error", "target_abs"]
display(err_df.loc[extreme_target, cols].sort_values("target_abs", ascending=False))

print('Events with |residual| >', EXTREME_ERROR_ARCSEC)
display(err_df.loc[extreme_error, cols].sort_values("abs_error", ascending=False))

print('Distribution by elevation band')
bands = pd.cut(err_df["Elevation"], bins=[0, 15, 20, 30, 45, 60, 75, 80, 90], include_lowest=True)
display(err_df.assign(elev_band=bands).groupby("elev_band", observed=True).agg(
    count=("abs_error", "size"),
    mae=("abs_error", "mean"),
    within_5=("abs_error", lambda s: (s <= SPEC_ARCSEC).mean()*100),
    extreme_target=("target_abs", lambda s: (s > EXTREME_TARGET_ARCSEC).sum()),
    extreme_error=("abs_error", lambda s: (s > EXTREME_ERROR_ARCSEC).sum()),
))


## Grafici riassuntivi

Questa sezione produce figure pensate per una presentazione: mostra la prestazione rispetto al requisito operativo di 5 arcsec, separa il regime nominale dalle code estreme e indica dove il modello sbaglia di pi?.

In [ ]:
if "err_df" not in globals():
    raise RuntimeError('Run cell 13 first to create err_df.')

PLOT_DIR = Path("aims_figures")
PLOT_DIR.mkdir(exist_ok=True)

plot_df = err_df.copy().sort_values("Timestamp").reset_index(drop=True)
plot_df["within_spec"] = plot_df["abs_error"] <= SPEC_ARCSEC
plot_df["extreme_target"] = plot_df["target_abs"] > EXTREME_TARGET_ARCSEC
plot_df["sample_id"] = np.arange(len(plot_df))

coverage = plot_df["within_spec"].mean() * 100
mae_all = plot_df["abs_error"].mean()
mae_nominal = plot_df.loc[plot_df["target_abs"] <= EXTREME_TARGET_ARCSEC, "abs_error"].mean()
coverage_nominal = (plot_df.loc[plot_df["target_abs"] <= EXTREME_TARGET_ARCSEC, "abs_error"] <= SPEC_ARCSEC).mean() * 100
n_extreme = int(plot_df["extreme_target"].sum())

print(f"Displayed model: {best_name}")
print(f"Overall MAE: {mae_all:.2f} arcsec | within {SPEC_ARCSEC:.0f} arcsec: {coverage:.1f}%")
print(f"Regime |target| <= {EXTREME_TARGET_ARCSEC:.0f}: MAE {mae_nominal:.2f} arcsec | within threshold {coverage_nominal:.1f}%")
print(f"Extreme events |target| > {EXTREME_TARGET_ARCSEC:.0f}: {n_extreme}/{len(plot_df)}")


fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(plot_df["Timestamp"].to_numpy(), plot_df["y_true"].to_numpy(), color="0.25", lw=1.5, label='Observed XEL')
ax.plot(plot_df["Timestamp"].to_numpy(), plot_df["y_pred"].to_numpy(), color="tab:blue", lw=1.7, label=f"Prediction {best_name}")

bad = ~plot_df["within_spec"]
extreme = plot_df["extreme_target"]
ax.scatter(plot_df.loc[bad, "Timestamp"].to_numpy(), plot_df.loc[bad, "y_true"].to_numpy(), s=55, facecolors="none", edgecolors="tab:orange", linewidths=1.8, label=f"Error > {SPEC_ARCSEC:.0f}\"")
ax.scatter(plot_df.loc[extreme, "Timestamp"].to_numpy(), plot_df.loc[extreme, "y_true"].to_numpy(), s=75, color="tab:red", marker="x", linewidths=2.0, label=f"|target| > {EXTREME_TARGET_ARCSEC:.0f}\"")
ax.axhline(0, color="k", lw=0.8, alpha=0.5)
ax.set_title(f"Observed versus predicted XEL - MAE {mae_all:.2f}\", coverage within {SPEC_ARCSEC:.0f}\": {coverage:.1f}%")
ax.set_ylabel("XEL [arcsec]")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_timeseries_prediction_vs_real.png", dpi=180)
plt.show()


fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(plot_df["abs_error"], bins=22, color="tab:blue", alpha=0.75, edgecolor="white")
ax.axvline(SPEC_ARCSEC, color="tab:green", lw=2.5, linestyle="--", label=f"Requirement {SPEC_ARCSEC:.0f}\"")
ax.axvline(plot_df["abs_error"].median(), color="k", lw=2, linestyle=":", label=f"Median {plot_df['abs_error'].median():.2f}\"")
ax.set_title('Absolute-error distribution')
ax.set_xlabel('|error| [arcsec]')
ax.set_ylabel('Sample count')
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "02_absolute_error_distribution.png", dpi=180)
plt.show()


fig, ax = plt.subplots(figsize=(8.5, 5.5))
colors = np.where(plot_df["within_spec"], "tab:green", "tab:orange")
colors = np.where(plot_df["extreme_target"], "tab:red", colors)
ax.scatter(plot_df["target_abs"], plot_df["abs_error"], c=colors, s=55, alpha=0.85, edgecolor="white", linewidth=0.5)
ax.axhline(SPEC_ARCSEC, color="tab:green", linestyle="--", lw=2, label=f"Requirement {SPEC_ARCSEC:.0f}\"")
ax.axvline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--", lw=2, label=f"Tail threshold |target|={EXTREME_TARGET_ARCSEC:.0f}\"")
ax.set_xlabel('|Observed XEL| [arcsec]')
ax.set_ylabel('|Model error| [arcsec]')
ax.set_title('The model meets the threshold on moderate cases but struggles with the tails')
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "03_target_magnitude_vs_error.png", dpi=180)
plt.show()


target_bins = [0, 3, 7, 15, 30, 100]
target_labels = ['0-3"', '3-7"', '7-15"', '15-30"', '>30"']
plot_df["target_bin_plot"] = pd.cut(plot_df["target_abs"], bins=target_bins, labels=target_labels, include_lowest=True)
bin_stats = plot_df.groupby("target_bin_plot", observed=False).agg(
    count=("abs_error", "size"),
    mae=("abs_error", "mean"),
    within_spec=("within_spec", lambda s: s.mean() * 100),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
bars = ax1.bar(bin_stats["target_bin_plot"].astype(str), bin_stats["within_spec"], color="tab:green", alpha=0.75)
ax1.axhline(80, color="0.35", linestyle=":", lw=1.5, label='80% reference')
ax1.set_ylim(0, 105)
ax1.set_ylabel(f"Samples within {SPEC_ARCSEC:.0f}\" [%]")
ax1.set_xlabel('Observed target magnitude |XEL|')
ax1.set_title('Within-threshold coverage by observed-error regime')
ax1.grid(True, axis="y", alpha=0.25)
for bar, cnt, mae in zip(bars, bin_stats["count"], bin_stats["mae"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"n={cnt}\nMAE={mae:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(PLOT_DIR / "04_within_spec_by_target_bin.png", dpi=180)
plt.show()


fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(plot_df["Elevation"], plot_df["abs_error"], c=colors, s=60, alpha=0.85, edgecolor="white", linewidth=0.5)
ax.axhline(SPEC_ARCSEC, color="tab:green", linestyle="--", lw=2, label=f"Requirement {SPEC_ARCSEC:.0f}\"")
ax.axvspan(20, 80, color="tab:blue", alpha=0.06, label='Example envelope: 20-80 deg')
ax.set_xlabel('Elevation [deg]')
ax.set_ylabel('|Model error| [arcsec]')
ax.set_title('Model errors: elevation and target tails')
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "05_error_vs_elevation.png", dpi=180)
plt.show()


sorted_err = plot_df.sort_values("abs_error", ascending=False).reset_index(drop=True)
sorted_err["cum_share"] = sorted_err["abs_error"].cumsum() / sorted_err["abs_error"].sum() * 100
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(np.arange(1, len(sorted_err)+1), sorted_err["abs_error"], color=np.where(sorted_err["extreme_target"], "tab:red", "tab:blue"), alpha=0.75)
ax1.set_xlabel('Samples ordered from worst to best')
ax1.set_ylabel('|error| [arcsec]')
ax1.axhline(SPEC_ARCSEC, color="tab:green", linestyle="--", lw=2, label=f"Requirement {SPEC_ARCSEC:.0f}\"")
ax2 = ax1.twinx()
ax2.plot(np.arange(1, len(sorted_err)+1), sorted_err["cum_share"], color="k", lw=2, label='Cumulative error')
ax2.set_ylabel('Cumulative absolute-error share [%]')
ax2.set_ylim(0, 105)
ax1.set_title('A few events dominate overall MAE')
ax1.grid(True, axis="y", alpha=0.25)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "06_pareto_worst_errors.png", dpi=180)
plt.show()

print(f"Figures saved in: {PLOT_DIR.resolve()}")
display(bin_stats)


In [ ]:
summary_rows = []
if "real_raw" in globals():
    summary_rows.append({
        "metric": "raw_rows",
        "value": len(real_raw),
    })
    summary_rows.append({
        "metric": "valid_targets_XEL",
        "value": int(real_raw["XEl Offset"].notna().sum()),
    })
    summary_rows.append({
        "metric": "time_min",
        "value": real_raw["Timestamp"].min(),
    })
    summary_rows.append({
        "metric": "time_max",
        "value": real_raw["Timestamp"].max(),
    })
if "X_seq" in globals():
    summary_rows.append({"metric": "valid_windows", "value": len(X_seq)})
    summary_rows.append({"metric": "valid_window_fraction_of_targets", "value": len(X_seq) / max(int(real_raw["XEl Offset"].notna().sum()), 1)})
if "skip" in globals():
    for k, v in skip.items():
        summary_rows.append({"metric": f"skipped_{k}", "value": v})
if "spec_df" in globals():
    best_spec = spec_df.sort_values(["within_spec_%", "MAE"], ascending=[False, True]).iloc[0]
    summary_rows.extend([
        {"metric": "best_operational_model", "value": best_spec["model"]},
        {"metric": "best_within_5_percent", "value": best_spec["within_spec_%"]},
        {"metric": "best_mae", "value": best_spec["MAE"]},
        {"metric": "best_p90_abs_error", "value": best_spec["p90_abs_error"]},
    ])

summary_total_df = pd.DataFrame(summary_rows)
display(summary_total_df)

if "err_df" in globals():
    def safe_p90(s):
        s = s.dropna()
        return np.percentile(s, 90) if len(s) else np.nan

    print('Performance by elevation band on the current test')
    bands = pd.cut(err_df["Elevation"], bins=[0, 15, 20, 30, 45, 60, 75, 80, 90], include_lowest=True)
    display(err_df.assign(elev_band=bands).groupby("elev_band", observed=True).agg(
        count=("abs_error", "size"),
        mae=("abs_error", "mean"),
        within_5=("abs_error", lambda s: (s <= SPEC_ARCSEC).mean()*100),
        p90=("abs_error", safe_p90),
        extreme_target=("target_abs", lambda s: (s > EXTREME_TARGET_ARCSEC).sum()),
    ))


In [ ]:
if "meta" not in globals():
    raise RuntimeError('Build the windows first.')

regime_meta = meta.copy()
regime_meta["split"] = ""
regime_meta.loc[train_idx, "split"] = "train"
regime_meta.loc[val_idx, "split"] = "val"
regime_meta.loc[test_idx, "split"] = "test"
regime_meta["month"] = pd.to_datetime(regime_meta["Timestamp"]).dt.month
regime_meta["abs_target"] = regime_meta["Target"].abs()
regime_meta["extreme_target"] = regime_meta["abs_target"] > EXTREME_TARGET_ARCSEC

print('Target distribution by split')
display(regime_meta.groupby("split").agg(
    count=("Target", "size"),
    mean_target=("Target", "mean"),
    std_target=("Target", "std"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_count=("extreme_target", "sum"),
    extreme_pct=("extreme_target", lambda s: s.mean()*100),
    min_time=("Timestamp", "min"),
    max_time=("Timestamp", "max"),
))

print('Distribution by month and split')
month_table = pd.pivot_table(regime_meta, index="month", columns="split", values="Target", aggfunc="count", fill_value=0)
display(month_table)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for split, g in regime_meta.groupby("split"):
    axs[0].hist(g["Target"], bins=35, alpha=0.5, label=split, density=True)
axs[0].set_title('XEL distribution by split')
axs[0].set_xlabel("XEL [arcsec]")
axs[0].grid(True, alpha=0.25)
axs[0].legend()

for split, g in regime_meta.groupby("split"):
    axs[1].hist(g["abs_target"], bins=35, alpha=0.5, label=split, density=True)
axs[1].axvline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--", label='tail threshold')
axs[1].set_title('|XEL| magnitude by split')
axs[1].set_xlabel("|XEL| [arcsec]")
axs[1].grid(True, alpha=0.25)
axs[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
if "F_all" not in globals():
    fb_all = FeatureBuilder(n_pca_current=8, n_pca_trend=8).fit(X_seq, X_elev)
    F_all = fb_all.transform(X_seq, X_elev, meta)
    order = np.argsort(meta["Timestamp"].to_numpy())
    F_all = F_all.iloc[order].reset_index(drop=True)
    y_all = y[order]
    meta_all = meta.iloc[order].reset_index(drop=True)
else:

    order = np.argsort(meta["Timestamp"].to_numpy())
    meta_all = meta.iloc[order].reset_index(drop=True)

meta_all = meta_all.copy()
meta_all["month"] = pd.to_datetime(meta_all["Timestamp"]).dt.month
meta_all["abs_target"] = np.abs(y_all)
meta_all["extreme"] = meta_all["abs_target"] > EXTREME_TARGET_ARCSEC

print('Targets by month among valid windows')
display(meta_all.groupby("month").agg(
    count=("Target", "size"),
    mean_target=("Target", "mean"),
    std_target=("Target", "std"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_pct=("extreme", lambda s: s.mean()*100),
))


rolling_rows = []
models_rolling = {
    "median": DummyRegressor(strategy="median"),
    "huber": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=2000))]),
    "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=8, max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1)),
    ]),
}


for month in sorted(meta_all["month"].unique()):
    te = np.flatnonzero(meta_all["month"].to_numpy() == month)
    if len(te) < 20:
        continue
    first_te = te[0]
    tr = np.arange(0, first_te)
    if len(tr) < 100:
        continue
    for name, model in models_rolling.items():
        m = clone(model)
        m.fit(F_all.iloc[tr], y_all[tr])
        pred = m.predict(F_all.iloc[te])
        abs_err = np.abs(y_all[te] - pred)
        rolling_rows.append({
            "test_month": int(month),
            "model": name,
            "n_train": len(tr),
            "n_test": len(te),
            "MAE": float(abs_err.mean()),
            "within_5_%": float((abs_err <= SPEC_ARCSEC).mean()*100),
            "p90_abs_error": float(np.percentile(abs_err, 90)),
            "extreme_pct_test": float((np.abs(y_all[te]) > EXTREME_TARGET_ARCSEC).mean()*100),
        })

rolling_df = pd.DataFrame(rolling_rows)
display(rolling_df)
if len(rolling_df):
    display(rolling_df.pivot_table(index="test_month", columns="model", values="MAE"))
    display(rolling_df.pivot_table(index="test_month", columns="model", values="within_5_%"))


In [ ]:
if "meta" not in globals():
    raise RuntimeError('Build the windows first.')

focus = meta.copy()
focus["month"] = pd.to_datetime(focus["Timestamp"]).dt.month
focus["date"] = pd.to_datetime(focus["Timestamp"]).dt.date
focus["hour"] = pd.to_datetime(focus["Timestamp"]).dt.hour + pd.to_datetime(focus["Timestamp"]).dt.minute / 60.0
focus["abs_target"] = focus["Target"].abs()
focus["extreme_target"] = focus["abs_target"] > EXTREME_TARGET_ARCSEC
focus["is_june"] = focus["month"] == 6
focus["split"] = ""
if "train_idx" in globals(): focus.loc[train_idx, "split"] = "train"
if "val_idx" in globals(): focus.loc[val_idx, "split"] = "val"
if "test_idx" in globals(): focus.loc[test_idx, "split"] = "test"

print('June versus other months - valid windows')
display(focus.groupby("is_june").agg(
    count=("Target", "size"),
    mean_target=("Target", "mean"),
    std_target=("Target", "std"),
    median_abs_target=("abs_target", "median"),
    p75_abs_target=("abs_target", lambda s: np.percentile(s, 75)),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_count=("extreme_target", "sum"),
    extreme_pct=("extreme_target", lambda s: s.mean()*100),
    mean_elev=("Elevation", "mean"),
    median_elev=("Elevation", "median"),
))

print('June by split')
display(focus[focus["is_june"]].groupby("split").agg(
    count=("Target", "size"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_count=("extreme_target", "sum"),
    extreme_pct=("extreme_target", lambda s: s.mean()*100),
))

print('June by elevation band')
elev_bins = [0, 15, 20, 30, 45, 60, 75, 80, 90]
focus_june = focus[focus["is_june"]].copy()
focus_june["elev_band"] = pd.cut(focus_june["Elevation"], bins=elev_bins, include_lowest=True)
display(focus_june.groupby("elev_band", observed=True).agg(
    count=("Target", "size"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_count=("extreme_target", "sum"),
    extreme_pct=("extreme_target", lambda s: s.mean()*100),
))

print('Extreme June events: |target| above threshold')
display(focus_june[focus_june["extreme_target"]].sort_values("abs_target", ascending=False)[
    ["Timestamp", "split", "Elevation", "Target", "abs_target", "hour"]
].head(40))


if "err_df" in globals():
    err_month = err_df.copy()
    err_month["month"] = pd.to_datetime(err_month["Timestamp"]).dt.month
    err_month["is_june"] = err_month["month"] == 6
    if err_month["is_june"].any():
        print('Test performance: June versus other months')
        display(err_month.groupby("is_june").agg(
            count=("abs_error", "size"),
            mae=("abs_error", "mean"),
            median_abs_error=("abs_error", "median"),
            within_5=("abs_error", lambda s: (s <= SPEC_ARCSEC).mean()*100),
            p90_abs_error=("abs_error", lambda s: np.percentile(s, 90)),
            extreme_target=("target_abs", lambda s: (s > EXTREME_TARGET_ARCSEC).sum()),
        ))
    else:
        print('The current test contains no June samples.')


fig, axs = plt.subplots(2, 2, figsize=(14, 9))
axs[0, 0].hist(focus.loc[~focus["is_june"], "Target"], bins=40, alpha=0.5, density=True, label='other months')
axs[0, 0].hist(focus.loc[focus["is_june"], "Target"], bins=40, alpha=0.5, density=True, label='June')
axs[0, 0].axvline(0, color="k", lw=0.8)
axs[0, 0].set_title('XEL distribution: June versus other months')
axs[0, 0].set_xlabel("XEL [arcsec]")
axs[0, 0].legend()
axs[0, 0].grid(True, alpha=0.25)

axs[0, 1].scatter(focus_june["Elevation"], focus_june["abs_target"], c=np.where(focus_june["extreme_target"], "tab:red", "tab:blue"), alpha=0.8)
axs[0, 1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[0, 1].set_title('June: |XEL| versus elevation')
axs[0, 1].set_xlabel('Elevation [deg]')
axs[0, 1].set_ylabel("|XEL| [arcsec]")
axs[0, 1].grid(True, alpha=0.25)

axs[1, 0].scatter(focus_june["hour"], focus_june["abs_target"], c=np.where(focus_june["extreme_target"], "tab:red", "tab:blue"), alpha=0.8)
axs[1, 0].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[1, 0].set_title('June: |XEL| versus time of day')
axs[1, 0].set_xlabel('UTC hour')
axs[1, 0].set_ylabel("|XEL| [arcsec]")
axs[1, 0].grid(True, alpha=0.25)

june_daily = focus_june.groupby("date").agg(count=("Target", "size"), median_abs=("abs_target", "median"), max_abs=("abs_target", "max"), extreme=("extreme_target", "sum")).reset_index()
axs[1, 1].plot(pd.to_datetime(june_daily["date"]).to_numpy(), june_daily["max_abs"].to_numpy(), marker="o", label="max |XEL|")
axs[1, 1].plot(pd.to_datetime(june_daily["date"]).to_numpy(), june_daily["median_abs"].to_numpy(), marker="s", label='median |XEL|')
axs[1, 1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[1, 1].set_title('June: daily evolution')
axs[1, 1].set_ylabel("|XEL| [arcsec]")
axs[1, 1].grid(True, alpha=0.25)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig(Path("aims_figures") / "07_focus_june.png", dpi=180)
plt.show()


In [ ]:
if "meta" not in globals():
    raise RuntimeError('Build the windows first.')

risk = meta.copy()
risk["month"] = pd.to_datetime(risk["Timestamp"]).dt.month
risk["abs_target"] = risk["Target"].abs()
risk["extreme_target"] = risk["abs_target"] > EXTREME_TARGET_ARCSEC
risk["elev_band"] = pd.cut(risk["Elevation"], bins=[0, 20, 30, 45, 60, 75, 80, 90], include_lowest=True)

risk_table = risk.groupby(["month", "elev_band"], observed=True).agg(
    count=("Target", "size"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: np.percentile(s, 90)),
    extreme_pct=("extreme_target", lambda s: s.mean() * 100),
).reset_index()

display(risk_table.sort_values(["extreme_pct", "count"], ascending=[False, False]).head(30))


pivot_extreme = risk_table.pivot(index="elev_band", columns="month", values="extreme_pct")
pivot_count = risk_table.pivot(index="elev_band", columns="month", values="count")

fig, ax = plt.subplots(figsize=(12, 5.5))
im = ax.imshow(pivot_extreme.fillna(0).to_numpy(), aspect="auto", cmap="Reds", vmin=0, vmax=max(50, np.nanmax(pivot_extreme.to_numpy())))
ax.set_xticks(np.arange(len(pivot_extreme.columns)))
ax.set_xticklabels(pivot_extreme.columns)
ax.set_yticks(np.arange(len(pivot_extreme.index)))
ax.set_yticklabels([str(x) for x in pivot_extreme.index])
ax.set_xlabel('Month')
ax.set_ylabel('Elevation band')
ax.set_title(f"Extreme-event risk |XEL| > {EXTREME_TARGET_ARCSEC:.0f}\" by month/elevation")
for i in range(pivot_extreme.shape[0]):
    for j in range(pivot_extreme.shape[1]):
        val = pivot_extreme.iloc[i, j]
        cnt = pivot_count.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.0f}%\nn={int(cnt)}", ha="center", va="center", fontsize=8, color="black")
fig.colorbar(im, ax=ax, label='extreme events [%]')
plt.tight_layout()
plt.savefig(Path("aims_figures") / "08_heatmap_month_elevation_risk.png", dpi=180)
plt.show()


if "err_df" in globals():
    scenarios = []
    base = err_df.copy()
    base["month"] = pd.to_datetime(base["Timestamp"]).dt.month
    scenario_masks = {
        "all_test": np.ones(len(base), dtype=bool),
        "20<=el<=80": base["Elevation"].between(20, 80),
        "20<=el<=75": base["Elevation"].between(20, 75),
        "30<=el<=75": base["Elevation"].between(30, 75),
        "exclude_jun": base["month"] != 6,
        "exclude_jul": base["month"] != 7,
        "exclude_jun_jul": ~base["month"].isin([6, 7]),
        "20<=el<=75_and_no_jun_jul": base["Elevation"].between(20, 75) & (~base["month"].isin([6, 7])),
    }
    for name, mask in scenario_masks.items():
        sub = base.loc[mask]
        if len(sub) == 0:
            continue
        scenarios.append({
            "scenario": name,
            "count": len(sub),
            "coverage_of_test_%": len(sub) / len(base) * 100,
            "MAE": sub["abs_error"].mean(),
            "within_5_%": (sub["abs_error"] <= SPEC_ARCSEC).mean() * 100,
            "p90_abs_error": np.percentile(sub["abs_error"], 90),
            "extreme_target_%": (sub["target_abs"] > EXTREME_TARGET_ARCSEC).mean() * 100,
            "max_abs_error": sub["abs_error"].max(),
        })
    scenario_df = pd.DataFrame(scenarios).sort_values("within_5_%", ascending=False)
    display(scenario_df)


In [ ]:
if "X_seq" not in globals() or "meta" not in globals():
    raise RuntimeError('Build the windows first.')

sensor_diag = meta.copy().reset_index(drop=True)
sensor_diag["abs_target"] = sensor_diag["Target"].abs()
sensor_diag["extreme_target"] = sensor_diag["abs_target"] > EXTREME_TARGET_ARCSEC
sensor_diag["month"] = pd.to_datetime(sensor_diag["Timestamp"]).dt.month
sensor_diag["hour"] = pd.to_datetime(sensor_diag["Timestamp"]).dt.hour + pd.to_datetime(sensor_diag["Timestamp"]).dt.minute / 60.0

current = X_seq[:, -1, :]
start = X_seq[:, 0, :]
diff = np.diff(X_seq, axis=1)
trend = current - start

sensor_diag["window_mean"] = X_seq.mean(axis=(1, 2))
sensor_diag["window_std"] = X_seq.std(axis=(1, 2))
sensor_diag["current_mean"] = current.mean(axis=1)
sensor_diag["current_std_spatial"] = current.std(axis=1)
sensor_diag["current_range"] = current.max(axis=1) - current.min(axis=1)
sensor_diag["trend_mean"] = trend.mean(axis=1)
sensor_diag["trend_std"] = trend.std(axis=1)
sensor_diag["trend_range"] = trend.max(axis=1) - trend.min(axis=1)
sensor_diag["abs_rate_mean"] = np.abs(diff).mean(axis=(1, 2))
sensor_diag["abs_rate_max"] = np.abs(diff).max(axis=(1, 2))

print('Window quality/dynamics: extreme versus normal events')
quality_cols = [
    "abs_target", "Elevation", "window_mean", "window_std", "current_mean",
    "current_std_spatial", "current_range", "trend_mean", "trend_std",
    "trend_range", "abs_rate_mean", "abs_rate_max",
]
display(sensor_diag.groupby("extreme_target")[quality_cols].agg(["count", "mean", "median", "std", "max"]))


ext_mask = sensor_diag["extreme_target"].to_numpy()
normal_mask = ~ext_mask

rows = []
for col in quality_cols[1:]:
    a = sensor_diag.loc[ext_mask, col].dropna().to_numpy()
    b = sensor_diag.loc[normal_mask, col].dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        continue
    pooled = np.sqrt((np.var(a) + np.var(b)) / 2)
    effect = (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan
    rows.append({
        "feature": col,
        "mean_extreme": np.mean(a),
        "mean_normal": np.mean(b),
        "median_extreme": np.median(a),
        "median_normal": np.median(b),
        "std_effect_extreme_minus_normal": effect,
    })
feature_shift_df = pd.DataFrame(rows).sort_values("std_effect_extreme_minus_normal", key=lambda s: s.abs(), ascending=False)
print('Aggregate differences: extreme minus normal')
display(feature_shift_df)


def top_sensor_shifts(matrix, label, top_n=20):
    ext_vals = matrix[ext_mask]
    norm_vals = matrix[normal_mask]
    if len(ext_vals) == 0 or len(norm_vals) == 0:
        return pd.DataFrame()
    mean_ext = ext_vals.mean(axis=0)
    mean_norm = norm_vals.mean(axis=0)
    std_ext = ext_vals.std(axis=0)
    std_norm = norm_vals.std(axis=0)
    pooled = np.sqrt((std_ext**2 + std_norm**2) / 2)
    effect = np.divide(mean_ext - mean_norm, pooled, out=np.zeros_like(mean_ext), where=pooled > 0)
    out = pd.DataFrame({
        "sensor": sensor_cols,
        f"mean_extreme_{label}": mean_ext,
        f"mean_normal_{label}": mean_norm,
        f"delta_{label}": mean_ext - mean_norm,
        f"std_effect_{label}": effect,
    })
    return out.reindex(out[f"std_effect_{label}"].abs().sort_values(ascending=False).index).head(top_n)

print('Sensors with the largest current-temperature shift')
top_current_shift = top_sensor_shifts(current, "current", top_n=25)
display(top_current_shift)

print('Sensors with the largest two-hour trend shift')
top_trend_shift = top_sensor_shifts(trend, "trend", top_n=25)
display(top_trend_shift)


thermal_vec = np.concatenate([current, trend], axis=1)
sc_ood = StandardScaler().fit(thermal_vec[normal_mask])
max_pca = min(12, thermal_vec[normal_mask].shape[0] - 1, thermal_vec.shape[1])
pca_ood = PCA(n_components=max_pca, random_state=RANDOM_SEED).fit(sc_ood.transform(thermal_vec[normal_mask]))
z = pca_ood.transform(sc_ood.transform(thermal_vec))

scale = np.sqrt(np.maximum(pca_ood.explained_variance_, 1e-9))
sensor_diag["thermal_ood_distance"] = np.sqrt(((z / scale) ** 2).sum(axis=1))

print('Thermal OOD distance: extreme versus normal')
display(sensor_diag.groupby("extreme_target")["thermal_ood_distance"].agg(["count", "mean", "median", "std", "min", "max"]))


print('Extreme events ordered by |target| with thermal diagnostics')
display(sensor_diag[sensor_diag["extreme_target"]].sort_values("abs_target", ascending=False)[
    ["Timestamp", "month", "hour", "Elevation", "Target", "abs_target", "window_mean", "current_std_spatial", "trend_mean", "trend_std", "abs_rate_max", "thermal_ood_distance"]
].head(50))


fig, axs = plt.subplots(2, 2, figsize=(14, 9))
colors = np.where(sensor_diag["extreme_target"], "tab:red", "tab:blue")
axs[0, 0].scatter(sensor_diag["current_std_spatial"], sensor_diag["abs_target"], c=colors, alpha=0.75, edgecolor="white", linewidth=0.4)
axs[0, 0].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[0, 0].set_xlabel('Current-temperature spatial std [C]')
axs[0, 0].set_ylabel("|XEL| [arcsec]")
axs[0, 0].set_title('Target versus thermal nonuniformity')
axs[0, 0].grid(True, alpha=0.25)

axs[0, 1].scatter(sensor_diag["trend_std"], sensor_diag["abs_target"], c=colors, alpha=0.75, edgecolor="white", linewidth=0.4)
axs[0, 1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[0, 1].set_xlabel('Two-hour trend spatial std [C]')
axs[0, 1].set_ylabel("|XEL| [arcsec]")
axs[0, 1].set_title('Target versus differential thermal trend')
axs[0, 1].grid(True, alpha=0.25)

axs[1, 0].scatter(sensor_diag["thermal_ood_distance"], sensor_diag["abs_target"], c=colors, alpha=0.75, edgecolor="white", linewidth=0.4)
axs[1, 0].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
axs[1, 0].set_xlabel('Thermal OOD distance [PCA]')
axs[1, 0].set_ylabel("|XEL| [arcsec]")
axs[1, 0].set_title('Are extreme events thermally out of distribution?')
axs[1, 0].grid(True, alpha=0.25)

axs[1, 1].hist(sensor_diag.loc[normal_mask, "thermal_ood_distance"], bins=30, alpha=0.55, density=True, label='normal')
axs[1, 1].hist(sensor_diag.loc[ext_mask, "thermal_ood_distance"], bins=30, alpha=0.55, density=True, label='extreme')
axs[1, 1].set_xlabel('Thermal OOD distance [PCA]')
axs[1, 1].set_title('OOD distribution: normal versus extreme events')
axs[1, 1].grid(True, alpha=0.25)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig(Path("aims_figures") / "09_sensor_extreme_diagnostics.png", dpi=180)
plt.show()


## Audit richiesti da Davide

Questa sezione raccoglie i test diagnostici suggeriti da Davide:

- scala del dataset simulato rispetto al reale;
- copertura delle condizioni difficili negli split;
- confronto tra split cronologico e split stratificato diagnostico;
- ablation di normalizzazione/adaptation semplice sui dati reali;
- calibrazione Sim2Real se sono disponibili predizioni FEM sui campioni reali;
- analisi degli esperti MoE/router se sono disponibili predizioni o pesi degli esperti.

Le celle sono pensate per girare anche quando alcuni artefatti non sono ancora disponibili: in quel caso stampano cosa manca e quale formato usare.

In [ ]:
from pathlib import Path

def count_csv_rows(path, chunksize=500_000):
    n = 0
    for chunk in pd.read_csv(path, chunksize=chunksize):
        n += len(chunk)
    return n

def count_excel_rows_metadata(path):

    try:
        import openpyxl
    except Exception as exc:
        print('openpyxl is unavailable for counting Excel rows:', exc)
        return None
    try:
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
        rows = {}
        for ws in wb.worksheets:

            rows[ws.title] = max(int(ws.max_row or 0) - 1, 0)
        wb.close()
        return rows
    except Exception as exc:
        print('Excel metadata count failed:', exc)
        return None

def dataset_scale_audit():
    rows = []
    real_target_count = int(real_raw[TARGET].notna().sum()) if "real_raw" in globals() and TARGET in real_raw.columns else np.nan
    rows.append({
        "dataset": 'real_raw',
        "rows_or_samples": len(real_raw) if "real_raw" in globals() else np.nan,
        'valid_targets': real_target_count,
        "note": 'thermal time-series rows'
    })
    rows.append({
        "dataset": 'real_windows',
        "rows_or_samples": len(X_seq) if "X_seq" in globals() else np.nan,
        'valid_targets': len(y) if "y" in globals() else np.nan,
        "note": 'supervised samples after cleaning'
    })
    if "train_idx" in globals():
        rows.extend([
            {"dataset": 'real_train', "rows_or_samples": len(train_idx), 'valid_targets': len(train_idx), "note": 'current split'},
            {"dataset": 'real_validation', "rows_or_samples": len(val_idx), 'valid_targets': len(val_idx), "note": 'current split'},
            {"dataset": 'real_test', "rows_or_samples": len(test_idx), 'valid_targets': len(test_idx), "note": 'current split'},
        ])

    fem_info = None
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
        suffix = Path(fem_path).suffix.lower()
        if suffix == ".csv":
            fem_rows = count_csv_rows(fem_path)
            fem_info = fem_rows
            rows.append({"dataset": "FEM", "rows_or_samples": fem_rows, 'valid_targets': np.nan, "note": str(fem_path)})
        elif suffix in [".xlsx", ".xlsm", ".xls"]:
            sheet_rows = count_excel_rows_metadata(fem_path)
            fem_info = sheet_rows
            if isinstance(sheet_rows, dict):
                for sheet, n in sheet_rows.items():
                    rows.append({"dataset": f"FEM:{sheet}", "rows_or_samples": n, 'valid_targets': np.nan, "note": str(fem_path)})
            else:
                rows.append({"dataset": "FEM", "rows_or_samples": np.nan, 'valid_targets': np.nan, "note": f"not counted: {fem_path}"})
        else:
            rows.append({"dataset": "FEM", "rows_or_samples": np.nan, 'valid_targets': np.nan, "note": f"unsupported format: {fem_path}"})
    except FileNotFoundError:
        rows.append({"dataset": "FEM", "rows_or_samples": np.nan, 'valid_targets': np.nan, "note": 'FEM file not found'})

    scale_df = pd.DataFrame(rows)
    display(scale_df)


    fem_counts = scale_df[scale_df["dataset"].str.startswith("FEM")]["rows_or_samples"].dropna()
    if len(fem_counts) and "train_idx" in globals():
        max_fem = float(fem_counts.max())
        print(f"FEM-to-real-training size ratio: {max_fem:.0f} / {len(train_idx)} = {max_fem / max(len(train_idx), 1):.1f}x")

dataset_scale_audit()


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit


if "EXTREME_TARGET_ARCSEC" not in globals():
    EXTREME_TARGET_ARCSEC = 15.0

def make_condition_strata(meta_df, y_values, min_count=3):

    df = meta_df.copy().reset_index(drop=True)
    abs_y = np.abs(np.asarray(y_values, dtype=float))
    elev = pd.to_numeric(df["Elevation"], errors="coerce").to_numpy()

    target_band = pd.cut(abs_y, bins=[0, 3, 7, 15, 30, np.inf], labels=["0-3", "3-7", "7-15", "15-30", ">30"], include_lowest=True)
    elev_band = pd.cut(elev, bins=[0, 20, 30, 45, 60, 75, 90], labels=["0-20", "20-30", "30-45", "45-60", "60-75", "75-90"], include_lowest=True)
    labels = pd.Series(target_band.astype(str) + "_el" + elev_band.astype(str))

    counts = labels.value_counts()
    rare = labels.map(counts) < min_count
    labels.loc[rare] = pd.Series(target_band.astype(str)).loc[rare]

    counts = labels.value_counts()
    rare = labels.map(counts) < min_count
    labels.loc[rare] = np.where(abs_y[rare] > EXTREME_TARGET_ARCSEC, "extreme", "nominal")

    return labels.astype(str).to_numpy()

def split_coverage_table(split_map, meta_df, y_values):
    rows = []
    all_months = sorted(pd.to_datetime(meta_df["Timestamp"]).dt.month.unique())
    for name, idx in split_map.items():
        idx = np.asarray(idx, dtype=int)
        if len(idx) == 0:
            continue
        sub = meta_df.iloc[idx].copy()
        yy = np.asarray(y_values)[idx]
        abs_y = np.abs(yy)
        months = pd.to_datetime(sub["Timestamp"]).dt.month
        rows.append({
            "split": name,
            "count": len(idx),
            "target_mean": float(np.mean(yy)),
            "median_abs_target": float(np.median(abs_y)),
            "p90_abs_target": float(np.quantile(abs_y, 0.90)),
            "extreme_%": float((abs_y > EXTREME_TARGET_ARCSEC).mean() * 100),
            "el>=75_%": float((sub["Elevation"].to_numpy() >= 75).mean() * 100),
            "month_count": int(months.nunique()),
            "months": ",".join(map(str, sorted(months.unique()))),
        })
    return pd.DataFrame(rows)

print('Coverage of the current chronological split')
chrono_coverage = split_coverage_table({"train": train_idx, "val": val_idx, "test": test_idx}, meta, y)
display(chrono_coverage)

labels = make_condition_strata(meta, y)
print('Distribution of the main diagnostic strata')
display(pd.Series(labels).value_counts().head(20).rename_axis("stratum").reset_index(name="count"))

# The stratified split is diagnostic, not the final chronological evaluation.

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_all = np.arange(len(y))
train_val_idx, strat_test_idx = next(sss1.split(idx_all, labels))

labels_train_val = labels[train_val_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED + 1)  
strat_train_rel, strat_val_rel = next(sss2.split(train_val_idx, labels_train_val))
strat_train_idx = train_val_idx[strat_train_rel]
strat_val_idx = train_val_idx[strat_val_rel]

print('Coverage of the diagnostic stratified split')
strat_coverage = split_coverage_table({"train_strat": strat_train_idx, "val_strat": strat_val_idx, "test_strat": strat_test_idx}, meta, y)
display(strat_coverage)

def run_tabular_split(split_name, tr, va, te, include_azimuth=True):
    n_pca = min(8, max(1, len(tr) - 1))
    fb_local = FeatureBuilder(n_pca_current=n_pca, n_pca_trend=n_pca).fit(X_seq[tr], X_elev[tr])
    Xtr = fb_local.transform(X_seq[tr], X_elev[tr], meta.iloc[tr])
    Xva = fb_local.transform(X_seq[va], X_elev[va], meta.iloc[va])
    Xte = fb_local.transform(X_seq[te], X_elev[te], meta.iloc[te])

    if not include_azimuth:
        az_cols_local = [c for c in Xtr.columns if c.startswith("az_")]
        Xtr = Xtr.drop(columns=az_cols_local, errors="ignore")
        Xva = Xva.drop(columns=az_cols_local, errors="ignore")
        Xte = Xte.drop(columns=az_cols_local, errors="ignore")

    yy_tr, yy_va, yy_te = y[tr], y[va], y[te]
    diag_models = {
        "median": None,
        "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))]),
        "huber": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=2000))]),
        "random_forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=6, random_state=RANDOM_SEED, n_jobs=-1))]),
    }
    rows = []
    preds = {}
    for name, model in diag_models.items():
        if model is None:
            pred_va = np.full_like(yy_va, np.median(yy_tr), dtype=float)
            pred_te = np.full_like(yy_te, np.median(yy_tr), dtype=float)
        else:
            m = clone(model)
            m.fit(Xtr, yy_tr)
            pred_va = m.predict(Xva)
            pred_te = m.predict(Xte)
        rv = eval_predictions(f"{split_name}:{name} / val", yy_va, pred_va)
        rt = eval_predictions(f"{split_name}:{name} / test", yy_te, pred_te)
        rt["within_5_%"] = float((np.abs(yy_te - pred_te) <= SPEC_ARCSEC).mean() * 100)
        rv["within_5_%"] = float((np.abs(yy_va - pred_va) <= SPEC_ARCSEC).mean() * 100)
        rows.extend([rv, rt])
        preds[name] = pred_te
    return pd.DataFrame(rows), preds

strat_results_df, strat_test_predictions = run_tabular_split("stratified", strat_train_idx, strat_val_idx, strat_test_idx)
display(strat_results_df.sort_values("MAE"))


In [ ]:
from sklearn.preprocessing import RobustScaler, QuantileTransformer, PowerTransformer

normalization_models = {
    "ridge_standard": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))]),
    "ridge_robust": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))]),
    "ridge_quantile_normal": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scaler", QuantileTransformer(output_distribution="normal", n_quantiles=min(100, len(F_train)), random_state=RANDOM_SEED)),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))
    ]),
    "huber_standard": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000))]),
    "huber_robust": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000))]),
    "huber_power": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", PowerTransformer(method="yeo-johnson", standardize=True)), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000))]),
}

norm_rows = []
for name, model in normalization_models.items():
    try:
        m = clone(model)
        m.fit(F_train, y_train)
        pv = m.predict(F_val)
        pt = m.predict(F_test)
        rv = eval_predictions(f"{name} / val", y_val, pv)
        rt = eval_predictions(f"{name} / test", y_test, pt)
        rv["within_5_%"] = float((np.abs(y_val - pv) <= SPEC_ARCSEC).mean() * 100)
        rt["within_5_%"] = float((np.abs(y_test - pt) <= SPEC_ARCSEC).mean() * 100)
        norm_rows.extend([rv, rt])
    except Exception as exc:
        norm_rows.append({"model": name, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "within_5_%": np.nan, "error": str(exc)})

normalization_ablation_df = pd.DataFrame(norm_rows)
display(normalization_ablation_df.sort_values("MAE"))


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, RobustScaler

SIM2REAL_PREDICTION_CSV = "fem_real_predictions.csv"

def load_fem_real_predictions():
    candidates = []

    if "meta" in globals():
        for c in meta.columns:
            cl = c.lower()
            if ("fem" in cl or "sim" in cl) and ("xel" in cl or "pred" in cl or "offset" in cl):
                candidates.append(("meta", c, meta[c].to_numpy()))


    if "F_all" in globals():
        for c in F_all.columns:
            cl = c.lower()
            if ("fem" in cl or "sim" in cl) and ("xel" in cl or "pred" in cl or "offset" in cl):
                candidates.append(("F_all", c, F_all[c].to_numpy()))


    csv_path = Path(SIM2REAL_PREDICTION_CSV)
    if csv_path.exists() and "meta" in globals():
        pred_df = pd.read_csv(csv_path)
        time_cols = [c for c in pred_df.columns if c.lower() in ["timestamp", "time", "date"]]
        pred_cols = [c for c in pred_df.columns if c.lower() not in [tc.lower() for tc in time_cols]]
        pred_cols = [c for c in pred_cols if any(k in c.lower() for k in ["fem", "sim", "pred", "xel"])]
        if time_cols and pred_cols:
            tmp = pred_df.copy()
            tmp[time_cols[0]] = pd.to_datetime(tmp[time_cols[0]])
            aligned = meta[["Timestamp"]].merge(tmp[[time_cols[0], pred_cols[0]]], left_on="Timestamp", right_on=time_cols[0], how="left")
            candidates.append(("csv", pred_cols[0], aligned[pred_cols[0]].to_numpy()))

    return candidates

fem_pred_candidates = load_fem_real_predictions()
if not fem_pred_candidates:
    print('No FEM predictions aligned with real samples were found.')
    print("To enable this test, create 'fem_real_predictions.csv' with columns Timestamp and fem_xel_pred.")
    print('Alternatively, add a fem_xel_pred column to meta before running this cell.')
else:
    sim2real_rows = []
    for source, col, fem_pred_all in fem_pred_candidates:
        fem_pred_all = np.asarray(fem_pred_all, dtype=float)
        valid = np.isfinite(fem_pred_all) & np.isfinite(y)
        tr = np.asarray(train_idx)[valid[train_idx]]
        va = np.asarray(val_idx)[valid[val_idx]]
        te = np.asarray(test_idx)[valid[test_idx]]
        if len(tr) < 20 or len(te) < 10:
            print(f"Skip {source}:{col}, too few valid samples ({valid.sum()}).")
            continue

        base_train = fem_pred_all[tr].reshape(-1, 1)
        base_val = fem_pred_all[va].reshape(-1, 1)
        base_test = fem_pred_all[te].reshape(-1, 1)


        ctx_cols = [c for c in F_train.columns if c in [
            "elevation", "sin_el", "cos_el", "current_mean", "current_std_spatial",
            "current_range", "window_mean", "window_std", "trend_mean", "trend_std",
            "abs_rate_mean", "az_sin", "az_cos", "hour_sin", "hour_cos", "doy_sin", "doy_cos"
        ]]
        Xcal_train = pd.DataFrame({"fem_pred": fem_pred_all[tr]})
        Xcal_val = pd.DataFrame({"fem_pred": fem_pred_all[va]})
        Xcal_test = pd.DataFrame({"fem_pred": fem_pred_all[te]})
        for cc in ctx_cols:
            Xcal_train[cc] = F_all.iloc[tr][cc].to_numpy() if "F_all" in globals() else fb.transform(X_seq[tr], X_elev[tr], meta.iloc[tr])[cc].to_numpy()
            Xcal_val[cc] = F_all.iloc[va][cc].to_numpy() if "F_all" in globals() else fb.transform(X_seq[va], X_elev[va], meta.iloc[va])[cc].to_numpy()
            Xcal_test[cc] = F_all.iloc[te][cc].to_numpy() if "F_all" in globals() else fb.transform(X_seq[te], X_elev[te], meta.iloc[te])[cc].to_numpy()

        calibration_models = {
            "fem_raw": None,
            "scale_bias": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-6, 3, 30)))]),
            "poly2_fem_context": Pipeline([("imputer", SimpleImputer(strategy="median")), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-6, 3, 30)))]),
            "huber_residual": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000))]),
        }

        for name, model in calibration_models.items():
            if name == "fem_raw":
                pred_val = fem_pred_all[va]
                pred_test = fem_pred_all[te]
            elif name == "huber_residual":

                m = clone(model)
                m.fit(Xcal_train, y[tr] - fem_pred_all[tr])
                pred_val = fem_pred_all[va] + m.predict(Xcal_val)
                pred_test = fem_pred_all[te] + m.predict(Xcal_test)
            else:
                m = clone(model)
                m.fit(Xcal_train, y[tr])
                pred_val = m.predict(Xcal_val)
                pred_test = m.predict(Xcal_test)

            rv = eval_predictions(f"{source}:{col}:{name} / val", y[va], pred_val)
            rt = eval_predictions(f"{source}:{col}:{name} / test", y[te], pred_test)
            rv["within_5_%"] = float((np.abs(y[va] - pred_val) <= SPEC_ARCSEC).mean() * 100)
            rt["within_5_%"] = float((np.abs(y[te] - pred_test) <= SPEC_ARCSEC).mean() * 100)
            sim2real_rows.extend([rv, rt])

    sim2real_calibration_df = pd.DataFrame(sim2real_rows)
    display(sim2real_calibration_df.sort_values("MAE"))


In [ ]:
MOE_DIAGNOSTIC_CSV = "moe_real_predictions.csv"

def get_moe_diagnostics_df():
    if "moe_diag_df" in globals():
        return moe_diag_df.copy(), "memory:moe_diag_df"
    path = Path(MOE_DIAGNOSTIC_CSV)
    if path.exists():
        df = pd.read_csv(path)
        if "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        return df, str(path)
    return None, None

moe_df, moe_source = get_moe_diagnostics_df()

if moe_df is None:
    print('No MoE diagnostics found.')
    print("To enable diagnostics, save 'moe_real_predictions.csv' with columns:")
    print('Timestamp, Target, pred_5, pred_30, pred_60, pred_85; optional: router_pred and gate_5...')
else:
    print('MoE diagnostics from:', moe_source)

    if "Timestamp" in moe_df.columns and "meta" in globals() and len(moe_df) != len(meta):
        align = meta[["Timestamp", "Elevation"]].copy()
        align["_row"] = np.arange(len(align))
        moe_df = align.merge(moe_df, on="Timestamp", how="left")
    elif "Elevation" not in moe_df.columns and "meta" in globals() and len(moe_df) == len(meta):
        moe_df["Elevation"] = meta["Elevation"].to_numpy()

    target_col_candidates = [c for c in ["Target", "y_true", TARGET, "XEl Offset"] if c in moe_df.columns]
    if target_col_candidates:
        target_col_moe = target_col_candidates[0]
        y_moe = pd.to_numeric(moe_df[target_col_moe], errors="coerce").to_numpy()
    elif len(moe_df) == len(y):
        target_col_moe = "y_from_notebook"
        y_moe = y
    else:
        raise ValueError('No target column found in the MoE file; alignment with y is unavailable.')

    pred_cols = [c for c in moe_df.columns if c.lower().startswith("pred_") or c.lower().startswith("expert_")]
    router_cols = [c for c in moe_df.columns if c.lower() in ["router_pred", "moe_pred", "mixture_pred", "ensemble_pred"]]
    gate_cols = [c for c in moe_df.columns if c.lower().startswith("gate_") or c.lower().startswith("w_") or c.lower().startswith("weight_")]

    if len(pred_cols) < 2:
        print('Too few expert columns found:', pred_cols)
        print('At least pred_5, pred_30, pred_60 and pred_85 are required.')
    else:
        valid = np.isfinite(y_moe)
        for c in pred_cols + router_cols:
            valid &= np.isfinite(pd.to_numeric(moe_df[c], errors="coerce").to_numpy())

        moe_eval = moe_df.loc[valid].copy().reset_index(drop=True)
        y_eval = y_moe[valid]
        pred_mat = moe_eval[pred_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)


        if len(moe_df) == len(y) and "train_idx" in globals():
            valid_positions = np.where(valid)[0]
            pos_map = {orig: i for i, orig in enumerate(valid_positions)}
            tr = np.array([pos_map[i] for i in train_idx if i in pos_map], dtype=int)
            va = np.array([pos_map[i] for i in val_idx if i in pos_map], dtype=int)
            te = np.array([pos_map[i] for i in test_idx if i in pos_map], dtype=int)
        else:
            order = np.argsort(pd.to_datetime(moe_eval["Timestamp"]).to_numpy()) if "Timestamp" in moe_eval.columns else np.arange(len(moe_eval))
            n = len(order)
            tr, va, te = order[:int(0.6*n)], order[int(0.6*n):int(0.8*n)], order[int(0.8*n):]

        moe_rows = []
        for j, col in enumerate(pred_cols):
            moe_rows.append({**eval_predictions(f"{col} / test", y_eval[te], pred_mat[te, j]), "type": "expert"})

        mean_pred = pred_mat.mean(axis=1)
        median_pred = np.median(pred_mat, axis=1)
        oracle_pred = pred_mat[np.arange(len(pred_mat)), np.argmin(np.abs(pred_mat - y_eval.reshape(-1, 1)), axis=1)]
        moe_rows.append({**eval_predictions("expert_mean / test", y_eval[te], mean_pred[te]), "type": "ensemble"})
        moe_rows.append({**eval_predictions("expert_median / test", y_eval[te], median_pred[te]), "type": "ensemble"})
        moe_rows.append({**eval_predictions("oracle_best_expert / test", y_eval[te], oracle_pred[te]), "type": "oracle"})


        stack_features = pd.DataFrame(pred_mat, columns=pred_cols)
        if "Elevation" in moe_eval.columns:
            stack_features["elevation"] = pd.to_numeric(moe_eval["Elevation"], errors="coerce").to_numpy()
            stack_features["sin_el"] = np.sin(np.deg2rad(stack_features["elevation"]))
            stack_features["cos_el"] = np.cos(np.deg2rad(stack_features["elevation"]))
        stack_model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-6, 3, 30)))])
        stack_model.fit(stack_features.iloc[tr], y_eval[tr])
        stack_pred = stack_model.predict(stack_features)
        moe_rows.append({**eval_predictions("linear_stack_experts / test", y_eval[te], stack_pred[te]), "type": "calibration"})

        for c in router_cols:
            pred_router = pd.to_numeric(moe_eval[c], errors="coerce").to_numpy(dtype=float)
            moe_rows.append({**eval_predictions(f"{c} / test", y_eval[te], pred_router[te]), "type": "router"})

        moe_results_df = pd.DataFrame(moe_rows).sort_values("MAE")
        display(moe_results_df)


        moe_eval["expert_std"] = pred_mat.std(axis=1)
        if router_cols:
            moe_eval["router_abs_error"] = np.abs(y_eval - pd.to_numeric(moe_eval[router_cols[0]], errors="coerce").to_numpy(dtype=float))
            corr = moe_eval[["expert_std", "router_abs_error"]].corr().iloc[0, 1]
            print(f"Correlation of expert disagreement with router error: {corr:.3f}")
        else:
            moe_eval["mean_abs_error"] = np.abs(y_eval - mean_pred)
            corr = moe_eval[["expert_std", "mean_abs_error"]].corr().iloc[0, 1]
            print(f"Correlation of expert disagreement with expert-mean error: {corr:.3f}")

        display(moe_eval.assign(target=y_eval).sort_values("expert_std", ascending=False).head(15)[
            [c for c in ["Timestamp", "Elevation", "target", "expert_std"] + pred_cols + router_cols + gate_cols if c in moe_eval.columns or c == "target"]
        ])

        fig, axs = plt.subplots(1, 2, figsize=(14, 5))
        axs[0].scatter(y_eval[te], mean_pred[te], s=25, alpha=0.7, label='expert mean')
        if router_cols:
            axs[0].scatter(y_eval[te], pd.to_numeric(moe_eval.iloc[te][router_cols[0]], errors="coerce"), s=25, alpha=0.7, label=router_cols[0])
        lim = np.nanmax(np.abs(np.concatenate([y_eval[te], mean_pred[te]])))
        axs[0].plot([-lim, lim], [-lim, lim], "k--", lw=1)
        axs[0].set_xlabel('Observed target [arcsec]')
        axs[0].set_ylabel('Prediction [arcsec]')
        axs[0].set_title('Experts/router: predicted versus observed')
        axs[0].legend()

        axs[1].scatter(moe_eval.iloc[te]["Elevation"] if "Elevation" in moe_eval.columns else np.arange(len(te)),
                       moe_eval.iloc[te]["expert_std"], c=np.abs(y_eval[te] - mean_pred[te]), cmap="magma", s=35)
        axs[1].set_xlabel('Elevation [deg]' if "Elevation" in moe_eval.columns else 'test index')
        axs[1].set_ylabel('Expert prediction std [arcsec]')
        axs[1].set_title('Expert disagreement')
        plt.tight_layout()
        plt.show()

        if gate_cols:
            gate_plot = moe_eval.iloc[te].copy()
            gate_plot = gate_plot.sort_values("Elevation") if "Elevation" in gate_plot.columns else gate_plot
            plt.figure(figsize=(12, 4))
            x_axis = gate_plot["Elevation"] if "Elevation" in gate_plot.columns else np.arange(len(gate_plot))
            for c in gate_cols:
                plt.plot(x_axis, gate_plot[c], marker="o", ms=3, lw=1, label=c)
            plt.xlabel('Elevation [deg]' if "Elevation" in gate_plot.columns else 'test sample')
            plt.ylabel('Router weight')
            plt.title('Router weights over experts')
            plt.legend()
            plt.grid(True, alpha=0.25)
            plt.show()


In [ ]:
FEM_COVERAGE_MAX_ROWS = 50_000

def load_fem_sensor_sample(max_rows=FEM_COVERAGE_MAX_ROWS):
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        return None, None

    suffix = Path(fem_path).suffix.lower()
    try:
        if suffix == ".csv":
            fem_sample = pd.read_csv(fem_path, nrows=max_rows)
        else:
            fem_sample = pd.read_excel(fem_path, nrows=max_rows)
    except Exception as exc:
        print('Cannot read the FEM sample:', exc)
        return None, fem_path

    fem_sensor_cols = [c for c in fem_sample.columns if str(c).startswith("T_") or str(c).startswith("Sensor_")]
    if not fem_sensor_cols:

        numeric_cols = fem_sample.select_dtypes(include=[np.number]).columns.tolist()
        exclude_tokens = ["offset", "target", "elev", "az", "time"]
        fem_sensor_cols = [c for c in numeric_cols if not any(tok in str(c).lower() for tok in exclude_tokens)]
    if len(fem_sensor_cols) < 20:
        print('FEM sensor columns not recognized. Candidate columns:', fem_sensor_cols[:20])
        return None, fem_path

    return fem_sample[fem_sensor_cols].astype("float32"), fem_path

fem_sensors_sample, fem_path_used = load_fem_sensor_sample()

if fem_sensors_sample is None:
    print('FEM coverage not evaluated: a readable FEM file with sensor columns is required.')
else:
    fem_arr = fem_sensors_sample.to_numpy(dtype="float32")
    real_current = X_seq[:, -1, :].astype("float32")

    fem_cov = pd.DataFrame({
        "domain": "FEM",
        "current_mean": np.nanmean(fem_arr, axis=1),
        "current_std_spatial": np.nanstd(fem_arr, axis=1),
        "current_range": np.nanmax(fem_arr, axis=1) - np.nanmin(fem_arr, axis=1),
    })
    real_cov = pd.DataFrame({
        "domain": "Real",
        "current_mean": np.nanmean(real_current, axis=1),
        "current_std_spatial": np.nanstd(real_current, axis=1),
        "current_range": np.nanmax(real_current, axis=1) - np.nanmin(real_current, axis=1),
        "target_abs": np.abs(y),
        "extreme_target": np.abs(y) > EXTREME_TARGET_ARCSEC,
    })

    display(pd.concat([
        fem_cov.drop(columns=["domain"]).describe(percentiles=[0.1, 0.5, 0.9]).assign(domain="FEM"),
        real_cov[["current_mean", "current_std_spatial", "current_range"]].describe(percentiles=[0.1, 0.5, 0.9]).assign(domain="Real")
    ]))


    from sklearn.neighbors import NearestNeighbors
    cov_cols = ["current_mean", "current_std_spatial", "current_range"]
    scaler_cov = StandardScaler().fit(fem_cov[cov_cols])
    fem_scaled = scaler_cov.transform(fem_cov[cov_cols])
    real_scaled = scaler_cov.transform(real_cov[cov_cols])
    nn = NearestNeighbors(n_neighbors=min(10, len(fem_scaled))).fit(fem_scaled)
    dist, _ = nn.kneighbors(real_scaled)
    real_cov["fem_nn_dist"] = dist[:, 0]

    print('Real-to-FEM distance in aggregate thermal space')
    display(real_cov.groupby("extreme_target").agg(
        count=("fem_nn_dist", "size"),
        median_dist=("fem_nn_dist", "median"),
        p90_dist=("fem_nn_dist", lambda s: s.quantile(0.90)),
        mean_temp=("current_mean", "mean"),
        std_spatial=("current_std_spatial", "mean"),
        range_temp=("current_range", "mean"),
    ))

    ood_threshold = real_cov["fem_nn_dist"].quantile(0.90)
    hard_real_states = real_cov[real_cov["fem_nn_dist"] >= ood_threshold].copy()
    hard_real_states = hard_real_states.join(meta[["Timestamp", "Elevation"]].reset_index(drop=True))
    print(f"Real states farthest from FEM: top {len(hard_real_states)} above the distance p90")
    display(hard_real_states.sort_values("fem_nn_dist", ascending=False).head(20))

    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    for ax, col in zip(axs, cov_cols):
        ax.hist(fem_cov[col], bins=50, alpha=0.45, density=True, label="FEM")
        ax.hist(real_cov[col], bins=50, alpha=0.45, density=True, label="Real")
        ax.set_title(col)
        ax.grid(True, alpha=0.25)
    axs[0].legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.scatter(real_cov["current_mean"], real_cov["current_std_spatial"], c=real_cov["target_abs"], cmap="magma", s=28, alpha=0.8)
    plt.colorbar(label='Observed |XEL| [arcsec]')
    plt.xlabel('Current mean temperature [C]')
    plt.ylabel('Current spatial std [C]')
    plt.title('Real states: target severity in thermal space')
    plt.grid(True, alpha=0.25)
    plt.show()


## Diagnostica MoE diretta dagli esperti FEM

Questa sezione riprende il test MoE del notebook originale, ma lo rende diagnostico:

- carica i 4 esperti FEM e i rispettivi scaler;
- predice ogni esperto sui campioni reali puliti del notebook corrente;
- valuta ogni esperto singolarmente;
- valuta media, interpolazione fisica per elevazione, oracle-best-expert e stacking lineare;
- opzionalmente addestra il router elevation-only come nel vecchio notebook e ne misura il mode collapse.

Il punto chiave è separare tre casi: esperti FEM non informativi, router che sceglie male, oppure esperti informativi ma da calibrare.

In [ ]:
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import RidgeCV, HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


MOE_MODEL_DIRS = [Path("/content/metrologyml"), Path.cwd()]
MOE_EXPERT_ELEVS = [5, 30, 60, 85]
RUN_DIRECT_MOE_DIAGNOSTIC = True
RUN_UNCONSTRAINED_ROUTER_RETRAIN = True  

def find_first_existing(patterns, dirs=MOE_MODEL_DIRS):
    for d in dirs:
        for p in patterns:
            candidate = d / p
            if candidate.exists():
                return candidate
    return None

def moe_artifact_paths(elev):
    model_path = find_first_existing([
        f"gru_expert_{elev}.h5",
        f"gru_expert_{elev}.keras",
        f"expert_{elev}.h5",
        f"expert_{elev}.keras",
    ])
    scaler_x_path = find_first_existing([
        f"scaler_X_{elev}.pkl",
        f"scaler_x_{elev}.pkl",
        f"scaler_X_{elev}.joblib",
        f"scaler_x_{elev}.joblib",
    ])
    scaler_y_path = find_first_existing([
        f"scaler_y_{elev}.pkl",
        f"scaler_Y_{elev}.pkl",
        f"scaler_y_{elev}.joblib",
        f"scaler_Y_{elev}.joblib",
    ])
    return model_path, scaler_x_path, scaler_y_path

def deterministic_elevation_weights(elev_values, expert_elevs=MOE_EXPERT_ELEVS):
    """Interpolate adjacent elevation experts; use the nearest expert outside their range."""
    elev_values = np.asarray(elev_values, dtype=float).reshape(-1)
    expert_elevs = np.asarray(expert_elevs, dtype=float)
    W = np.zeros((len(elev_values), len(expert_elevs)), dtype=float)
    for i, el in enumerate(elev_values):
        if el <= expert_elevs[0]:
            W[i, 0] = 1.0
        elif el >= expert_elevs[-1]:
            W[i, -1] = 1.0
        else:
            hi = np.searchsorted(expert_elevs, el)
            lo = hi - 1
            t = (el - expert_elevs[lo]) / (expert_elevs[hi] - expert_elevs[lo])
            W[i, lo] = 1.0 - t
            W[i, hi] = t
    return W

if not RUN_DIRECT_MOE_DIAGNOSTIC:
    print('RUN_DIRECT_MOE_DIAGNOSTIC=False: cell skipped.')
else:
    artifact_rows = []
    missing = []
    for elev in MOE_EXPERT_ELEVS:
        model_path, scaler_x_path, scaler_y_path = moe_artifact_paths(elev)
        artifact_rows.append({
            "expert": elev,
            "model": str(model_path) if model_path else None,
            "scaler_X": str(scaler_x_path) if scaler_x_path else None,
            "scaler_y": str(scaler_y_path) if scaler_y_path else None,
        })
        if model_path is None or scaler_x_path is None or scaler_y_path is None:
            missing.append(elev)
    artifacts_df = pd.DataFrame(artifact_rows)
    display(artifacts_df)

    if missing:
        print('Missing expert artifacts:', missing)
        print('Expected files include /content/metrologyml/gru_expert_30.h5, scaler_X_30.pkl and scaler_y_30.pkl.')
    else:
        import joblib
        import tensorflow as tf
        from tensorflow.keras.models import load_model, Model
        from tensorflow.keras.layers import Input, Dense, Multiply, Add, Lambda

        tf.keras.backend.clear_session()
        expert_models = {}
        expert_scalers_X = {}
        expert_scalers_y = {}

        print('Loading FEM experts and scalers...')
        for elev in MOE_EXPERT_ELEVS:
            model_path, scaler_x_path, scaler_y_path = moe_artifact_paths(elev)
            try:
                expert_models[elev] = load_model(
                    model_path,
                    custom_objects={"mae": tf.keras.metrics.MeanAbsoluteError()},
                    compile=False,
                )
            except Exception:
                expert_models[elev] = load_model(
                    model_path,
                    custom_objects={"mae": tf.keras.metrics.MeanAbsoluteError()},
                )
            expert_scalers_X[elev] = joblib.load(scaler_x_path)
            expert_scalers_y[elev] = joblib.load(scaler_y_path)

        print('Predicting expert outputs for clean real samples...')
        X_real = X_seq.astype("float32")
        n_samples, t_steps, n_features = X_real.shape
        expert_pred = {}

        for elev in MOE_EXPERT_ELEVS:
            print(f" -> Expert {elev} deg")
            X_flat = X_real.reshape(-1, n_features)
            X_scaled_flat = expert_scalers_X[elev].transform(X_flat).astype("float32")
            X_scaled = X_scaled_flat.reshape(n_samples, t_steps, n_features)
            pred_scaled = expert_models[elev].predict(X_scaled, verbose=0)
            pred_arcsec = expert_scalers_y[elev].inverse_transform(pred_scaled).reshape(-1)
            expert_pred[elev] = pred_arcsec

        pred_cols = [f"pred_{e}" for e in MOE_EXPERT_ELEVS]
        pred_mat = np.column_stack([expert_pred[e] for e in MOE_EXPERT_ELEVS])
        elev_real = meta["Elevation"].to_numpy(dtype=float)
        y_real = y.astype(float)

        moe_direct_df = meta[["Timestamp", "Elevation", "Target"]].copy()
        for j, col in enumerate(pred_cols):
            moe_direct_df[col] = pred_mat[:, j]
        moe_direct_df["expert_std"] = pred_mat.std(axis=1)
        moe_direct_df["expert_range"] = pred_mat.max(axis=1) - pred_mat.min(axis=1)


        moe_direct_df.rename(columns={"Target": "Target"}).to_csv("moe_real_predictions_from_experts.csv", index=False)
        print('Diagnostic CSV saved: moe_real_predictions_from_experts.csv')

        tr, va, te = np.asarray(train_idx), np.asarray(val_idx), np.asarray(test_idx)
        rows = []

        for j, e in enumerate(MOE_EXPERT_ELEVS):
            rows.append({**eval_predictions(f"expert_{e} / test", y_real[te], pred_mat[te, j]), "kind": "single_expert"})

        pred_mean = pred_mat.mean(axis=1)
        pred_median = np.median(pred_mat, axis=1)
        W_det = deterministic_elevation_weights(elev_real)
        pred_det_interp = np.sum(W_det * pred_mat, axis=1)
        oracle_idx = np.argmin(np.abs(pred_mat - y_real.reshape(-1, 1)), axis=1)
        pred_oracle = pred_mat[np.arange(len(y_real)), oracle_idx]

        rows.extend([
            {**eval_predictions("expert_mean / test", y_real[te], pred_mean[te]), "kind": "ensemble"},
            {**eval_predictions("expert_median / test", y_real[te], pred_median[te]), "kind": "ensemble"},
            {**eval_predictions("deterministic_elevation_interp / test", y_real[te], pred_det_interp[te]), "kind": "physical_router"},
            {**eval_predictions("oracle_best_expert / test", y_real[te], pred_oracle[te]), "kind": "oracle"},
        ])


        X_stack = pd.DataFrame(pred_mat, columns=pred_cols)
        X_stack["elevation"] = elev_real
        X_stack["sin_el"] = np.sin(np.deg2rad(elev_real))
        X_stack["cos_el"] = np.cos(np.deg2rad(elev_real))
        if "Azimuth" in meta.columns:
            az_real = pd.to_numeric(meta["Azimuth"], errors="coerce").to_numpy(dtype=float)
            X_stack["az_sin"] = np.sin(np.deg2rad(az_real))
            X_stack["az_cos"] = np.cos(np.deg2rad(az_real))

        stack_models = {
            "ridge_stack_experts": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", RidgeCV(alphas=np.logspace(-6, 3, 30))),
            ]),
            "huber_stack_experts": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000)),
            ]),
        }
        stack_predictions = {}
        for name, model in stack_models.items():
            m = clone(model)
            m.fit(X_stack.iloc[tr], y_real[tr])
            p = m.predict(X_stack)
            stack_predictions[name] = p
            rows.append({**eval_predictions(f"{name} / val", y_real[va], p[va]), "kind": "calibration"})
            rows.append({**eval_predictions(f"{name} / test", y_real[te], p[te]), "kind": "calibration"})


        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            elev_train_scaled = (elev_real[tr] / 90.0).reshape(-1, 1).astype("float32")
            elev_val_scaled = (elev_real[va] / 90.0).reshape(-1, 1).astype("float32")
            elev_test_scaled = (elev_real[te] / 90.0).reshape(-1, 1).astype("float32")

            elevation_input = Input(shape=(1,), name="elevation_input")
            exp_inputs = [Input(shape=(1,), name=f"pred_expert_{e}") for e in MOE_EXPERT_ELEVS]
            x = Dense(32, activation="relu")(elevation_input)
            x = Dense(16, activation="relu")(x)
            gating_weights = Dense(4, activation="softmax", name="gating_weights")(x)
            weighted = []
            for j, inp in enumerate(exp_inputs):
                wj = Lambda(lambda w, jj=j: w[:, jj:jj+1], name=f"weight_{MOE_EXPERT_ELEVS[j]}")(gating_weights)
                weighted.append(Multiply()([inp, wj]))
            final_prediction = Add(name="final_prediction")(weighted)
            router_model = Model(inputs=[elevation_input] + exp_inputs, outputs=final_prediction)
            router_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss="mae", metrics=["mae"])

            callbacks = [
                tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-5),
            ]
            router_model.fit(
                x=[elev_train_scaled] + [pred_mat[tr, j].reshape(-1, 1).astype("float32") for j in range(4)],
                y=y_real[tr].reshape(-1, 1).astype("float32"),
                validation_data=(
                    [elev_val_scaled] + [pred_mat[va, j].reshape(-1, 1).astype("float32") for j in range(4)],
                    y_real[va].reshape(-1, 1).astype("float32"),
                ),
                batch_size=16,
                epochs=200,
                callbacks=callbacks,
                verbose=0,
            )
            pred_router_test = router_model.predict(
                [elev_test_scaled] + [pred_mat[te, j].reshape(-1, 1).astype("float32") for j in range(4)],
                verbose=0,
            ).reshape(-1)
            pred_router_val = router_model.predict(
                [elev_val_scaled] + [pred_mat[va, j].reshape(-1, 1).astype("float32") for j in range(4)],
                verbose=0,
            ).reshape(-1)
            rows.append({**eval_predictions("unconstrained_router / val", y_real[va], pred_router_val), "kind": "learned_router"})
            rows.append({**eval_predictions("unconstrained_router / test", y_real[te], pred_router_test), "kind": "learned_router"})

            gating_model = Model(inputs=router_model.inputs[0], outputs=router_model.get_layer("gating_weights").output)
            elev_grid = np.linspace(5, 85, 200)
            W_grid = gating_model.predict((elev_grid / 90.0).reshape(-1, 1), verbose=0)
            W_real = gating_model.predict((elev_real / 90.0).reshape(-1, 1), verbose=0)
            for j, e in enumerate(MOE_EXPERT_ELEVS):
                moe_direct_df[f"gate_{e}"] = W_real[:, j]
            moe_direct_df["router_pred_unconstrained"] = np.nan
            moe_direct_df.loc[va, "router_pred_unconstrained"] = pred_router_val
            moe_direct_df.loc[te, "router_pred_unconstrained"] = pred_router_test

            collapse_df = pd.DataFrame({
                "expert": MOE_EXPERT_ELEVS,
                "mean_weight_real": W_real.mean(axis=0),
                "mean_weight_test": W_real[te].mean(axis=0),
                "max_weight_real": W_real.max(axis=0),
                "fraction_weight_gt_0.8_real": (W_real > 0.8).mean(axis=0),
            })
            print('Router mode-collapse diagnostics')
            display(collapse_df)

            plt.figure(figsize=(10, 5))
            colors = ["red", "orange", "green", "blue"]
            for j, e in enumerate(MOE_EXPERT_ELEVS):
                plt.plot(elev_grid, W_grid[:, j], label=f"Expert weight {e} deg", color=colors[j], lw=2.5)
                plt.axvline(e, color="black", linestyle="--", alpha=0.25)
            plt.xlabel('Elevation [deg]')
            plt.ylabel('Router weight')
            plt.title('Elevation-only router: learned weights')
            plt.grid(True, alpha=0.25)
            plt.legend()
            plt.tight_layout()
            plt.show()

        moe_direct_results_df = pd.DataFrame(rows)
        moe_direct_results_df["within_5_%"] = [
            np.nan if not str(row["model"]).endswith("/ test") else
            float((np.abs(y_real[te] - (
                pred_mat[te, int(row["model"].split("_")[1].split(" ")[0] == "30")] if False else y_real[te]
            )) <= SPEC_ARCSEC).mean() * 100)
            for _, row in moe_direct_results_df.iterrows()
        ]

        within_map = {}
        for j, e in enumerate(MOE_EXPERT_ELEVS):
            within_map[f"expert_{e} / test"] = (np.abs(y_real[te] - pred_mat[te, j]) <= SPEC_ARCSEC).mean() * 100
        within_map["expert_mean / test"] = (np.abs(y_real[te] - pred_mean[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["expert_median / test"] = (np.abs(y_real[te] - pred_median[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["deterministic_elevation_interp / test"] = (np.abs(y_real[te] - pred_det_interp[te]) <= SPEC_ARCSEC).mean() * 100
        within_map["oracle_best_expert / test"] = (np.abs(y_real[te] - pred_oracle[te]) <= SPEC_ARCSEC).mean() * 100
        for name, p in stack_predictions.items():
            within_map[f"{name} / test"] = (np.abs(y_real[te] - p[te]) <= SPEC_ARCSEC).mean() * 100
        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            within_map["unconstrained_router / test"] = (np.abs(y_real[te] - pred_router_test) <= SPEC_ARCSEC).mean() * 100
        moe_direct_results_df["within_5_%"] = moe_direct_results_df["model"].map(within_map)

        display(moe_direct_results_df.sort_values("MAE"))

        print('Expert disagreement versus aggregator errors')
        err_corr_rows = []
        aggregators = {
            "mean": pred_mean,
            "deterministic_interp": pred_det_interp,
        }
        if RUN_UNCONSTRAINED_ROUTER_RETRAIN:
            router_full_pred = np.full_like(y_real, np.nan, dtype=float)
            router_full_pred[va] = pred_router_val
            router_full_pred[te] = pred_router_test
            aggregators["unconstrained_router_val_test"] = router_full_pred
        for name, p in aggregators.items():
            mask = np.isfinite(p)
            err_corr_rows.append({
                "aggregator": name,
                "corr_expert_std_abs_error": float(np.corrcoef(moe_direct_df.loc[mask, "expert_std"], np.abs(y_real[mask] - p[mask]))[0, 1]),
                "corr_expert_range_abs_error": float(np.corrcoef(moe_direct_df.loc[mask, "expert_range"], np.abs(y_real[mask] - p[mask]))[0, 1]),
            })
        display(pd.DataFrame(err_corr_rows))

        print('Worst test cases for the deterministic aggregator')
        worst_moe = meta.iloc[te][["Timestamp", "Elevation", "Target"]].copy()
        worst_moe["pred_det_interp"] = pred_det_interp[te]
        worst_moe["abs_error_det_interp"] = np.abs(y_real[te] - pred_det_interp[te])
        worst_moe["expert_std"] = moe_direct_df.iloc[te]["expert_std"].to_numpy()
        display(worst_moe.sort_values("abs_error_det_interp", ascending=False).head(15))


## Shift characterization: FEM vs Real

Questa sezione risponde direttamente alla domanda di Davide:

- siamo fuori distribuzione sugli input termici?
- oppure, anche in regioni apparentemente coperte dal FEM, la relazione input-target cambia?
- quali regioni del dominio richiedono adattamento e quali richiedono una sovrascrittura del comportamento FEM?

Le celle sotto costruiscono una rappresentazione compatta comune FEM/reale, addestrano un domain classifier, misurano overlap/supporto, confrontano label reali con vicini FEM e producono una matrice decisionale per regione operativa.

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors

SHIFT_FEM_MAX_ROWS = 120_000
SHIFT_NN_NEIGHBORS = 25
SHIFT_RANDOM_SEED = RANDOM_SEED if "RANDOM_SEED" in globals() else 42

def infer_fem_columns(fem_df):
    sensor_like = [c for c in fem_df.columns if str(c).startswith("Sensors_") or str(c).startswith("Sensor_") or str(c).startswith("T_")]
    if len(sensor_like) >= 20:
        sensor_cols_fem = sensor_like
    else:
        numeric_cols = fem_df.select_dtypes(include=[np.number]).columns.tolist()
        exclude_tokens = ["offset", "accuracy", "target", "elev", "az", "time", "date", "xel", "el"]
        sensor_cols_fem = [c for c in numeric_cols if not any(tok in str(c).lower() for tok in exclude_tokens)]

    elev_candidates = [c for c in fem_df.columns if str(c).lower() in ["elevation", "srt elevation ange[deg]", "pointingelevation"]]
    elev_col = elev_candidates[0] if elev_candidates else None

    target_col = None
    if "XEL" in fem_df.columns:
        target_col = "XEL"
    elif "Accuracy_1" in fem_df.columns and elev_col is not None:
        fem_df = fem_df.copy()
        fem_df["XEL"] = fem_df["Accuracy_1"] * np.cos(np.deg2rad(fem_df[elev_col]))
        target_col = "XEL"
    else:
        for c in fem_df.columns:
            cl = str(c).lower()
            if "xel" in cl or ("offset" in cl and "az" in cl):
                target_col = c
                break
    return fem_df, sensor_cols_fem, elev_col, target_col

def load_fem_for_shift(max_rows=SHIFT_FEM_MAX_ROWS):
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        print('FEM file not found.')
        return None, None, None, None, None

    print('Loading the FEM sample for shift characterization:', fem_path)
    try:
        if Path(fem_path).suffix.lower() == ".csv":
            fem_df = pd.read_csv(fem_path, nrows=max_rows)
        else:
            fem_df = pd.read_excel(fem_path, nrows=max_rows)
    except Exception as exc:
        print('FEM loading failed:', exc)
        return None, fem_path, None, None, None

    fem_df, fem_sensor_cols, fem_elev_col, fem_target_col = infer_fem_columns(fem_df)
    print("FEM sensor cols:", len(fem_sensor_cols), "| elev:", fem_elev_col, "| target:", fem_target_col)
    if len(fem_sensor_cols) < 20:
        print('Too few FEM sensor columns were recognized.')
        return None, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col
    return fem_df, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col

def aggregate_sensor_state(arr_2d, prefix=""):
    arr_2d = np.asarray(arr_2d, dtype="float32")
    return pd.DataFrame({
        prefix + "temp_mean": np.nanmean(arr_2d, axis=1),
        prefix + "temp_std_spatial": np.nanstd(arr_2d, axis=1),
        prefix + "temp_min": np.nanmin(arr_2d, axis=1),
        prefix + "temp_max": np.nanmax(arr_2d, axis=1),
        prefix + "temp_range": np.nanmax(arr_2d, axis=1) - np.nanmin(arr_2d, axis=1),
        prefix + "temp_p10": np.nanpercentile(arr_2d, 10, axis=1),
        prefix + "temp_p50": np.nanpercentile(arr_2d, 50, axis=1),
        prefix + "temp_p90": np.nanpercentile(arr_2d, 90, axis=1),
        prefix + "temp_iqr": np.nanpercentile(arr_2d, 75, axis=1) - np.nanpercentile(arr_2d, 25, axis=1),
    })

def build_shift_feature_tables():
    fem_df, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col = load_fem_for_shift()
    if fem_df is None:
        return None


    real_current = X_seq[:, -1, :]
    real_start = X_seq[:, 0, :]
    real_trend = real_current - real_start
    real_feat = aggregate_sensor_state(real_current)
    trend_feat = aggregate_sensor_state(real_trend, prefix="trend_")

    real_feat = pd.concat([real_feat, trend_feat], axis=1)
    real_feat["elevation"] = meta["Elevation"].to_numpy(dtype=float)
    if "Azimuth" in meta.columns:
        az = pd.to_numeric(meta["Azimuth"], errors="coerce").to_numpy(dtype=float)
        real_feat["az_sin"] = np.sin(np.deg2rad(az))
        real_feat["az_cos"] = np.cos(np.deg2rad(az))
    real_feat["domain"] = "real"
    real_feat["target"] = y.astype(float)
    real_feat["abs_target"] = np.abs(y.astype(float))
    real_feat["Timestamp"] = meta["Timestamp"].to_numpy()


    fem_sensors = fem_df[fem_sensor_cols].to_numpy(dtype="float32")
    fem_feat = aggregate_sensor_state(fem_sensors)
    for c in [c for c in real_feat.columns if c.startswith("trend_")]:
        fem_feat[c] = np.nan
    if fem_elev_col is not None:
        fem_feat["elevation"] = pd.to_numeric(fem_df[fem_elev_col], errors="coerce").to_numpy(dtype=float)
    else:
        fem_feat["elevation"] = np.nan

    if "az_sin" in real_feat.columns:
        fem_feat["az_sin"] = np.nan
        fem_feat["az_cos"] = np.nan
    fem_feat["domain"] = "fem"
    if fem_target_col is not None:
        fem_feat["target"] = pd.to_numeric(fem_df[fem_target_col], errors="coerce").to_numpy(dtype=float)
    else:
        fem_feat["target"] = np.nan
    fem_feat["abs_target"] = np.abs(fem_feat["target"].to_numpy(dtype=float))
    fem_feat["Timestamp"] = pd.NaT

    common_cols = [c for c in real_feat.columns if c in fem_feat.columns]
    real_feat = real_feat[common_cols].copy()
    fem_feat = fem_feat[common_cols].copy()


    feature_cols = [c for c in common_cols if c not in ["domain", "target", "abs_target", "Timestamp"]]
    shift_data = {
        "real_feat": real_feat,
        "fem_feat": fem_feat,
        "feature_cols": feature_cols,
        "fem_path": fem_path,
        "fem_target_col": fem_target_col,
    }
    return shift_data

shift_data = build_shift_feature_tables()
if shift_data is not None:
    real_shift_feat = shift_data["real_feat"]
    fem_shift_feat = shift_data["fem_feat"]
    shift_feature_cols = shift_data["feature_cols"]
    print('Shared features for shift analysis:', shift_feature_cols)
    display(pd.concat([
        real_shift_feat.assign(domain="real")[shift_feature_cols + ["target", "domain"]].describe().assign(domain_summary="real"),
        fem_shift_feat.assign(domain="fem")[shift_feature_cols + ["target", "domain"]].describe().assign(domain_summary="fem"),
    ]))


In [ ]:
if "shift_data" not in globals() or shift_data is None:
    print('Shift data is unavailable; run the preceding cell.')
else:

    rng = np.random.default_rng(SHIFT_RANDOM_SEED)
    n_real = len(real_shift_feat)
    n_fem = len(fem_shift_feat)
    fem_sample_idx = rng.choice(n_fem, size=min(n_fem, max(n_real * 3, n_real)), replace=False)
    fem_sample = fem_shift_feat.iloc[fem_sample_idx].copy()
    real_sample = real_shift_feat.copy()

    domain_df = pd.concat([real_sample, fem_sample], ignore_index=True)
    domain_y = (domain_df["domain"] == "real").astype(int).to_numpy()
    X_domain = domain_df[shift_feature_cols].copy()

    Xtr, Xte, ytr, yte = train_test_split(
        X_domain, domain_y, test_size=0.30, stratify=domain_y, random_state=SHIFT_RANDOM_SEED
    )

    domain_clf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=600,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SHIFT_RANDOM_SEED,
            n_jobs=-1,
        )),
    ])
    domain_clf.fit(Xtr, ytr)
    proba = domain_clf.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(yte, proba)
    acc = accuracy_score(yte, pred)
    print(f"Domain classifier FEM-vs-Real: AUC={auc:.3f} | accuracy={acc:.3f}")
    print(classification_report(yte, pred, target_names=["FEM", "Real"]))


    perm = permutation_importance(
        domain_clf, Xte, yte,
        n_repeats=20,
        random_state=SHIFT_RANDOM_SEED,
        scoring="roc_auc",
        n_jobs=-1,
    )
    domain_importance_df = pd.DataFrame({
        "feature": shift_feature_cols,
        "importance_mean_auc_drop": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean_auc_drop", ascending=False)
    display(domain_importance_df.head(20))


    real_shift_feat["domain_score_real_proba"] = domain_clf.predict_proba(real_shift_feat[shift_feature_cols])[:, 1]
    fem_shift_feat["domain_score_real_proba"] = domain_clf.predict_proba(fem_shift_feat[shift_feature_cols])[:, 1]
    print("Domain-score distribution: probability of 'real'")
    display(pd.DataFrame({
        "real": real_shift_feat["domain_score_real_proba"].describe(percentiles=[.1, .25, .5, .75, .9]),
        "fem": fem_shift_feat["domain_score_real_proba"].describe(percentiles=[.1, .25, .5, .75, .9]),
    }))

    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    axs[0].hist(fem_shift_feat["domain_score_real_proba"], bins=40, alpha=0.55, density=True, label="FEM")
    axs[0].hist(real_shift_feat["domain_score_real_proba"], bins=40, alpha=0.55, density=True, label="Real")
    axs[0].set_xlabel('P(real) from the domain classifier')
    axs[0].set_title('FEM-versus-real input separability')
    axs[0].legend()
    axs[0].grid(True, alpha=0.25)

    top_imp = domain_importance_df.head(12).iloc[::-1]
    axs[1].barh(top_imp["feature"], top_imp["importance_mean_auc_drop"])
    axs[1].set_xlabel('Permutation AUC drop')
    axs[1].set_title('Features localizing input shift')
    axs[1].grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_data" not in globals() or shift_data is None:
    print('Shift data is unavailable.')
else:
    support_cols = [c for c in shift_feature_cols if c not in ["az_sin", "az_cos"]]
    # Estimate support using features shared with FEM; do not impute azimuth.
    fem_support = fem_shift_feat[support_cols].copy()
    real_support = real_shift_feat[support_cols].copy()

    support_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    fem_scaled = support_pipe.fit_transform(fem_support)
    real_scaled = support_pipe.transform(real_support)

    nn = NearestNeighbors(n_neighbors=min(SHIFT_NN_NEIGHBORS, len(fem_scaled))).fit(fem_scaled)
    dist, nn_idx = nn.kneighbors(real_scaled)
    real_shift_feat["fem_nn_dist_1"] = dist[:, 0]
    real_shift_feat["fem_nn_dist_med"] = np.median(dist, axis=1)

    # Calibrate support thresholds against FEM-to-FEM distances.
    fem_ref_idx = np.random.default_rng(SHIFT_RANDOM_SEED).choice(
        len(fem_scaled), size=min(20_000, len(fem_scaled)), replace=False
    )
    nn_fem = NearestNeighbors(n_neighbors=2).fit(fem_scaled)
    fem_self_dist, _ = nn_fem.kneighbors(fem_scaled[fem_ref_idx])
    fem_neighbor_dist = fem_self_dist[:, 1]
    thr_p95 = np.quantile(fem_neighbor_dist, 0.95)
    thr_p99 = np.quantile(fem_neighbor_dist, 0.99)
    real_shift_feat["support_region"] = np.select(
        [
            real_shift_feat["fem_nn_dist_1"] <= thr_p95,
            real_shift_feat["fem_nn_dist_1"] <= thr_p99,
        ],
        ["inside_fem_p95", "border_fem_p99"],
        default="outside_fem_p99",
    )

    print(f"FEM support-distance thresholds: p95={thr_p95:.3f}, p99={thr_p99:.3f}")
    support_summary = real_shift_feat.groupby("support_region").agg(
        count=("target", "size"),
        coverage_pct=("target", lambda s: len(s) / len(real_shift_feat) * 100),
        median_abs_target=("abs_target", "median"),
        p90_abs_target=("abs_target", lambda s: s.quantile(0.90)),
        extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
        median_domain_score=("domain_score_real_proba", "median") if "domain_score_real_proba" in real_shift_feat.columns else ("abs_target", "median"),
    ).reset_index()
    display(support_summary)


    if "moe_direct_df" in globals() and "pred_85" in moe_direct_df.columns:
        support_eval = real_shift_feat[["support_region"]].copy()
        support_eval["y_true"] = y
        for col in [c for c in moe_direct_df.columns if c.startswith("pred_")]:
            support_eval[col] = moe_direct_df[col].to_numpy()
        if "router_pred_unconstrained" in moe_direct_df.columns:
            support_eval["router_pred_unconstrained"] = moe_direct_df["router_pred_unconstrained"].to_numpy()
        rows = []
        for region, g in support_eval.groupby("support_region"):
            for pred_col in [c for c in support_eval.columns if c.startswith("pred_") or c == "router_pred_unconstrained"]:
                mask = g[pred_col].notna()
                if mask.sum() < 5:
                    continue
                rows.append({
                    "support_region": region,
                    "model": pred_col,
                    "count": int(mask.sum()),
                    "MAE": float(mean_absolute_error(g.loc[mask, "y_true"], g.loc[mask, pred_col])),
                    "within_5_%": float((np.abs(g.loc[mask, "y_true"] - g.loc[mask, pred_col]) <= SPEC_ARCSEC).mean() * 100),
                })
        if rows:
            print('MoE/FEM error by support region')
            display(pd.DataFrame(rows).sort_values(["support_region", "MAE"]))

    fig, ax = plt.subplots(figsize=(8, 4))
    for region, g in real_shift_feat.groupby("support_region"):
        ax.hist(g["abs_target"], bins=30, alpha=0.55, density=True, label=f"{region} n={len(g)}")
    ax.axvline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--", label='extreme-event threshold')
    ax.set_xlabel('|Observed XEL| [arcsec]')
    ax.set_title('Observed target magnitude inside/outside FEM support')
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_data" not in globals() or shift_data is None:
    print('Shift data is unavailable.')
elif fem_shift_feat["target"].notna().sum() < 100:
    print('FEM targets are unavailable or insufficient; cannot compare FEM and real P(y|X).')
else:
    cond_cols = [c for c in shift_feature_cols if c not in ["az_sin", "az_cos"]]

    cond_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    fem_valid = fem_shift_feat[fem_shift_feat["target"].notna()].copy()
    X_fem_cond = cond_pipe.fit_transform(fem_valid[cond_cols])
    X_real_cond = cond_pipe.transform(real_shift_feat[cond_cols])

    nn_cond = NearestNeighbors(n_neighbors=min(SHIFT_NN_NEIGHBORS, len(fem_valid))).fit(X_fem_cond)
    dist_cond, idx_cond = nn_cond.kneighbors(X_real_cond)
    fem_targets_neighbors = fem_valid["target"].to_numpy(dtype=float)[idx_cond]
    fem_neighbor_pred_median = np.nanmedian(fem_targets_neighbors, axis=1)
    fem_neighbor_pred_mean = np.nanmean(fem_targets_neighbors, axis=1)

    real_shift_feat["fem_nn_target_median"] = fem_neighbor_pred_median
    real_shift_feat["fem_nn_target_mean"] = fem_neighbor_pred_mean
    real_shift_feat["conditional_residual_median"] = real_shift_feat["target"] - real_shift_feat["fem_nn_target_median"]
    real_shift_feat["conditional_abs_residual_median"] = real_shift_feat["conditional_residual_median"].abs()
    real_shift_feat["conditional_nn_dist"] = dist_cond[:, 0]

    print('Local P(y|X): observed targets versus neighboring FEM targets')
    conditional_summary = real_shift_feat.groupby("support_region" if "support_region" in real_shift_feat.columns else real_shift_feat["abs_target"] > EXTREME_TARGET_ARCSEC).agg(
        count=("target", "size"),
        mae_fem_neighbor_median=("conditional_abs_residual_median", "mean"),
        median_abs_residual=("conditional_abs_residual_median", "median"),
        p90_abs_residual=("conditional_abs_residual_median", lambda s: s.quantile(0.90)),
        extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
    ).reset_index()
    display(conditional_summary)

    print('Real samples sharing FEM neighbors but having different labels')
    cols_show = ["Timestamp", "elevation", "target", "abs_target", "fem_nn_target_median", "conditional_residual_median", "conditional_nn_dist"]
    if "support_region" in real_shift_feat.columns:
        cols_show.append("support_region")
    display(real_shift_feat.sort_values("conditional_abs_residual_median", ascending=False).head(20)[cols_show])

    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    axs[0].scatter(real_shift_feat["fem_nn_target_median"], real_shift_feat["target"], c=real_shift_feat["abs_target"], cmap="magma", s=28, alpha=0.8)
    lim = np.nanmax(np.abs(real_shift_feat[["fem_nn_target_median", "target"]].to_numpy()))
    axs[0].plot([-lim, lim], [-lim, lim], "k--", lw=1)
    axs[0].set_xlabel('Median neighboring FEM target [arcsec]')
    axs[0].set_ylabel('Observed target [arcsec]')
    axs[0].set_title('Conditional shift: real y versus local FEM y')
    axs[0].grid(True, alpha=0.25)

    axs[1].scatter(real_shift_feat["conditional_nn_dist"], real_shift_feat["conditional_abs_residual_median"], c=real_shift_feat["abs_target"], cmap="magma", s=28, alpha=0.8)
    axs[1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
    axs[1].set_xlabel('Distance from FEM in input space')
    axs[1].set_ylabel('|Real y - local FEM y| [arcsec]')
    axs[1].set_title("Input distance vs conditional residual")
    axs[1].grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_data" not in globals() or shift_data is None:
    print('Shift data is unavailable.')
else:
    decision_df = real_shift_feat.copy()
    decision_df["month"] = pd.to_datetime(decision_df["Timestamp"]).dt.month
    decision_df["elev_band"] = pd.cut(
        decision_df["elevation"],
        bins=[0, 20, 30, 45, 60, 75, 90],
        include_lowest=True,
    )
    decision_df["target_extreme"] = decision_df["abs_target"] > EXTREME_TARGET_ARCSEC

    if "domain_score_real_proba" not in decision_df.columns:
        decision_df["domain_score_real_proba"] = np.nan
    if "fem_nn_dist_1" not in decision_df.columns:
        decision_df["fem_nn_dist_1"] = np.nan
    if "conditional_abs_residual_median" not in decision_df.columns:
        decision_df["conditional_abs_residual_median"] = np.nan


    dist_thr = decision_df["fem_nn_dist_1"].quantile(0.75) if decision_df["fem_nn_dist_1"].notna().any() else np.nan
    cond_thr = EXTREME_TARGET_ARCSEC

    def assign_action(row):
        if pd.notna(row.get("conditional_abs_residual_median")) and row["conditional_abs_residual_median"] > cond_thr:
            if pd.notna(row.get("fem_nn_dist_1")) and row["fem_nn_dist_1"] <= dist_thr:
                return "overwrite_model_behavior"
            return "simulate_or_overwrite"
        if pd.notna(row.get("fem_nn_dist_1")) and row["fem_nn_dist_1"] > dist_thr:
            return "domain_adaptation_or_new_FEM"
        return "nominal_or_monitor"

    decision_df["suggested_action"] = decision_df.apply(assign_action, axis=1)

    region_summary = decision_df.groupby(["month", "elev_band"], observed=True).agg(
        count=("target", "size"),
        median_abs_target=("abs_target", "median"),
        p90_abs_target=("abs_target", lambda s: s.quantile(0.90)),
        extreme_pct=("target_extreme", lambda s: s.mean() * 100),
        median_domain_score=("domain_score_real_proba", "median"),
        median_fem_dist=("fem_nn_dist_1", "median"),
        median_conditional_abs_residual=("conditional_abs_residual_median", "median"),
        dominant_action=("suggested_action", lambda s: s.value_counts().idxmax()),
    ).reset_index()

    print('Decision matrix by month/elevation')
    display(region_summary.sort_values(["extreme_pct", "median_conditional_abs_residual", "count"], ascending=False).head(40))

    action_summary = decision_df.groupby("suggested_action").agg(
        count=("target", "size"),
        coverage_pct=("target", lambda s: len(s) / len(decision_df) * 100),
        median_abs_target=("abs_target", "median"),
        p90_abs_target=("abs_target", lambda s: s.quantile(0.90)),
        extreme_pct=("target_extreme", lambda s: s.mean() * 100),
    ).reset_index().sort_values("coverage_pct", ascending=False)
    display(action_summary)


    heat = region_summary.pivot(index="elev_band", columns="month", values="extreme_pct")
    count_heat = region_summary.pivot(index="elev_band", columns="month", values="count")
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(heat.fillna(0).to_numpy(), aspect="auto", cmap="Reds", vmin=0, vmax=max(50, np.nanmax(heat.to_numpy())))
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([str(x) for x in heat.index])
    ax.set_xlabel('Month')
    ax.set_ylabel('Elevation band')
    ax.set_title('Extreme-event risk and shift localization')
    for i in range(len(heat.index)):
        for j in range(len(heat.columns)):
            val = heat.iloc[i, j]
            cnt = count_heat.iloc[i, j]
            if pd.notna(val) and pd.notna(cnt) and cnt > 0:
                ax.text(j, i, f"{val:.0f}%\nn={int(cnt)}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label='extreme events [%]')
    plt.tight_layout()
    plt.show()

    print('Samples for metrology review: overwrite/simulate_or_overwrite')
    review_cases = decision_df[decision_df["suggested_action"].isin(["overwrite_model_behavior", "simulate_or_overwrite"])].copy()
    show_cols = ["Timestamp", "elevation", "target", "abs_target", "month", "elev_band", "suggested_action", "fem_nn_dist_1", "domain_score_real_proba", "conditional_abs_residual_median"]
    display(review_cases.sort_values("abs_target", ascending=False).head(30)[show_cols])


## Shift characterization v2: controlli anti-leakage

La prima caratterizzazione mostra una separazione FEM/reale quasi perfetta, ma include feature non comparabili tra domini (`trend_*`, `az_*`) e può campionare solo le prime elevazioni del file FEM. Questa versione corregge due punti:

- usa solo feature disponibili in entrambi i domini senza NaN strutturali;
- campiona il FEM in modo stratificato per elevazione, così 5°, 30°, 60° e 85° entrano tutti nel confronto.

Questa è la versione da usare per la risposta quantitativa finale sul covariate shift.

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors

SHIFT_V2_MAX_FEM_ROWS = 160_000
SHIFT_V2_TARGET_PER_ELEV = 40_000
SHIFT_V2_EXPERT_ELEVS = [5, 30, 60, 85]
SHIFT_V2_RANDOM_SEED = RANDOM_SEED if "RANDOM_SEED" in globals() else 42

def load_fem_stratified_for_shift_v2():
    try:
        fem_path = find_data_file(FEM_CANDIDATES)
    except FileNotFoundError:
        print('FEM file not found.')
        return None, None, None, None, None

    print('Loading FEM data stratified by elevation:', fem_path)
    suffix = Path(fem_path).suffix.lower()

    if suffix == ".csv":


        fem_all = pd.read_csv(fem_path)
    else:
        fem_all = pd.read_excel(fem_path)

    fem_all, fem_sensor_cols, fem_elev_col, fem_target_col = infer_fem_columns(fem_all)
    if fem_elev_col is None:
        print('FEM elevation column not found; using a global random sample.')
        fem_sample = fem_all.sample(n=min(SHIFT_V2_MAX_FEM_ROWS, len(fem_all)), random_state=SHIFT_V2_RANDOM_SEED)
    else:
        parts = []
        for elev, g in fem_all.groupby(fem_elev_col):
            n_take = min(SHIFT_V2_TARGET_PER_ELEV, len(g))
            parts.append(g.sample(n=n_take, random_state=SHIFT_V2_RANDOM_SEED))
        fem_sample = pd.concat(parts, ignore_index=True)
        if len(fem_sample) > SHIFT_V2_MAX_FEM_ROWS:
            fem_sample = fem_sample.sample(n=SHIFT_V2_MAX_FEM_ROWS, random_state=SHIFT_V2_RANDOM_SEED)

    print("FEM sample v2:", len(fem_sample))
    if fem_elev_col is not None:
        display(fem_sample[fem_elev_col].value_counts().sort_index().rename_axis("Elevation").reset_index(name="count"))
    print("FEM sensor cols:", len(fem_sensor_cols), "| elev:", fem_elev_col, "| target:", fem_target_col)
    return fem_sample, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col

def build_shift_v2_tables():
    fem_sample, fem_path, fem_sensor_cols, fem_elev_col, fem_target_col = load_fem_stratified_for_shift_v2()
    if fem_sample is None or len(fem_sensor_cols) < 20:
        return None


    comparable_cols = [
        "temp_mean", "temp_std_spatial", "temp_min", "temp_max", "temp_range",
        "temp_p10", "temp_p50", "temp_p90", "temp_iqr", "elevation"
    ]

    real_current = X_seq[:, -1, :]
    real_v2 = aggregate_sensor_state(real_current)
    real_v2["elevation"] = meta["Elevation"].to_numpy(dtype=float)
    real_v2["target"] = y.astype(float)
    real_v2["abs_target"] = np.abs(y.astype(float))
    real_v2["domain"] = "real"
    real_v2["Timestamp"] = meta["Timestamp"].to_numpy()

    fem_arr = fem_sample[fem_sensor_cols].to_numpy(dtype="float32")
    fem_v2 = aggregate_sensor_state(fem_arr)
    fem_v2["elevation"] = pd.to_numeric(fem_sample[fem_elev_col], errors="coerce").to_numpy(dtype=float) if fem_elev_col else np.nan
    fem_v2["target"] = pd.to_numeric(fem_sample[fem_target_col], errors="coerce").to_numpy(dtype=float) if fem_target_col else np.nan
    fem_v2["abs_target"] = np.abs(fem_v2["target"].to_numpy(dtype=float))
    fem_v2["domain"] = "fem"
    fem_v2["Timestamp"] = pd.NaT

    return {
        "real_v2": real_v2,
        "fem_v2": fem_v2,
        "feature_cols_v2": comparable_cols,
        "fem_target_col": fem_target_col,
        "fem_path": fem_path,
    }

shift_v2 = build_shift_v2_tables()
if shift_v2 is not None:
    real_v2 = shift_v2["real_v2"]
    fem_v2 = shift_v2["fem_v2"]
    feature_cols_v2 = shift_v2["feature_cols_v2"]
    print('Comparable v2 features:', feature_cols_v2)
    display(pd.concat([
        real_v2[feature_cols_v2 + ["target"]].describe(percentiles=[.1, .5, .9]).assign(domain_summary="real"),
        fem_v2[feature_cols_v2 + ["target"]].describe(percentiles=[.1, .5, .9]).assign(domain_summary="fem"),
    ]))


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Shift v2 is unavailable.')
else:
    rng = np.random.default_rng(SHIFT_V2_RANDOM_SEED)
    n_real = len(real_v2)
    fem_idx = rng.choice(len(fem_v2), size=min(len(fem_v2), n_real * 3), replace=False)
    domain_v2_df = pd.concat([real_v2, fem_v2.iloc[fem_idx]], ignore_index=True)
    y_domain_v2 = (domain_v2_df["domain"] == "real").astype(int).to_numpy()
    X_domain_v2 = domain_v2_df[feature_cols_v2]

    Xtr, Xte, ytr, yte = train_test_split(
        X_domain_v2, y_domain_v2, test_size=0.30, stratify=y_domain_v2, random_state=SHIFT_V2_RANDOM_SEED
    )

    domain_clf_v2 = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=600,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SHIFT_V2_RANDOM_SEED,
            n_jobs=-1,
        )),
    ])
    domain_clf_v2.fit(Xtr, ytr)
    proba = domain_clf_v2.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"Domain classifier v2 FEM-vs-Real: AUC={roc_auc_score(yte, proba):.3f} | accuracy={accuracy_score(yte, pred):.3f}")
    print(classification_report(yte, pred, target_names=["FEM", "Real"]))

    perm_v2 = permutation_importance(
        domain_clf_v2, Xte, yte,
        n_repeats=20,
        random_state=SHIFT_V2_RANDOM_SEED,
        scoring="roc_auc",
        n_jobs=-1,
    )
    domain_importance_v2_df = pd.DataFrame({
        "feature": feature_cols_v2,
        "importance_mean_auc_drop": perm_v2.importances_mean,
        "importance_std": perm_v2.importances_std,
    }).sort_values("importance_mean_auc_drop", ascending=False)
    display(domain_importance_v2_df)

    real_v2["domain_score_real_proba_v2"] = domain_clf_v2.predict_proba(real_v2[feature_cols_v2])[:, 1]
    fem_v2["domain_score_real_proba_v2"] = domain_clf_v2.predict_proba(fem_v2[feature_cols_v2])[:, 1]
    display(pd.DataFrame({
        "real": real_v2["domain_score_real_proba_v2"].describe(percentiles=[.1, .25, .5, .75, .9]),
        "fem": fem_v2["domain_score_real_proba_v2"].describe(percentiles=[.1, .25, .5, .75, .9]),
    }))

    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    axs[0].hist(fem_v2["domain_score_real_proba_v2"], bins=50, alpha=0.55, density=True, label="FEM")
    axs[0].hist(real_v2["domain_score_real_proba_v2"], bins=50, alpha=0.55, density=True, label="Real")
    axs[0].set_xlabel('P(real), comparable features')
    axs[0].set_title("Domain classifier v2")
    axs[0].grid(True, alpha=0.25)
    axs[0].legend()

    top = domain_importance_v2_df.head(10).iloc[::-1]
    axs[1].barh(top["feature"], top["importance_mean_auc_drop"])
    axs[1].set_xlabel('Permutation AUC drop')
    axs[1].set_title('Input-shift features v2')
    axs[1].grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Shift v2 is unavailable.')
else:
    support_pipe_v2 = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    fem_scaled_v2 = support_pipe_v2.fit_transform(fem_v2[feature_cols_v2])
    real_scaled_v2 = support_pipe_v2.transform(real_v2[feature_cols_v2])

    nn_v2 = NearestNeighbors(n_neighbors=min(SHIFT_NN_NEIGHBORS, len(fem_scaled_v2))).fit(fem_scaled_v2)
    dist_v2, idx_v2 = nn_v2.kneighbors(real_scaled_v2)
    real_v2["fem_nn_dist_1_v2"] = dist_v2[:, 0]
    real_v2["fem_nn_dist_med_v2"] = np.median(dist_v2, axis=1)


    rng = np.random.default_rng(SHIFT_V2_RANDOM_SEED)
    ref_idx = rng.choice(len(fem_scaled_v2), size=min(20_000, len(fem_scaled_v2)), replace=False)
    nn_self_v2 = NearestNeighbors(n_neighbors=2).fit(fem_scaled_v2)
    self_dist_v2, _ = nn_self_v2.kneighbors(fem_scaled_v2[ref_idx])
    fem_neighbor_dist_v2 = self_dist_v2[:, 1]
    thr95_v2 = np.quantile(fem_neighbor_dist_v2, 0.95)
    thr99_v2 = np.quantile(fem_neighbor_dist_v2, 0.99)
    real_v2["support_region_v2"] = np.select(
        [
            real_v2["fem_nn_dist_1_v2"] <= thr95_v2,
            real_v2["fem_nn_dist_1_v2"] <= thr99_v2,
        ],
        ["inside_fem_p95", "border_fem_p99"],
        default="outside_fem_p99",
    )
    print(f"Support thresholds v2: p95={thr95_v2:.3f}, p99={thr99_v2:.3f}")
    support_v2_summary = real_v2.groupby("support_region_v2").agg(
        count=("target", "size"),
        coverage_pct=("target", lambda s: len(s) / len(real_v2) * 100),
        median_abs_target=("abs_target", "median"),
        p90_abs_target=("abs_target", lambda s: s.quantile(.90)),
        extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
        median_domain_score=("domain_score_real_proba_v2", "median") if "domain_score_real_proba_v2" in real_v2.columns else ("abs_target", "median"),
    ).reset_index()
    display(support_v2_summary)


    if fem_v2["target"].notna().sum() >= 100:
        fem_targets_nn_v2 = fem_v2["target"].to_numpy(dtype=float)[idx_v2]
        real_v2["fem_nn_target_median_v2"] = np.nanmedian(fem_targets_nn_v2, axis=1)
        real_v2["conditional_residual_median_v2"] = real_v2["target"] - real_v2["fem_nn_target_median_v2"]
        real_v2["conditional_abs_residual_median_v2"] = real_v2["conditional_residual_median_v2"].abs()

        conditional_v2_summary = real_v2.groupby("support_region_v2").agg(
            count=("target", "size"),
            mae_fem_neighbor_median=("conditional_abs_residual_median_v2", "mean"),
            median_abs_residual=("conditional_abs_residual_median_v2", "median"),
            p90_abs_residual=("conditional_abs_residual_median_v2", lambda s: s.quantile(.90)),
            extreme_pct=("abs_target", lambda s: (s > EXTREME_TARGET_ARCSEC).mean() * 100),
        ).reset_index()
        print("Conditional shift v2")
        display(conditional_v2_summary)

        print('Largest conditional residuals v2')
        show_cols = ["Timestamp", "elevation", "target", "abs_target", "fem_nn_target_median_v2", "conditional_residual_median_v2", "fem_nn_dist_1_v2", "support_region_v2"]
        display(real_v2.sort_values("conditional_abs_residual_median_v2", ascending=False).head(20)[show_cols])

        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        axs[0].scatter(real_v2["fem_nn_target_median_v2"], real_v2["target"], c=real_v2["abs_target"], cmap="magma", s=28, alpha=0.8)
        lim = np.nanmax(np.abs(real_v2[["fem_nn_target_median_v2", "target"]].to_numpy()))
        axs[0].plot([-lim, lim], [-lim, lim], "k--", lw=1)
        axs[0].set_xlabel('Local FEM target v2 [arcsec]')
        axs[0].set_ylabel('Observed target [arcsec]')
        axs[0].set_title('P(y|X): local FEM versus real')
        axs[0].grid(True, alpha=0.25)

        axs[1].scatter(real_v2["fem_nn_dist_1_v2"], real_v2["conditional_abs_residual_median_v2"], c=real_v2["abs_target"], cmap="magma", s=28, alpha=0.8)
        axs[1].axhline(EXTREME_TARGET_ARCSEC, color="tab:red", linestyle="--")
        axs[1].set_xlabel('FEM distance v2')
        axs[1].set_ylabel('|Conditional residual| [arcsec]')
        axs[1].set_title("Support distance vs conditional residual")
        axs[1].grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
    else:
        print('FEM targets are unavailable for conditional shift v2.')


## Drift tests ispirati a Hinder et al.

Questa sezione integra test leggeri ispirati al repository indicato da Davide:

- D3/discriminative drift detector;
- MMD kernel two-sample test;
- KS univariato per spiegare quali feature driftano;
- DAWIDD-like test per drift temporale nel reale;
- localizzazione temporale con sliding MMD;
- one-class support model addestrato su FEM.

Le funzioni sono riscritte qui in forma controllata, senza importare gli script del repo, perché alcuni file del repository avviano benchmark pesanti all'import.

In [ ]:
from scipy.stats import ks_2samp
from sklearn.metrics.pairwise import pairwise_kernels, pairwise_distances
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor

DRIFT_RANDOM_SEED = RANDOM_SEED if "RANDOM_SEED" in globals() else 42
DRIFT_N_PERM = 500  

def _as_clean_matrix(df_or_arr):
    X = np.asarray(df_or_arr, dtype=float)
    X = np.where(np.isfinite(X), X, np.nan)

    med = np.nanmedian(X, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(med, inds[1])
    return X

def d3_auc_score(X_a, X_b, clf=None, random_state=DRIFT_RANDOM_SEED):
    """Measure domain separability with a D3-style discriminative detector."""
    X = np.vstack([_as_clean_matrix(X_a), _as_clean_matrix(X_b)])
    y = np.r_[np.zeros(len(X_a), dtype=int), np.ones(len(X_b), dtype=int)]
    if clf is None:
        clf = LogisticRegression(solver="liblinear", class_weight="balanced", random_state=random_state)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35, stratify=y, random_state=random_state)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clone(clf)),
    ])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, proba)

def mmd_rbf_permutation(X_a, X_b, n_perm=DRIFT_N_PERM, gamma=None, random_state=DRIFT_RANDOM_SEED):
    """Estimate RBF MMD squared and its permutation p-value."""
    rng = np.random.default_rng(random_state)
    Xa = _as_clean_matrix(X_a)
    Xb = _as_clean_matrix(X_b)
    X = np.vstack([Xa, Xb])
    n_a = len(Xa)
    if gamma is None:

        sub_n = min(1000, len(X))
        sub_idx = rng.choice(len(X), size=sub_n, replace=False)
        D = pairwise_distances(X[sub_idx], metric="euclidean")
        med = np.median(D[D > 0])
        gamma = 1.0 / (2.0 * med**2) if np.isfinite(med) and med > 0 else 1.0
    K = pairwise_kernels(X, metric="rbf", gamma=gamma)

    def stat_from_mask(mask_a):
        idx_a = np.where(mask_a)[0]
        idx_b = np.where(~mask_a)[0]
        Kaa = K[np.ix_(idx_a, idx_a)]
        Kbb = K[np.ix_(idx_b, idx_b)]
        Kab = K[np.ix_(idx_a, idx_b)]
        return Kaa.mean() + Kbb.mean() - 2.0 * Kab.mean()

    mask0 = np.zeros(len(X), dtype=bool)
    mask0[:n_a] = True
    stat = stat_from_mask(mask0)
    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(len(X))
        mask = np.zeros(len(X), dtype=bool)
        mask[perm[:n_a]] = True
        perm_stats[i] = stat_from_mask(mask)
    p_value = (np.sum(perm_stats >= stat) + 1.0) / (n_perm + 1.0)
    return float(stat), float(p_value), float(gamma)

def ks_feature_table(X_a_df, X_b_df, feature_cols, label_a="FEM", label_b="Real"):
    rows = []
    for col in feature_cols:
        a = pd.to_numeric(X_a_df[col], errors="coerce").dropna().to_numpy(dtype=float)
        b = pd.to_numeric(X_b_df[col], errors="coerce").dropna().to_numpy(dtype=float)
        if len(a) < 5 or len(b) < 5:
            continue
        res = ks_2samp(a, b, mode="auto")
        rows.append({
            "feature": col,
            "ks_statistic": float(res.statistic),
            "p_value": float(res.pvalue),
            f"{label_a}_median": float(np.median(a)),
            f"{label_b}_median": float(np.median(b)),
            f"{label_a}_p10": float(np.quantile(a, 0.10)),
            f"{label_b}_p10": float(np.quantile(b, 0.10)),
            f"{label_a}_p90": float(np.quantile(a, 0.90)),
            f"{label_b}_p90": float(np.quantile(b, 0.90)),
        })
    return pd.DataFrame(rows).sort_values(["ks_statistic", "p_value"], ascending=[False, True])

def dawidd_like_time_dependence(X, n_perm=DRIFT_N_PERM, random_state=DRIFT_RANDOM_SEED):
    """Test feature-time dependence with an HSIC-like statistic inspired by DAWIDD."""
    rng = np.random.default_rng(random_state)
    X = _as_clean_matrix(X)
    n = len(X)
    if n < 20:
        return np.nan, np.nan

    Xs = StandardScaler().fit_transform(X)
    t = np.linspace(-1, 1, n).reshape(-1, 1)
    Kx = pairwise_kernels(Xs, metric="rbf")
    Kt = pairwise_kernels(t, metric="rbf")
    H = np.eye(n) - np.ones((n, n)) / n
    Kxc = H @ Kx @ H
    Ktc = H @ Kt @ H
    stat = float(np.sum(Kxc * Ktc) / (n * n))
    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(n)
        Ktp = Kt[np.ix_(perm, perm)]
        Ktpc = H @ Ktp @ H
        perm_stats[i] = np.sum(Kxc * Ktpc) / (n * n)
    p_value = (np.sum(perm_stats >= stat) + 1.0) / (n_perm + 1.0)
    return stat, float(p_value)

def sliding_mmd_time(df, feature_cols, window=80, step=20, n_perm=200):
    """Compute MMD between consecutive chronological windows."""
    rows = []
    if len(df) < 2 * window:
        return pd.DataFrame(rows)
    X = df[feature_cols].copy()
    times = pd.to_datetime(df["Timestamp"]).reset_index(drop=True)
    for start in range(0, len(df) - 2 * window + 1, step):
        a = X.iloc[start:start + window]
        b = X.iloc[start + window:start + 2 * window]
        stat, p, gamma = mmd_rbf_permutation(a, b, n_perm=n_perm, random_state=DRIFT_RANDOM_SEED + start)
        rows.append({
            "start_idx": start,
            "mid_time": times.iloc[start + window],
            "end_time": times.iloc[start + 2 * window - 1],
            "mmd_stat": stat,
            "p_value": p,
            "gamma": gamma,
        })
    return pd.DataFrame(rows)


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Run Shift characterization v2 first.')
else:
    rng = np.random.default_rng(DRIFT_RANDOM_SEED)
    n_real = len(real_v2)
    fem_idx = rng.choice(len(fem_v2), size=min(len(fem_v2), n_real * 3), replace=False)
    fem_cmp = fem_v2.iloc[fem_idx].copy()
    real_cmp = real_v2.copy()

    X_fem_cmp = fem_cmp[feature_cols_v2]
    X_real_cmp = real_cmp[feature_cols_v2]

    d3_rows = []
    for name, clf in {
        "D3_logistic": LogisticRegression(solver="liblinear", class_weight="balanced", random_state=DRIFT_RANDOM_SEED),
        "D3_extra_trees": ExtraTreesClassifier(n_estimators=400, max_depth=6, min_samples_leaf=5, class_weight="balanced", random_state=DRIFT_RANDOM_SEED, n_jobs=-1),
    }.items():
        auc = d3_auc_score(X_fem_cmp, X_real_cmp, clf=clf)
        d3_rows.append({"method": name, "auc_domain_separation": auc, "drift_strength": abs(auc - 0.5) * 2})
    d3_df = pd.DataFrame(d3_rows)
    display(d3_df)

    print('RBF MMD: FEM versus real, comparable v2 features')
    mmd_stat, mmd_p, mmd_gamma = mmd_rbf_permutation(X_fem_cmp, X_real_cmp, n_perm=DRIFT_N_PERM)
    mmd_global_df = pd.DataFrame([{
        "comparison": "FEM_vs_Real_global",
        "mmd_stat": mmd_stat,
        "p_value": mmd_p,
        "gamma": mmd_gamma,
        "n_fem": len(X_fem_cmp),
        "n_real": len(X_real_cmp),
    }])
    display(mmd_global_df)

    print('KS by feature: FEM versus real')
    ks_v2_df = ks_feature_table(fem_cmp, real_cmp, feature_cols_v2, label_a="FEM", label_b="Real")
    display(ks_v2_df)

    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    top_ks = ks_v2_df.head(10).iloc[::-1]
    axs[0].barh(top_ks["feature"], top_ks["ks_statistic"])
    axs[0].set_xlabel("KS statistic")
    axs[0].set_title('Features with the largest univariate drift')
    axs[0].grid(True, axis="x", alpha=0.25)


    if len(ks_v2_df):
        f0 = ks_v2_df.iloc[0]["feature"]
        axs[1].hist(fem_cmp[f0], bins=50, alpha=0.55, density=True, label="FEM")
        axs[1].hist(real_cmp[f0], bins=50, alpha=0.55, density=True, label="Real")
        axs[1].set_title(f"Distribution of {f0}")
        axs[1].legend()
        axs[1].grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Run Shift characterization v2 first.')
else:
    elev_bins = [0, 20, 30, 45, 60, 75, 90]
    real_band = pd.cut(real_v2["elevation"], bins=elev_bins, include_lowest=True)
    fem_band = pd.cut(fem_v2["elevation"], bins=elev_bins, include_lowest=True)

    band_rows = []
    for band in sorted(real_band.dropna().unique()):
        r = real_v2[real_band == band]
        f = fem_v2[fem_band == band]
        if len(r) < 10 or len(f) < 30:
            band_rows.append({
                "elev_band": str(band),
                "n_real": len(r),
                "n_fem": len(f),
                "note": 'too few samples',
            })
            continue
        f_sample = f.sample(n=min(len(f), len(r) * 3), random_state=DRIFT_RANDOM_SEED)
        auc = d3_auc_score(f_sample[feature_cols_v2], r[feature_cols_v2])
        mmd_stat, mmd_p, _ = mmd_rbf_permutation(f_sample[feature_cols_v2], r[feature_cols_v2], n_perm=300)
        ks_band = ks_feature_table(f_sample, r, feature_cols_v2, label_a="FEM", label_b="Real")
        band_rows.append({
            "elev_band": str(band),
            "n_real": len(r),
            "n_fem": len(f),
            "d3_auc": auc,
            "d3_strength": abs(auc - 0.5) * 2,
            "mmd_stat": mmd_stat,
            "mmd_p_value": mmd_p,
            "top_ks_feature": ks_band.iloc[0]["feature"] if len(ks_band) else None,
            "top_ks_stat": ks_band.iloc[0]["ks_statistic"] if len(ks_band) else np.nan,
            "note": "",
        })
    drift_by_elev_df = pd.DataFrame(band_rows)
    display(drift_by_elev_df)


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Run Shift characterization v2 first.')
else:
    real_time = real_v2.sort_values("Timestamp").reset_index(drop=True).copy()

    temporal_cols = feature_cols_v2.copy()
    if "real_shift_feat" in globals():
        trend_cols = [c for c in real_shift_feat.columns if c.startswith("trend_")]
        if trend_cols:
            real_time = real_time.merge(
                real_shift_feat[["Timestamp"] + trend_cols],
                on="Timestamp",
                how="left",
            )
            temporal_cols += trend_cols

    print('DAWIDD-like temporal dependence in real data')
    daw_stat, daw_p = dawidd_like_time_dependence(real_time[temporal_cols], n_perm=DRIFT_N_PERM)
    dawidd_real_df = pd.DataFrame([{
        "dataset": "real_ordered_by_time",
        "stat": daw_stat,
        "p_value": daw_p,
        "n": len(real_time),
        "n_features": len(temporal_cols),
    }])
    display(dawidd_real_df)

    print('Sliding MMD between consecutive time windows')
    sliding_mmd_df = sliding_mmd_time(real_time[["Timestamp"] + temporal_cols], temporal_cols, window=80, step=20, n_perm=200)
    display(sliding_mmd_df.sort_values("p_value").head(20))

    if len(sliding_mmd_df):
        fig, ax1 = plt.subplots(figsize=(13, 4))
        ax1.plot(sliding_mmd_df["mid_time"], sliding_mmd_df["mmd_stat"], marker="o", label="MMD stat")
        ax1.set_ylabel("MMD statistic")
        ax1.grid(True, alpha=0.25)
        ax2 = ax1.twinx()
        ax2.plot(sliding_mmd_df["mid_time"], sliding_mmd_df["p_value"], color="tab:red", marker="x", label="p-value")
        ax2.axhline(0.05, color="tab:red", linestyle="--", alpha=0.6)
        ax2.set_ylabel("p-value")
        ax1.set_title('Temporal drift in real data: sliding MMD')
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()


        context_rows = []
        for _, row in sliding_mmd_df.sort_values("p_value").head(10).iterrows():
            start = int(row["start_idx"])
            end = start + 160
            sub = real_time.iloc[start:end]
            context_rows.append({
                "mid_time": row["mid_time"],
                "p_value": row["p_value"],
                "mmd_stat": row["mmd_stat"],
                "start_time": sub["Timestamp"].min(),
                "end_time": sub["Timestamp"].max(),
                "months": ",".join(map(str, sorted(pd.to_datetime(sub["Timestamp"]).dt.month.unique()))),
                "median_abs_target": float(sub["abs_target"].median()),
                "p90_abs_target": float(sub["abs_target"].quantile(.90)),
                "extreme_pct": float((sub["abs_target"] > EXTREME_TARGET_ARCSEC).mean() * 100),
            })
        print('Context of time windows with the strongest drift')
        display(pd.DataFrame(context_rows))


In [ ]:
if "shift_v2" not in globals() or shift_v2 is None:
    print('Run Shift characterization v2 first.')
else:
    rng = np.random.default_rng(DRIFT_RANDOM_SEED)

    fem_train = fem_v2.sample(n=min(50_000, len(fem_v2)), random_state=DRIFT_RANDOM_SEED)
    real_eval = real_v2.copy()
    support_models = {
        "IsolationForest": IsolationForest(
            n_estimators=500,
            contamination="auto",
            random_state=DRIFT_RANDOM_SEED,
            n_jobs=-1,
        ),
        "LOF_novelty": LocalOutlierFactor(n_neighbors=35, novelty=True, contamination="auto"),
    }
    rows = []
    score_df = real_eval[["Timestamp", "elevation", "target", "abs_target"]].copy()
    X_fem_train = _as_clean_matrix(fem_train[feature_cols_v2])
    X_real_eval = _as_clean_matrix(real_eval[feature_cols_v2])
    scaler_oc = StandardScaler().fit(X_fem_train)
    Xf = scaler_oc.transform(X_fem_train)
    Xr = scaler_oc.transform(X_real_eval)

    for name, model in support_models.items():
        m = clone(model)
        m.fit(Xf)

        score_fem = m.score_samples(Xf)
        score_real = m.score_samples(Xr)
        score_df[f"{name}_score"] = score_real
        thr_01 = np.quantile(score_fem, 0.01)
        thr_05 = np.quantile(score_fem, 0.05)
        rows.append({
            "model": name,
            "fem_score_median": float(np.median(score_fem)),
            "real_score_median": float(np.median(score_real)),
            "fem_p01_threshold": float(thr_01),
            "fem_p05_threshold": float(thr_05),
            "real_below_fem_p01_%": float((score_real < thr_01).mean() * 100),
            "real_below_fem_p05_%": float((score_real < thr_05).mean() * 100),
            "extreme_pct_below_p05": float((real_eval.loc[score_real < thr_05, "abs_target"] > EXTREME_TARGET_ARCSEC).mean() * 100) if np.any(score_real < thr_05) else np.nan,
        })

    oneclass_support_df = pd.DataFrame(rows)
    display(oneclass_support_df)

    print('Most anomalous real samples according to IsolationForest')
    if "IsolationForest_score" in score_df.columns:
        display(score_df.sort_values("IsolationForest_score").head(20))


## Espansione controllata con dataset completo

Questa sezione usa tutte le scansioni disponibili come sorgente target, recupera azimuth e filename dal file di puntamento e confronta coorti progressive. Gli outlier non vengono rimossi in base alla sola ampiezza del target, salvo soglie molto larghe per risultati non credibili. Il confronto principale usa sempre lo stesso validation/test affidabile e assegna ogni sessione a un solo split.

In [ ]:
TOTAL_REAL_CANDIDATES = ["Total_EffectiveDataset_Tseries_outliers.csv"]
SESSION_GAP_MIN = 180.0
NEAR_AZ_DEG = 10.0
NEAR_EL_DEG = 5.0
NEAR_THERMAL_MEDIAN_C = 0.75
PROCEDURAL_DXEL_ARCSEC = 15.0
PROCEDURAL_DEL_ARCSEC = 20.0
RUN_EXPANDED_GRU = True

total_real_path = find_data_file(TOTAL_REAL_CANDIDATES)
pointing_full_path = find_data_file(POINTING_CANDIDATES)
print('Complete thermal dataset:', total_real_path)
print('Pointing master:', pointing_full_path)

total_real_raw = normalize_sensor_columns(read_table(total_real_path))
total_real_raw = ensure_datetime(total_real_raw, "Timestamp").sort_values("Timestamp").reset_index(drop=True)
total_sensor_cols = get_sensor_cols(total_real_raw)
if total_sensor_cols != sensor_cols:
    print('Warning: complete and original datasets have different sensor columns.')
print(f"Complete rows: {len(total_real_raw):,} | sensors: {len(total_sensor_cols)}")
print('Targets embedded in the complete file:', int(total_real_raw[TARGET].notna().sum()))

pointing_master = read_table(pointing_full_path).rename(columns={
    "Time": "Timestamp",
    "Elevation": "Elevation",
    "Offset_XEL_arcsec": "XEl Offset",
    "Offset_EL_arcsec": "El Offset",
})
pointing_master["Timestamp"] = pd.to_datetime(pointing_master["Timestamp"], errors="coerce")
pointing_master = pointing_master.dropna(subset=["Timestamp", "Elevation", "XEl Offset"]).copy()
pointing_master = pointing_master.sort_values("Timestamp").drop_duplicates("Timestamp", keep="first").reset_index(drop=True)

tmin, tmax = total_real_raw["Timestamp"].min(), total_real_raw["Timestamp"].max()
pointing_master = pointing_master[pointing_master["Timestamp"].between(tmin, tmax)].copy().reset_index(drop=True)
embedded_times = set(total_real_raw.loc[total_real_raw[TARGET].notna(), "Timestamp"])
original_times = set(real_raw.loc[real_raw[TARGET].notna(), "Timestamp"])
pointing_master["in_embedded_total"] = pointing_master["Timestamp"].isin(embedded_times)
pointing_master["in_original_dataset"] = pointing_master["Timestamp"].isin(original_times)
pointing_master["abs_xel"] = pointing_master["XEl Offset"].abs()
pointing_master["abs_el"] = pointing_master["El Offset"].abs()
pointing_master["flag_gross_invalid"] = (
    (pointing_master["abs_xel"] > MAX_ABS_XEL_FOR_TRAIN) |
    (pointing_master["abs_el"] > MAX_ABS_EL_FOR_TRAIN)
)
pointing_master["Source"] = (
    pointing_master["Filename"].astype(str)
    .str.extract(r"-([^-]+)\.dat$", expand=False)
    .fillna("unknown")
)

embedded_check = total_real_raw.loc[
    total_real_raw[TARGET].notna(), ["Timestamp", "XEl Offset", "El Offset"]
].merge(
    pointing_master[["Timestamp", "XEl Offset", "El Offset"]],
    on="Timestamp", how="inner", suffixes=("_embedded", "_master")
)
if len(embedded_check):
    max_dx = (embedded_check["XEl Offset_embedded"] - embedded_check["XEl Offset_master"]).abs().max()
    max_de = (embedded_check["El Offset_embedded"] - embedded_check["El Offset_master"]).abs().max()
    print(f"Embedded/master offset agreement: max dXEL={max_dx:.6g}, max dEL={max_de:.6g}")

reconciliation_df = pd.DataFrame([{
    "pointing_master": len(pointing_master),
    "target_embedded_total": int(pointing_master["in_embedded_total"].sum()),
    "missing_from_embedded_total": int((~pointing_master["in_embedded_total"]).sum()),
    'original_targets': int(pointing_master["in_original_dataset"].sum()),
    'new_targets': int((~pointing_master["in_original_dataset"]).sum()),
    "gross_invalid": int(pointing_master["flag_gross_invalid"].sum()),
    'valid_azimuth_count': int(pointing_master["Azimuth"].notna().sum()),
}])
display(reconciliation_df)

print('Targets missing from the complete thermal file by month')
missing_embedded = pointing_master[~pointing_master["in_embedded_total"]].copy()
if len(missing_embedded):
    display(missing_embedded.groupby(missing_embedded["Timestamp"].dt.to_period("M")).size().rename("count").reset_index())
else:
    print("Nessuno.")


In [ ]:
X_exp, X_elev_exp, y_exp, meta_exp, skip_exp = build_supervised_windows(
    total_real_raw,
    total_sensor_cols,
    target_table=pointing_master,
)

meta_exp = meta_exp.merge(
    pointing_master[[
        "Timestamp", "El Offset", "Source", "in_embedded_total",
        "in_original_dataset", "flag_gross_invalid", "abs_xel", "abs_el"
    ]],
    on="Timestamp", how="left", validate="one_to_one"
)
if not np.allclose(meta_exp["Target"].to_numpy(dtype=float), y_exp, equal_nan=False):
    raise RuntimeError('y_exp and meta_exp are misaligned after the merge.')

order_exp = np.argsort(meta_exp["Timestamp"].to_numpy())
X_exp = X_exp[order_exp]
X_elev_exp = X_elev_exp[order_exp]
y_exp = y_exp[order_exp]
meta_exp = meta_exp.iloc[order_exp].reset_index(drop=True)
meta_exp["sample_pos"] = np.arange(len(meta_exp))

print('Complete valid windows:', len(meta_exp), "/", len(pointing_master))
print('Window-construction rejections:', skip_exp)
print('Original samples recovered:', int(meta_exp["in_original_dataset"].sum()))
print('New samples recovered:', int((~meta_exp["in_original_dataset"]).sum()))
print('Scans absent from embedded targets but recovered through the master:', int((~meta_exp["in_embedded_total"]).sum()))
print('Gross-invalid events among valid windows:', int(meta_exp["flag_gross_invalid"].sum()))

window_recovery_df = pd.DataFrame([{
    "master_targets": len(pointing_master),
    "valid_windows": len(meta_exp),
    "coverage_%": 100 * len(meta_exp) / max(len(pointing_master), 1),
    "original_valid": int(meta_exp["in_original_dataset"].sum()),
    "new_valid": int((~meta_exp["in_original_dataset"]).sum()),
    "missing_embedded_recovered": int((~meta_exp["in_embedded_total"]).sum()),
}])
display(window_recovery_df)


old_pos_by_time = {pd.Timestamp(t): i for i, t in enumerate(meta["Timestamp"])}
exp_pos_by_time = {pd.Timestamp(t): i for i, t in enumerate(meta_exp["Timestamp"])}
common_window_times = sorted(set(old_pos_by_time).intersection(exp_pos_by_time))
if common_window_times:
    old_pos = np.array([old_pos_by_time[t] for t in common_window_times], dtype=int)
    exp_pos = np.array([exp_pos_by_time[t] for t in common_window_times], dtype=int)
    window_delta = np.abs(X_seq[old_pos] - X_exp[exp_pos])
    alignment_df = pd.DataFrame([{
        "common_valid_windows": len(common_window_times),
        "max_abs_target_delta": float(np.max(np.abs(y[old_pos] - y_exp[exp_pos]))),
        "median_abs_temperature_delta_C": float(np.median(window_delta)),
        "p99_abs_temperature_delta_C": float(np.quantile(window_delta, 0.99)),
        "max_abs_temperature_delta_C": float(np.max(window_delta)),
    }])
    print('Window alignment: original versus complete container')
    display(alignment_df)
else:
    print('No common windows found; check timestamps.')


In [ ]:
diag_exp = meta_exp.copy()
diag_exp["Source"] = diag_exp["Source"].fillna("unknown").astype(str)
diag_exp["dt_prev_source_min"] = diag_exp.groupby("Source", sort=False)["Timestamp"].diff().dt.total_seconds() / 60
session_break = diag_exp["dt_prev_source_min"].isna() | (diag_exp["dt_prev_source_min"] > SESSION_GAP_MIN)
diag_exp["source_session_num"] = session_break.groupby(diag_exp["Source"]).cumsum().astype(int)
diag_exp["session_id"] = diag_exp["Source"] + "::" + diag_exp["source_session_num"].astype(str)
diag_exp["sequence_position"] = diag_exp.groupby("session_id", sort=False).cumcount()
diag_exp["sequence_size"] = diag_exp.groupby("session_id", sort=False)["session_id"].transform("size")
diag_exp["flag_sequence_followup"] = diag_exp["sequence_position"] > 0

for col in ["Target", "El Offset", "Elevation", "Azimuth", "sample_pos"]:
    diag_exp[f"prev_{col}"] = diag_exp.groupby("session_id", sort=False)[col].shift(1)

diag_exp["dXEL_prev"] = (diag_exp["Target"] - diag_exp["prev_Target"]).abs()
diag_exp["dEL_prev"] = (diag_exp["El Offset"] - diag_exp["prev_El Offset"]).abs()
diag_exp["dElevation_prev"] = (diag_exp["Elevation"] - diag_exp["prev_Elevation"]).abs()
raw_daz = (diag_exp["Azimuth"] - diag_exp["prev_Azimuth"]).abs()
diag_exp["dAzimuth_prev"] = np.minimum(raw_daz, 360.0 - raw_daz)

diag_exp["thermal_delta_prev_C"] = np.nan
valid_prev = diag_exp["prev_sample_pos"].notna().to_numpy()
if valid_prev.any():
    current_pos = diag_exp.loc[valid_prev, "sample_pos"].to_numpy(dtype=int)
    previous_pos = diag_exp.loc[valid_prev, "prev_sample_pos"].to_numpy(dtype=int)
    diag_exp.loc[valid_prev, "thermal_delta_prev_C"] = np.median(
        np.abs(X_exp[current_pos, -1, :] - X_exp[previous_pos, -1, :]), axis=1
    )

diag_exp["flag_geometry_near"] = (
    (diag_exp["dAzimuth_prev"] <= NEAR_AZ_DEG) &
    (diag_exp["dElevation_prev"] <= NEAR_EL_DEG)
)
diag_exp["flag_thermal_near"] = diag_exp["thermal_delta_prev_C"] <= NEAR_THERMAL_MEDIAN_C
diag_exp["flag_offset_jump"] = (
    (diag_exp["dXEL_prev"] > PROCEDURAL_DXEL_ARCSEC) |
    (diag_exp["dEL_prev"] > PROCEDURAL_DEL_ARCSEC)
)


diag_exp["flag_procedural_high"] = (
    diag_exp["flag_sequence_followup"] &
    diag_exp["flag_geometry_near"] &
    diag_exp["flag_thermal_near"] &
    diag_exp["flag_offset_jump"]
)
diag_exp["flag_procedural_ambiguous"] = (
    diag_exp["flag_sequence_followup"] & ~diag_exp["flag_procedural_high"]
)
diag_exp["procedural_score"] = (
    diag_exp["flag_sequence_followup"].astype(int) +
    diag_exp["flag_geometry_near"].astype(int) +
    diag_exp["flag_thermal_near"].astype(int) +
    2 * diag_exp["flag_offset_jump"].astype(int) +
    3 * diag_exp["flag_gross_invalid"].astype(int)
)

meta_exp = diag_exp
procedural_summary_df = pd.DataFrame([{
    "valid_windows": len(meta_exp),
    "sessions": meta_exp["session_id"].nunique(),
    "followup": int(meta_exp["flag_sequence_followup"].sum()),
    "procedural_high": int(meta_exp["flag_procedural_high"].sum()),
    "procedural_ambiguous": int(meta_exp["flag_procedural_ambiguous"].sum()),
    "gross_invalid": int(meta_exp["flag_gross_invalid"].sum()),
    "new_high": int((~meta_exp["in_original_dataset"] & meta_exp["flag_procedural_high"]).sum()),
}])
display(procedural_summary_df)

print('Cases with the strongest procedural evidence')
display(meta_exp.sort_values(
    ["flag_gross_invalid", "procedural_score", "dXEL_prev", "dEL_prev"], ascending=False
).head(30)[[
    "Timestamp", "Filename", "Source", "Azimuth", "Elevation", "Target", "El Offset",
    "dt_prev_source_min", "dAzimuth_prev", "dElevation_prev", "thermal_delta_prev_C",
    "dXEL_prev", "dEL_prev", "sequence_position", "procedural_score",
    "flag_procedural_high", "flag_gross_invalid", "in_original_dataset"
]])

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
plot_follow = meta_exp[meta_exp["flag_sequence_followup"]].copy()
colors = np.where(plot_follow["flag_procedural_high"], "tab:red", "tab:blue")
axs[0].scatter(plot_follow["dt_prev_source_min"], plot_follow["dXEL_prev"], c=colors, alpha=0.65)
axs[0].axhline(PROCEDURAL_DXEL_ARCSEC, color="tab:red", linestyle="--")
axs[0].set(xlabel='Time difference for the same source [min]', ylabel="Delta XEL [arcsec]", title='XEL jumps within sequences')
axs[0].grid(True, alpha=0.3)

axs[1].scatter(plot_follow["thermal_delta_prev_C"], plot_follow["dXEL_prev"], c=colors, alpha=0.65)
axs[1].axhline(PROCEDURAL_DXEL_ARCSEC, color="tab:red", linestyle="--")
axs[1].axvline(NEAR_THERMAL_MEDIAN_C, color="0.4", linestyle=":")
axs[1].set(xlabel='Median thermal difference [C]', ylabel="Delta XEL [arcsec]", title='Thermal continuity versus target jumps')
axs[1].grid(True, alpha=0.3)

quality_labels = np.select(
    [meta_exp["flag_gross_invalid"], meta_exp["flag_procedural_high"], meta_exp["flag_procedural_ambiguous"]],
    ["gross invalid", "procedural high", "follow-up ambiguous"], default="first/isolated"
)
pd.Series(quality_labels).value_counts().plot.bar(ax=axs[2], color=["tab:blue", "tab:orange", "tab:red", "0.4"])
axs[2].set_title('Target-quality classes')
axs[2].set_ylabel('samples')
axs[2].grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
is_original = meta_exp["in_original_dataset"].fillna(False).to_numpy(dtype=bool)
is_new = ~is_original
gross = meta_exp["flag_gross_invalid"].fillna(True).to_numpy(dtype=bool)
followup = meta_exp["flag_sequence_followup"].fillna(False).to_numpy(dtype=bool)
procedural_high = meta_exp["flag_procedural_high"].fillna(False).to_numpy(dtype=bool)

cohort_masks = {
    "baseline_original": is_original,
    "expanded_conservative": is_original | (is_new & ~gross & ~followup),
    "expanded_weighted": is_original | (is_new & ~gross & ~procedural_high),
    "expanded_all_plausible": ~gross,
    "raw_diagnostic": np.ones(len(meta_exp), dtype=bool),
}

cohort_weights = {}
for cohort_name, mask in cohort_masks.items():
    weights = np.ones(len(meta_exp), dtype=float)
    if cohort_name == "expanded_weighted":
        weights[is_new & ~followup] = 0.75
        weights[is_new & followup] = 0.35
    cohort_weights[cohort_name] = weights


fixed_val_start = pd.to_datetime(meta.iloc[val_idx]["Timestamp"].min())
fixed_test_start = pd.to_datetime(meta.iloc[test_idx]["Timestamp"].min())
meta_exp["raw_split"] = np.where(
    meta_exp["Timestamp"] < fixed_val_start, "train",
    np.where(meta_exp["Timestamp"] < fixed_test_start, "val", "test")
)

# Assign each complete session to the split containing its last scan.
session_end = meta_exp.groupby("session_id")["Timestamp"].transform("max")
meta_exp["split"] = np.where(
    session_end < fixed_val_start, "train",
    np.where(session_end < fixed_test_start, "val", "test")
)
crossing_sessions = meta_exp.groupby("session_id")["raw_split"].nunique()
print("Boundary validation:", fixed_val_start, "| boundary test:", fixed_test_start)
print('Sessions that would cross a split boundary:', int((crossing_sessions > 1).sum()))

cohort_rows = []
for cohort_name, mask in cohort_masks.items():
    for split_name in ["train", "val", "test"]:
        idx = mask & meta_exp["split"].eq(split_name).to_numpy()
        yy = np.abs(y_exp[idx])
        cohort_rows.append({
            "cohort": cohort_name,
            "split": split_name,
            "count": int(idx.sum()),
            "original_count": int((idx & is_original).sum()),
            "new_count": int((idx & is_new).sum()),
            "median_abs_target": float(np.median(yy)) if len(yy) else np.nan,
            "p90_abs_target": float(np.quantile(yy, 0.90)) if len(yy) else np.nan,
            "target_gt15_%": float((yy > 15).mean() * 100) if len(yy) else np.nan,
            "followup_%": float(followup[idx].mean() * 100) if idx.sum() else np.nan,
        })
cohort_audit_df = pd.DataFrame(cohort_rows)
display(cohort_audit_df)

fixed_eval_masks = {
    "trusted_original": is_original,
    "expanded_plausible": ~gross & ~procedural_high,
    "new_plausible": is_new & ~gross & ~procedural_high,
    "all_raw": np.ones(len(meta_exp), dtype=bool),
}

print('Common evaluation samples')
eval_rows = []
for eval_name, eval_mask in fixed_eval_masks.items():
    for split_name in ["val", "test"]:
        mask = eval_mask & meta_exp["split"].eq(split_name).to_numpy()
        eval_rows.append({"eval_set": eval_name, "split": split_name, "count": int(mask.sum())})
display(pd.DataFrame(eval_rows))


In [ ]:
# Reuse the original training-fitted builder to isolate the effect of added labels.

base_train_mask = cohort_masks["baseline_original"] & meta_exp["split"].eq("train").to_numpy()
if base_train_mask.sum() < 30:
    raise RuntimeError('Too few original samples in the expanded training set.')

n_pca_exp = min(8, max(1, int(base_train_mask.sum()) - 1))
fb_exp = FeatureBuilder(n_pca_current=n_pca_exp, n_pca_trend=n_pca_exp).fit(
    X_exp[base_train_mask], X_elev_exp[base_train_mask]
)
F_exp = fb_exp.transform(X_exp, X_elev_exp, meta_exp)

expanded_models = {
    "median": None,
    "ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
    ]),
    "huber": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000)),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=400, min_samples_leaf=6, max_features="sqrt",
            random_state=RANDOM_SEED, n_jobs=-1
        )),
    ]),
}

def weighted_median(values, weights):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]
    cutoff = 0.5 * weights.sum()
    return float(values[np.searchsorted(np.cumsum(weights), cutoff, side="left")])

def expanded_metrics(y_true, y_pred):
    err = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    return {
        "MAE": float(err.mean()),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "within_5_%": float((err <= SPEC_ARCSEC).mean() * 100),
        "p50_abs_error": float(np.quantile(err, 0.50)),
        "p90_abs_error": float(np.quantile(err, 0.90)),
        "p95_abs_error": float(np.quantile(err, 0.95)),
        "max_abs_error": float(err.max()),
    }

expanded_results = []
expanded_predictions = {}
for cohort_name, cohort_mask in cohort_masks.items():
    train_mask = cohort_mask & meta_exp["split"].eq("train").to_numpy()
    train_pos = np.flatnonzero(train_mask)
    weights_train = cohort_weights[cohort_name][train_mask]
    print(f"Training {cohort_name}: n={len(train_pos)}, effective_n={weights_train.sum():.1f}")

    for model_name, model in expanded_models.items():
        fit_note = "weighted" if not np.allclose(weights_train, 1.0) else "unweighted"
        if model is None:
            center = weighted_median(y_exp[train_mask], weights_train)
            pred_all = np.full(len(y_exp), center, dtype=float)
        else:
            fitted = clone(model)
            try:
                fitted.fit(F_exp.iloc[train_pos], y_exp[train_mask], model__sample_weight=weights_train)
            except (TypeError, ValueError):
                fitted.fit(F_exp.iloc[train_pos], y_exp[train_mask])
                fit_note = "sample_weight_not_supported"
            pred_all = np.asarray(fitted.predict(F_exp), dtype=float)
        expanded_predictions[(cohort_name, model_name)] = pred_all

        for split_name in ["val", "test"]:
            for eval_name, eval_mask in fixed_eval_masks.items():
                mask = eval_mask & meta_exp["split"].eq(split_name).to_numpy()
                if mask.sum() == 0:
                    continue
                row = {
                    "cohort": cohort_name,
                    "model": model_name,
                    "split": split_name,
                    "eval_set": eval_name,
                    "n_train": len(train_pos),
                    "effective_n_train": float(weights_train.sum()),
                    "n_eval": int(mask.sum()),
                    "fit_note": fit_note,
                }
                row.update(expanded_metrics(y_exp[mask], pred_all[mask]))
                expanded_results.append(row)

expanded_ablation_df = pd.DataFrame(expanded_results)
trusted_test_results = expanded_ablation_df[
    (expanded_ablation_df["split"] == "test") &
    (expanded_ablation_df["eval_set"] == "trusted_original")
].sort_values("MAE")
print('Results on the same trusted original test')
display(trusted_test_results)

print('Results on plausible new test samples')
display(expanded_ablation_df[
    (expanded_ablation_df["split"] == "test") &
    (expanded_ablation_df["eval_set"] == "new_plausible")
].sort_values("MAE"))


In [ ]:
trusted_val = expanded_ablation_df[
    (expanded_ablation_df["split"] == "val") &
    (expanded_ablation_df["eval_set"] == "trusted_original")
].copy()
selected_pairs = trusted_val.loc[trusted_val.groupby("cohort")["MAE"].idxmin(), ["cohort", "model", "MAE"]]
selected_pairs = selected_pairs.rename(columns={"MAE": "validation_MAE"})

selected_rows = []
for _, selected in selected_pairs.iterrows():
    row = expanded_ablation_df[
        (expanded_ablation_df["cohort"] == selected["cohort"]) &
        (expanded_ablation_df["model"] == selected["model"]) &
        (expanded_ablation_df["split"] == "test") &
        (expanded_ablation_df["eval_set"] == "trusted_original")
    ].iloc[0].to_dict()
    row["validation_MAE"] = selected["validation_MAE"]
    selected_rows.append(row)
selected_cohort_df = pd.DataFrame(selected_rows).sort_values("MAE")
print('Model selected on validation only for each cohort')
display(selected_cohort_df)

baseline_sel = selected_cohort_df[selected_cohort_df["cohort"] == "baseline_original"].iloc[0]
baseline_pred = expanded_predictions[(baseline_sel["cohort"], baseline_sel["model"])]
fixed_test_mask = fixed_eval_masks["trusted_original"] & meta_exp["split"].eq("test").to_numpy()
test_positions = np.flatnonzero(fixed_test_mask)
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
for _, selected in selected_cohort_df.iterrows():
    pred = expanded_predictions[(selected["cohort"], selected["model"])]
    paired_delta = []
    for _ in range(2000):
        sampled = rng.choice(test_positions, size=len(test_positions), replace=True)
        mae_candidate = np.mean(np.abs(y_exp[sampled] - pred[sampled]))
        mae_baseline = np.mean(np.abs(y_exp[sampled] - baseline_pred[sampled]))
        paired_delta.append(mae_candidate - mae_baseline)
    paired_delta = np.asarray(paired_delta)
    bootstrap_rows.append({
        "cohort": selected["cohort"],
        "selected_model": selected["model"],
        "delta_MAE_vs_baseline": float(paired_delta.mean()),
        "ci95_low": float(np.quantile(paired_delta, 0.025)),
        "ci95_high": float(np.quantile(paired_delta, 0.975)),
        "probability_better_%": float((paired_delta < 0).mean() * 100),
    })
paired_bootstrap_df = pd.DataFrame(bootstrap_rows)
print('Paired bootstrap on the original test: negative delta means improvement')
display(paired_bootstrap_df.sort_values("delta_MAE_vs_baseline"))


base_train_pos = np.flatnonzero(cohort_masks["baseline_original"] & meta_exp["split"].eq("train").to_numpy())
new_conservative_train = np.flatnonzero(
    is_new & ~gross & ~followup & meta_exp["split"].eq("train").to_numpy()
)
learning_rows = []
fractions = [0.0, 0.25, 0.50, 0.75, 1.0]
for fraction in fractions:
    repeats = 1 if fraction in [0.0, 1.0] else 5
    for repeat in range(repeats):
        local_rng = np.random.default_rng(RANDOM_SEED + repeat + int(fraction * 1000))
        n_add = int(round(len(new_conservative_train) * fraction))
        added = local_rng.choice(new_conservative_train, size=n_add, replace=False) if n_add else np.array([], dtype=int)
        train_pos = np.concatenate([base_train_pos, added])
        lc_model = clone(expanded_models["huber"])
        lc_model.fit(F_exp.iloc[train_pos], y_exp[train_pos])
        pred = lc_model.predict(F_exp.iloc[test_positions])
        metrics = expanded_metrics(y_exp[test_positions], pred)
        learning_rows.append({
            "fraction_new": fraction,
            "new_added": n_add,
            "n_train": len(train_pos),
            "repeat": repeat,
            **metrics,
        })
expanded_learning_curve_df = pd.DataFrame(learning_rows)
learning_summary_df = expanded_learning_curve_df.groupby(["fraction_new", "new_added", "n_train"]).agg(
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    within_5_mean=("within_5_%", "mean"), p90_mean=("p90_abs_error", "mean")
).reset_index()
display(learning_summary_df)


In [ ]:
expanded_gru_results = []
expanded_gru_predictions = {}
if RUN_EXPANDED_GRU and "TF_AVAILABLE" in globals() and TF_AVAILABLE:
    gru_cohorts = ["baseline_original", "expanded_conservative", "expanded_weighted"]
    seq_pca_exp = SequencePCA(n_components=min(16, int(base_train_mask.sum()) - 1)).fit(X_exp[base_train_mask])
    Z_exp = seq_pca_exp.transform(X_exp)

    az_rad = np.deg2rad(pd.to_numeric(meta_exp["Azimuth"], errors="coerce").fillna(0).to_numpy(dtype=float))
    context_raw = np.column_stack([
        X_elev_exp.ravel(), np.sin(az_rad), np.cos(az_rad)
    ]).astype("float32")
    context_scaler_exp = StandardScaler().fit(context_raw[base_train_mask])
    C_exp = context_scaler_exp.transform(context_raw).astype("float32")

    val_mask_gru = fixed_eval_masks["trusted_original"] & meta_exp["split"].eq("val").to_numpy()
    test_mask_gru = fixed_eval_masks["trusted_original"] & meta_exp["split"].eq("test").to_numpy()
    plausible_test_gru = fixed_eval_masks["new_plausible"] & meta_exp["split"].eq("test").to_numpy()

    for cohort_offset, cohort_name in enumerate(gru_cohorts):
        train_mask = cohort_masks[cohort_name] & meta_exp["split"].eq("train").to_numpy()
        sample_weight = cohort_weights[cohort_name][train_mask].astype("float32")
        y_scaler_exp = StandardScaler().fit(y_exp[train_mask].reshape(-1, 1))
        ys_train = y_scaler_exp.transform(y_exp[train_mask].reshape(-1, 1)).astype("float32")
        ys_val = y_scaler_exp.transform(y_exp[val_mask_gru].reshape(-1, 1)).astype("float32")

        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_SEED + cohort_offset)
        seq_in = keras.Input(shape=(SEQ_LEN, Z_exp.shape[-1]), name="thermal_pca_history")
        context_in = keras.Input(shape=(3,), name="elevation_azimuth")
        h_seq = layers.GRU(32, dropout=0.25, recurrent_dropout=0.0)(seq_in)
        h_seq = layers.Dense(16, activation="gelu")(h_seq)
        h_ctx = layers.Dense(8, activation="gelu")(context_in)
        h = layers.Concatenate()([h_seq, h_ctx])
        h = layers.Dropout(0.35)(h)
        h = layers.Dense(16, activation="gelu")(h)
        out = layers.Dense(1, dtype="float32")(h)
        model = keras.Model([seq_in, context_in], out)
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-3),
            loss="mae"
        )
        callbacks = [
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-6),
        ]
        history = model.fit(
            [Z_exp[train_mask], C_exp[train_mask]], ys_train,
            sample_weight=sample_weight,
            validation_data=([Z_exp[val_mask_gru], C_exp[val_mask_gru]], ys_val),
            epochs=300, batch_size=16, callbacks=callbacks, verbose=0,
        )
        pred_all = y_scaler_exp.inverse_transform(
            model.predict([Z_exp, C_exp], verbose=0)
        ).ravel()
        expanded_gru_predictions[cohort_name] = pred_all
        for eval_name, eval_mask in [("trusted_original", test_mask_gru), ("new_plausible", plausible_test_gru)]:
            if eval_mask.sum() == 0:
                continue
            expanded_gru_results.append({
                "cohort": cohort_name,
                "model": "compact_pca_gru",
                "eval_set": eval_name,
                "n_train": int(train_mask.sum()),
                "effective_n_train": float(sample_weight.sum()),
                "epochs": len(history.history["loss"]),
                **expanded_metrics(y_exp[eval_mask], pred_all[eval_mask]),
            })
    expanded_gru_df = pd.DataFrame(expanded_gru_results)
    display(expanded_gru_df.sort_values(["eval_set", "MAE"]))
else:
    print('Expanded GRU not run. RUN_EXPANDED_GRU=', RUN_EXPANDED_GRU, "TF_AVAILABLE=", globals().get("TF_AVAILABLE", False))


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 10))

selected_plot = selected_cohort_df.sort_values("MAE")
axs[0, 0].barh(selected_plot["cohort"], selected_plot["MAE"], color="tab:blue")
axs[0, 0].axvline(SPEC_ARCSEC, color="tab:green", linestyle="--", label='5-arcsec requirement')
axs[0, 0].set_title('MAE on the common original test')
axs[0, 0].set_xlabel("MAE [arcsec]")
axs[0, 0].legend()
axs[0, 0].grid(True, axis="x", alpha=0.3)

axs[0, 1].barh(selected_plot["cohort"], selected_plot["within_5_%"], color="tab:orange")
axs[0, 1].set_title('Within-threshold samples on the common test')
axs[0, 1].set_xlabel('within 5 arcsec [%]')
axs[0, 1].set_xlim(0, 100)
axs[0, 1].grid(True, axis="x", alpha=0.3)

axs[1, 0].errorbar(
    learning_summary_df["new_added"], learning_summary_df["MAE_mean"],
    yerr=learning_summary_df["MAE_std"].fillna(0), marker="o", capsize=4
)
axs[1, 0].axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[1, 0].set(xlabel='new conservative samples added', ylabel="MAE [arcsec]", title="Learning curve Huber")
axs[1, 0].grid(True, alpha=0.3)

target_groups = pd.DataFrame({
    "abs_target": np.abs(y_exp),
    "quality": np.select(
        [gross, procedural_high, followup],
        ["gross invalid", "procedural high", "follow-up"], default="first/isolated"
    )
})
for name, sub in target_groups.groupby("quality"):
    axs[1, 1].hist(sub["abs_target"].clip(upper=100), bins=30, alpha=0.45, density=True, label=name)
axs[1, 1].axvline(15, color="tab:red", linestyle="--", label="15 arcsec")
axs[1, 1].set(xlabel='|XEL| [arcsec], clipped at 100', ylabel="densita'", title='Targets by quality class')
axs[1, 1].legend()
axs[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

audit_export_cols = [
    "Timestamp", "Filename", "Source", "Azimuth", "Elevation", "Target", "El Offset",
    "in_embedded_total", "in_original_dataset", "session_id", "sequence_position", "sequence_size",
    "dt_prev_source_min", "dAzimuth_prev", "dElevation_prev", "thermal_delta_prev_C",
    "dXEL_prev", "dEL_prev", "procedural_score", "flag_sequence_followup",
    "flag_procedural_high", "flag_procedural_ambiguous", "flag_gross_invalid", "split"
]
meta_exp[audit_export_cols].to_csv("expanded_target_quality_audit.csv", index=False)
cohort_audit_df.to_csv("expanded_cohort_audit.csv", index=False)
expanded_ablation_df.to_csv("expanded_ablation_results.csv", index=False)
selected_cohort_df.to_csv("expanded_selected_models.csv", index=False)
paired_bootstrap_df.to_csv("expanded_paired_bootstrap.csv", index=False)
expanded_learning_curve_df.to_csv("expanded_learning_curve.csv", index=False)
if "expanded_gru_df" in globals():
    expanded_gru_df.to_csv("expanded_gru_results.csv", index=False)

print('Exports completed:')
print(" - expanded_target_quality_audit.csv")
print(" - expanded_cohort_audit.csv")
print(" - expanded_ablation_results.csv")
print(" - expanded_selected_models.csv")
print(" - expanded_paired_bootstrap.csv")
print(" - expanded_learning_curve.csv")
if "expanded_gru_df" in globals():
    print(" - expanded_gru_results.csv")


## Pulizia fisica delle sessioni e recupero robusto delle finestre

Questa sezione verifica se la label rappresenta davvero lo stato termo-strutturale prima di una correzione. Confronta diverse definizioni di sessione, usa solo la prima scansione, cerca firme cumulative degli offset, recupera in modo controllato finestre con pochi buchi o pochi sensori anomali e seleziona il modello con cross-validation temporale a blocchi.

In [ ]:
MAX_MISSING_GRID_STEPS_ROBUST = 2
MAX_BAD_SENSORS_PER_WINDOW = 3
ISOLATED_SPIKE_NEIGHBOR_TOL_C = 2.0
SESSION_GAPS_TO_TEST_MIN = [30.0, 60.0, 120.0, 180.0]
SESSION_GAP_SELECTED_MIN = 60.0  
RUN_FIRST_ONLY_GRU = True

def repair_small_sensor_artifacts(values, rate_threshold=MAX_SENSOR_RATE_C,
                                  max_bad_sensors=MAX_BAD_SENSORS_PER_WINDOW,
                                  neighbor_tol=ISOLATED_SPIKE_NEIGHBOR_TOL_C):
    """Repair isolated spikes and replace at most max_bad_sensors remaining bad channels."""
    repaired = np.asarray(values, dtype="float32").copy()
    if repaired.shape[0] >= 3:
        midpoint = 0.5 * (repaired[:-2] + repaired[2:])
        isolated = (
            (np.abs(repaired[1:-1] - midpoint) > rate_threshold) &
            (np.abs(repaired[:-2] - repaired[2:]) <= neighbor_tol)
        )
        rows, cols = np.where(isolated)
        repaired[rows + 1, cols] = midpoint[rows, cols]
        isolated_count = int(len(rows))
    else:
        isolated_count = 0

    bad_sensors = np.flatnonzero((np.abs(np.diff(repaired, axis=0)) > rate_threshold).any(axis=0))
    if len(bad_sensors) > max_bad_sensors:
        return repaired, isolated_count, len(bad_sensors), False

    if len(bad_sensors):
        good_sensors = np.setdiff1d(np.arange(repaired.shape[1]), bad_sensors)
        if len(good_sensors) == 0:
            return repaired, isolated_count, len(bad_sensors), False
        spatial_curve = np.median(repaired[:, good_sensors], axis=1)
        for sensor_idx in bad_sensors:
            sensor_offset = np.median(repaired[:, sensor_idx] - spatial_curve)
            repaired[:, sensor_idx] = spatial_curve + sensor_offset

    accepted = np.abs(np.diff(repaired, axis=0)).max() <= rate_threshold
    return repaired, isolated_count, len(bad_sensors), bool(accepted)


def build_supervised_windows_robust(df, sensor_cols, target_table, seq_len=SEQ_LEN,
                                    expected_step_sec=EXPECTED_STEP_SEC,
                                    max_missing_steps=MAX_MISSING_GRID_STEPS_ROBUST):
    df = df.sort_values("Timestamp").copy()
    target_df = target_table.rename(columns={"Elevation": "SRT Elevation Ange[DEG]"}).copy()

    sensor_ts = df[["Timestamp"] + sensor_cols].dropna(subset=["Timestamp"]).copy()
    sensor_ts["GridTime"] = sensor_ts["Timestamp"].dt.floor(f"{expected_step_sec}s")
    observed_grid = sensor_ts.groupby("GridTime", sort=True)[sensor_cols].mean().astype("float32")
    full_index = pd.date_range(observed_grid.index.min(), observed_grid.index.max(), freq=f"{expected_step_sec}s")
    full_grid = observed_grid.reindex(full_index)

    X_out, elev_out, y_out, rows_out = [], [], [], []
    skip = {
        "too_early": 0,
        "missing_grid_gt_limit": 0,
        "nan_after_interpolation": 0,
        "broad_thermal_jump": 0,
        "recovered_missing_grid": 0,
        "repaired_sensor_window": 0,
    }
    step = pd.Timedelta(seconds=expected_step_sec)

    for _, row in target_df.iterrows():
        target_time = row["Timestamp"]
        end_time = target_time.floor(f"{expected_step_sec}s") if INCLUDE_CURRENT_TEMP else target_time.floor(f"{expected_step_sec}s") - step
        expected_index = pd.date_range(end=end_time, periods=seq_len, freq=step)
        if expected_index[0] < full_grid.index.min():
            skip["too_early"] += 1
            continue

        missing_steps = int((~expected_index.isin(observed_grid.index)).sum())
        if missing_steps > max_missing_steps:
            skip["missing_grid_gt_limit"] += 1
            continue

        window_df = full_grid.loc[expected_index, sensor_cols].copy()
        if missing_steps:
            window_df = window_df.interpolate(axis=0, method="linear", limit=max_missing_steps, limit_direction="both")
        values = window_df.to_numpy(dtype="float32")
        if not np.isfinite(values).all():
            skip["nan_after_interpolation"] += 1
            continue

        repaired, isolated_count, bad_sensor_count, accepted = repair_small_sensor_artifacts(values)
        if not accepted:
            skip["broad_thermal_jump"] += 1
            continue
        if missing_steps:
            skip["recovered_missing_grid"] += 1
        if isolated_count or bad_sensor_count:
            skip["repaired_sensor_window"] += 1

        X_out.append(repaired)
        elev_out.append(float(row["SRT Elevation Ange[DEG]"]))
        y_out.append(float(row[TARGET]))
        rows_out.append({
            "Timestamp": target_time,
            "Elevation": float(row["SRT Elevation Ange[DEG]"]),
            "Azimuth": float(row["Azimuth"]) if pd.notna(row.get("Azimuth", np.nan)) else np.nan,
            "Target": float(row[TARGET]),
            "Filename": row.get("Filename", None),
            "missing_grid_steps": missing_steps,
            "isolated_spikes_repaired": isolated_count,
            "bad_sensors_replaced": bad_sensor_count,
        })

    return (
        np.asarray(X_out, dtype="float32"),
        np.asarray(elev_out, dtype="float32").reshape(-1, 1),
        np.asarray(y_out, dtype="float32"),
        pd.DataFrame(rows_out),
        skip,
    )


X_robust, X_elev_robust, y_robust, meta_robust, skip_robust = build_supervised_windows_robust(
    total_real_raw, total_sensor_cols, pointing_master
)
meta_robust = meta_robust.merge(
    pointing_master[[
        "Timestamp", "El Offset", "Source", "in_embedded_total", "in_original_dataset",
        "flag_gross_invalid", "abs_xel", "abs_el"
    ]], on="Timestamp", how="left", validate="one_to_one"
)
order_robust = np.argsort(meta_robust["Timestamp"].to_numpy())
X_robust = X_robust[order_robust]
X_elev_robust = X_elev_robust[order_robust]
y_robust = y_robust[order_robust]
meta_robust = meta_robust.iloc[order_robust].reset_index(drop=True)
meta_robust["sample_pos"] = np.arange(len(meta_robust))
if not np.allclose(meta_robust["Target"].to_numpy(dtype=float), y_robust):
    raise RuntimeError('y/meta misalignment in the robust pipeline.')

strict_times = set(pd.to_datetime(meta_exp["Timestamp"]))
meta_robust["strict_window_available"] = meta_robust["Timestamp"].isin(strict_times)
thermal_recovery_df = pd.DataFrame([{
    "master_targets": len(pointing_master),
    "strict_valid_windows": len(meta_exp),
    "robust_valid_windows": len(meta_robust),
    "net_windows_recovered": len(meta_robust) - len(meta_exp),
    "robust_original": int(meta_robust["in_original_dataset"].sum()),
    "robust_new": int((~meta_robust["in_original_dataset"]).sum()),
    "recovered_not_in_strict": int((~meta_robust["strict_window_available"]).sum()),
    "windows_with_grid_interpolation": int((meta_robust["missing_grid_steps"] > 0).sum()),
    "windows_with_sensor_repair": int((meta_robust["bad_sensors_replaced"] > 0).sum()),
}])
print('Robust-pipeline rejections:', skip_robust)
display(thermal_recovery_df)

print('Distribution of replaced sensor counts')
display(meta_robust["bad_sensors_replaced"].value_counts().sort_index().rename_axis("bad_sensors").reset_index(name="windows"))


In [ ]:
def make_session_diagnostics(meta_base, X_base, gap_min):
    out = meta_base.copy().sort_values("Timestamp").reset_index(drop=True)
    out["sample_pos"] = np.arange(len(out))
    out["Source"] = out["Source"].fillna("unknown").astype(str)
    out["dt_prev_source_min"] = out.groupby("Source", sort=False)["Timestamp"].diff().dt.total_seconds() / 60
    session_break = out["dt_prev_source_min"].isna() | (out["dt_prev_source_min"] > gap_min)
    out["source_session_num"] = session_break.groupby(out["Source"]).cumsum().astype(int)
    out["session_id"] = out["Source"] + "::" + out["source_session_num"].astype(str)
    out["sequence_position"] = out.groupby("session_id", sort=False).cumcount()
    out["sequence_size"] = out.groupby("session_id", sort=False)["session_id"].transform("size")
    out["flag_sequence_followup"] = out["sequence_position"] > 0

    for col in ["Target", "El Offset", "Elevation", "Azimuth", "sample_pos"]:
        out[f"prev_{col}"] = out.groupby("session_id", sort=False)[col].shift(1)
    out["session_first_target"] = out.groupby("session_id", sort=False)["Target"].transform("first")
    out["dXEL_prev"] = (out["Target"] - out["prev_Target"]).abs()
    out["dEL_prev"] = (out["El Offset"] - out["prev_El Offset"]).abs()
    out["dElevation_prev"] = (out["Elevation"] - out["prev_Elevation"]).abs()
    raw_daz = (out["Azimuth"] - out["prev_Azimuth"]).abs()
    out["dAzimuth_prev"] = np.minimum(raw_daz, 360.0 - raw_daz)

    out["thermal_delta_prev_C"] = np.nan
    valid_prev = out["prev_sample_pos"].notna().to_numpy()
    if valid_prev.any():
        current_pos = out.loc[valid_prev, "sample_pos"].to_numpy(dtype=int)
        previous_pos = out.loc[valid_prev, "prev_sample_pos"].to_numpy(dtype=int)
        out.loc[valid_prev, "thermal_delta_prev_C"] = np.median(
            np.abs(X_base[current_pos, -1, :] - X_base[previous_pos, -1, :]), axis=1
        )
    out["flag_geometry_near"] = (out["dAzimuth_prev"] <= NEAR_AZ_DEG) & (out["dElevation_prev"] <= NEAR_EL_DEG)
    out["flag_thermal_near"] = out["thermal_delta_prev_C"] <= NEAR_THERMAL_MEDIAN_C
    out["flag_offset_jump"] = (out["dXEL_prev"] > PROCEDURAL_DXEL_ARCSEC) | (out["dEL_prev"] > PROCEDURAL_DEL_ARCSEC)
    out["flag_procedural_high"] = (
        out["flag_sequence_followup"] & out["flag_geometry_near"] &
        out["flag_thermal_near"] & out["flag_offset_jump"]
    )
    return out


session_gap_rows = []
session_versions = {}
for gap_min in SESSION_GAPS_TO_TEST_MIN:
    session_df = make_session_diagnostics(meta_robust, X_robust, gap_min)
    session_versions[gap_min] = session_df
    first = session_df["sequence_position"].eq(0).to_numpy()
    new = ~session_df["in_original_dataset"].to_numpy(dtype=bool)
    gross_local = session_df["flag_gross_invalid"].to_numpy(dtype=bool)
    for split_name, split_mask in {
        "all": np.ones(len(session_df), dtype=bool),
        "pre_validation": session_df["Timestamp"].lt(fixed_val_start).to_numpy(),
        "final_test_period": session_df["Timestamp"].ge(fixed_test_start).to_numpy(),
    }.items():
        mask = split_mask
        session_gap_rows.append({
            "gap_min": gap_min,
            "period": split_name,
            "samples": int(mask.sum()),
            "sessions": int(session_df.loc[mask, "session_id"].nunique()),
            "first_scans": int((mask & first).sum()),
            "followups": int((mask & ~first).sum()),
            "new_first_plausible": int((mask & first & new & ~gross_local).sum()),
            "procedural_high": int((mask & session_df["flag_procedural_high"].to_numpy()).sum()),
        })
session_gap_sensitivity_df = pd.DataFrame(session_gap_rows)
display(session_gap_sensitivity_df)

gap_all = session_gap_sensitivity_df[session_gap_sensitivity_df["period"] == "all"]
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(gap_all["gap_min"], gap_all["sessions"], marker="o", label='sessions')
axs[0].plot(gap_all["gap_min"], gap_all["first_scans"], marker="s", label='first scans')
axs[0].set(xlabel='session gap [min]', ylabel='count', title='Segmentation sensitivity')
axs[0].legend(); axs[0].grid(True, alpha=0.3)
axs[1].plot(gap_all["gap_min"], gap_all["new_first_plausible"], marker="o", color="tab:green")
axs[1].set(xlabel='session gap [min]', ylabel='new samples', title='Plausible new first scans')
axs[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
session_selected = session_versions[SESSION_GAP_SELECTED_MIN].copy()
session_selected["position_band"] = np.select(
    [session_selected["sequence_position"] == 0, session_selected["sequence_position"] == 1],
    ["first", "second"], default="third_or_later"
)
session_selected["abs_target"] = session_selected["Target"].abs()
session_selected["target_delta_signed"] = session_selected["Target"] - session_selected["prev_Target"]
session_selected["abs_ratio_vs_prev"] = session_selected["abs_target"] / session_selected["prev_Target"].abs().clip(lower=0.5)
session_selected["cum_previous_xel"] = session_selected.groupby("session_id")["Target"].cumsum() - session_selected["Target"]
session_selected["xel_reconstructed_cumulative"] = session_selected["Target"] + session_selected["cum_previous_xel"]
session_selected["observed_distance_from_first"] = (session_selected["Target"] - session_selected["session_first_target"]).abs()
session_selected["cumulative_distance_from_first"] = (
    session_selected["xel_reconstructed_cumulative"] - session_selected["session_first_target"]
).abs()

sequence_position_df = session_selected.groupby("position_band", observed=True).agg(
    count=("Target", "size"),
    mean_target=("Target", "mean"),
    median_abs_target=("abs_target", "median"),
    p90_abs_target=("abs_target", lambda s: s.quantile(0.90)),
    median_dXEL_prev=("dXEL_prev", "median"),
    median_thermal_delta=("thermal_delta_prev_C", "median"),
    gross_invalid_pct=("flag_gross_invalid", lambda s: s.mean() * 100),
).reset_index()
display(sequence_position_df)

follow_selected = session_selected[session_selected["flag_sequence_followup"]].copy()
cumulative_signature_df = pd.DataFrame([{
    "followup_count": len(follow_selected),
    "pct_abs_target_smaller_than_previous": float((follow_selected["abs_ratio_vs_prev"] < 1).mean() * 100),
    "median_abs_ratio_vs_previous": float(follow_selected["abs_ratio_vs_prev"].median()),
    "corr_target_with_previous": float(follow_selected[["Target", "prev_Target"]].corr().iloc[0, 1]),
    "median_observed_distance_from_first": float(follow_selected["observed_distance_from_first"].median()),
    "median_cumulative_distance_from_first": float(follow_selected["cumulative_distance_from_first"].median()),
    "cumulative_reconstruction_better_%": float((
        follow_selected["cumulative_distance_from_first"] < follow_selected["observed_distance_from_first"]
    ).mean() * 100),
}])
display(cumulative_signature_df)

near_pairs = follow_selected[
    follow_selected["flag_geometry_near"] & follow_selected["flag_thermal_near"]
].copy()
near_pair_stability_df = pd.DataFrame([{
    "near_pairs": len(near_pairs),
    "median_abs_dXEL": float(near_pairs["dXEL_prev"].median()) if len(near_pairs) else np.nan,
    "p90_abs_dXEL": float(near_pairs["dXEL_prev"].quantile(.90)) if len(near_pairs) else np.nan,
    "pct_dXEL_gt15": float((near_pairs["dXEL_prev"] > 15).mean() * 100) if len(near_pairs) else np.nan,
    "median_abs_dEL": float(near_pairs["dEL_prev"].median()) if len(near_pairs) else np.nan,
}])
display(near_pair_stability_df)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
axs[0].scatter(follow_selected["prev_Target"], follow_selected["Target"], alpha=0.45)
lims = np.nanquantile(np.r_[follow_selected["prev_Target"], follow_selected["Target"]], [.01, .99])
axs[0].plot(lims, lims, "k--")
axs[0].set(xlabel='Previous XEL', ylabel='Current XEL', title='Within-sequence persistence')
axs[0].grid(True, alpha=0.3)

session_selected.boxplot(column="abs_target", by="position_band", ax=axs[1], showfliers=False)
axs[1].set_title('|XEL| by sequence position'); axs[1].set_xlabel(""); axs[1].set_ylabel("arcsec")

axs[2].scatter(follow_selected["observed_distance_from_first"], follow_selected["cumulative_distance_from_first"], alpha=0.45)
max_lim = min(100, np.nanquantile(np.r_[follow_selected["observed_distance_from_first"], follow_selected["cumulative_distance_from_first"]], .95))
axs[2].plot([0, max_lim], [0, max_lim], "k--")
axs[2].set(xlim=(0, max_lim), ylim=(0, max_lim), xlabel='observed distance from the first scan', ylabel='cumulatively reconstructed distance', title='Cumulative-signature test')
axs[2].grid(True, alpha=0.3)
plt.suptitle(""); plt.tight_layout(); plt.show()


In [ ]:
meta_first = session_selected.copy()
session_end_first = meta_first.groupby("session_id")["Timestamp"].transform("max")
meta_first["split_final"] = np.where(
    session_end_first < fixed_val_start, "train",
    np.where(session_end_first < fixed_test_start, "val", "test")
)

first_scan = meta_first["sequence_position"].eq(0).to_numpy()
original_robust = meta_first["in_original_dataset"].to_numpy(dtype=bool)
gross_robust = meta_first["flag_gross_invalid"].to_numpy(dtype=bool)
strict_available = meta_first["strict_window_available"].to_numpy(dtype=bool)

first_cohort_masks = {
    "original_first_strict_window": first_scan & original_robust & ~gross_robust & strict_available,
    "original_first_robust_window": first_scan & original_robust & ~gross_robust,
    "expanded_first_robust_window": first_scan & ~gross_robust,
}

first_cohort_rows = []
for cohort_name, cohort_mask in first_cohort_masks.items():
    for split_name in ["train", "val", "test"]:
        mask = cohort_mask & meta_first["split_final"].eq(split_name).to_numpy()
        yy = np.abs(y_robust[mask])
        first_cohort_rows.append({
            "cohort": cohort_name,
            "split": split_name,
            "count": int(mask.sum()),
            "original_count": int((mask & original_robust).sum()),
            "new_count": int((mask & ~original_robust).sum()),
            "recovered_window_count": int((mask & ~strict_available).sum()),
            "median_abs_target": float(np.median(yy)) if len(yy) else np.nan,
            "p90_abs_target": float(np.quantile(yy, .90)) if len(yy) else np.nan,
            "target_gt15_%": float((yy > 15).mean() * 100) if len(yy) else np.nan,
        })
first_cohort_audit_df = pd.DataFrame(first_cohort_rows)
display(first_cohort_audit_df)

first_eval_masks = {
    "original_first_common": first_cohort_masks["original_first_strict_window"],
    "original_first_robust": first_cohort_masks["original_first_robust_window"],
    "new_first_robust": first_scan & ~original_robust & ~gross_robust,
}

boundary_cross_first = meta_first.groupby("session_id").apply(
    lambda g: int((g["Timestamp"] < fixed_test_start).any() and (g["Timestamp"] >= fixed_test_start).any())
).sum()
print('Selected session gap:', SESSION_GAP_SELECTED_MIN, "min")
print('Sessions crossing the final boundary and assigned entirely to the test:', int(boundary_cross_first))


In [ ]:
first_models = {
    "median": None,
    "ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
    ]),
    "huber": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=3000)),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300, min_samples_leaf=6, max_features="sqrt",
            random_state=RANDOM_SEED, n_jobs=-1
        )),
    ]),
}

cv_candidate_masks = {
    "original_first_strict": first_cohort_masks["original_first_strict_window"],
    "original_first_robust": first_cohort_masks["original_first_robust_window"],
    "expanded_first_robust": first_cohort_masks["expanded_first_robust_window"],
}
cv_reference_mask = first_eval_masks["original_first_common"]
pretest_sessions_df = (
    meta_first[["session_id", "Timestamp"]]
    .groupby("session_id", as_index=False)["Timestamp"].max()
)
pretest_sessions_df = (
    pretest_sessions_df[pretest_sessions_df["Timestamp"] < fixed_test_start]
    .sort_values("Timestamp").reset_index(drop=True)
)
session_tscv = TimeSeriesSplit(n_splits=4)
blocked_cv_rows = []

for fold, (train_session_pos, eval_session_pos) in enumerate(session_tscv.split(pretest_sessions_df), start=1):
    train_sessions = set(pretest_sessions_df.iloc[train_session_pos]["session_id"])
    eval_sessions = set(pretest_sessions_df.iloc[eval_session_pos]["session_id"])
    common_base_train = (
        first_cohort_masks["original_first_strict_window"] &
        meta_first["session_id"].isin(train_sessions).to_numpy()
    )
    fold_eval = cv_reference_mask & meta_first["session_id"].isin(eval_sessions).to_numpy()
    if common_base_train.sum() < 20 or fold_eval.sum() < 5:
        print('Fold skipped because of too few samples:', fold, int(common_base_train.sum()), int(fold_eval.sum()))
        continue

    n_pca_fold = min(8, int(common_base_train.sum()) - 1)
    fb_fold = FeatureBuilder(n_pca_current=n_pca_fold, n_pca_trend=n_pca_fold).fit(
        X_robust[common_base_train], X_elev_robust[common_base_train]
    )
    fold_needed = meta_first["session_id"].isin(train_sessions | eval_sessions).to_numpy()
    fold_positions = np.flatnonzero(fold_needed)
    F_fold = fb_fold.transform(
        X_robust[fold_positions], X_elev_robust[fold_positions], meta_first.iloc[fold_positions]
    )
    feature_by_pos = {pos: local for local, pos in enumerate(fold_positions)}
    eval_pos = np.flatnonzero(fold_eval)
    eval_local = [feature_by_pos[pos] for pos in eval_pos]

    for cohort_name, cohort_mask in cv_candidate_masks.items():
        train_mask = cohort_mask & meta_first["session_id"].isin(train_sessions).to_numpy()
        train_pos = np.flatnonzero(train_mask)
        train_local = [feature_by_pos[pos] for pos in train_pos]
        for model_name, model in first_models.items():
            if model is None:
                pred = np.full(len(eval_pos), np.median(y_robust[train_pos]), dtype=float)
            else:
                fitted = clone(model)
                fitted.fit(F_fold.iloc[train_local], y_robust[train_pos])
                pred = fitted.predict(F_fold.iloc[eval_local])
            metrics = expanded_metrics(y_robust[eval_pos], pred)
            blocked_cv_rows.append({
                "fold": fold,
                "cohort": cohort_name,
                "model": model_name,
                "n_train": len(train_pos),
                "n_eval": len(eval_pos),
                "eval_start": meta_first.iloc[eval_pos]["Timestamp"].min(),
                "eval_end": meta_first.iloc[eval_pos]["Timestamp"].max(),
                **metrics,
            })

blocked_cv_df = pd.DataFrame(blocked_cv_rows)
blocked_cv_summary_df = blocked_cv_df.groupby(["cohort", "model"]).agg(
    folds=("fold", "nunique"),
    n_eval_total=("n_eval", "sum"),
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    within_5_mean=("within_5_%", "mean"),
    p90_mean=("p90_abs_error", "mean"),
).reset_index().sort_values(["cohort", "MAE_mean"])
print('Temporal-fold results')
display(blocked_cv_df)
print('Blocked cross-validation summary')
display(blocked_cv_summary_df)


In [ ]:
selected_first_models = blocked_cv_summary_df.loc[
    blocked_cv_summary_df.groupby("cohort")["MAE_mean"].idxmin()
].copy()
display(selected_first_models)

final_base_train = (
    first_cohort_masks["original_first_strict_window"] &
    meta_first["split_final"].isin(["train", "val"]).to_numpy()
)
final_test_common = (
    first_eval_masks["original_first_common"] &
    meta_first["split_final"].eq("test").to_numpy()
)
final_test_new = (
    first_eval_masks["new_first_robust"] &
    meta_first["split_final"].eq("test").to_numpy()
)
if final_base_train.sum() < 20 or final_test_common.sum() < 5:
    raise RuntimeError('Insufficient samples for the first-only holdout.')

fb_first_final = FeatureBuilder(
    n_pca_current=min(8, int(final_base_train.sum()) - 1),
    n_pca_trend=min(8, int(final_base_train.sum()) - 1),
).fit(X_robust[final_base_train], X_elev_robust[final_base_train])
F_first_final = fb_first_final.transform(X_robust, X_elev_robust, meta_first)

first_final_rows = []
first_final_predictions = {}
for _, selected in selected_first_models.iterrows():
    cohort_name, model_name = selected["cohort"], selected["model"]
    train_mask = cv_candidate_masks[cohort_name] & meta_first["split_final"].isin(["train", "val"]).to_numpy()
    train_pos = np.flatnonzero(train_mask)
    model = first_models[model_name]
    if model is None:
        pred_all = np.full(len(y_robust), np.median(y_robust[train_pos]), dtype=float)
    else:
        fitted = clone(model)
        fitted.fit(F_first_final.iloc[train_pos], y_robust[train_pos])
        pred_all = fitted.predict(F_first_final)
    first_final_predictions[cohort_name] = pred_all
    for eval_name, eval_mask in [
        ("original_first_common", final_test_common),
        ("new_first_robust", final_test_new),
    ]:
        if eval_mask.sum() == 0:
            continue
        first_final_rows.append({
            "cohort": cohort_name,
            "selected_model": model_name,
            "cv_MAE_mean": float(selected["MAE_mean"]),
            "eval_set": eval_name,
            "n_train": len(train_pos),
            "n_eval": int(eval_mask.sum()),
            **expanded_metrics(y_robust[eval_mask], pred_all[eval_mask]),
        })
first_only_final_df = pd.DataFrame(first_final_rows)
print('Final first-only holdout')
display(first_only_final_df)

test_pos_first = np.flatnonzero(final_test_common)
rng_first = np.random.default_rng(RANDOM_SEED + 2026)
bootstrap_comparisons = [
    ("original_robust_minus_original_strict", "original_first_robust", "original_first_strict"),
    ("expanded_robust_minus_original_robust", "expanded_first_robust", "original_first_robust"),
    ("expanded_robust_minus_original_strict", "expanded_first_robust", "original_first_strict"),
]
bootstrap_first_rows = []
for comparison_name, candidate_name, reference_name in bootstrap_comparisons:
    candidate_pred = first_final_predictions[candidate_name]
    reference_pred = first_final_predictions[reference_name]
    deltas = []
    for _ in range(3000):
        sampled = rng_first.choice(test_pos_first, size=len(test_pos_first), replace=True)
        deltas.append(
            np.mean(np.abs(y_robust[sampled] - candidate_pred[sampled])) -
            np.mean(np.abs(y_robust[sampled] - reference_pred[sampled]))
        )
    deltas = np.asarray(deltas)
    bootstrap_first_rows.append({
        "comparison": comparison_name,
        "delta_MAE": float(deltas.mean()),
        "ci95_low": float(np.quantile(deltas, .025)),
        "ci95_high": float(np.quantile(deltas, .975)),
        "probability_candidate_better_%": float((deltas < 0).mean() * 100),
        "n_test": len(test_pos_first),
    })
first_only_bootstrap_df = pd.DataFrame(bootstrap_first_rows)
display(first_only_bootstrap_df)


In [ ]:
first_only_gru_rows = []
if RUN_FIRST_ONLY_GRU and "TF_AVAILABLE" in globals() and TF_AVAILABLE:
    gru_val_mask = (
        first_eval_masks["original_first_common"] &
        meta_first["split_final"].eq("val").to_numpy()
    )
    gru_test_mask = final_test_common
    gru_new_test_mask = final_test_new
    gru_base_train = (
        first_cohort_masks["original_first_strict_window"] &
        meta_first["split_final"].eq("train").to_numpy()
    )
    seq_pca_first = SequencePCA(n_components=min(16, int(gru_base_train.sum()) - 1)).fit(X_robust[gru_base_train])
    Z_first = seq_pca_first.transform(X_robust)
    az_first = np.deg2rad(pd.to_numeric(meta_first["Azimuth"], errors="coerce").fillna(0).to_numpy())
    context_first_raw = np.column_stack([
        X_elev_robust.ravel(), np.sin(az_first), np.cos(az_first)
    ]).astype("float32")
    context_first_scaler = StandardScaler().fit(context_first_raw[gru_base_train])
    C_first = context_first_scaler.transform(context_first_raw).astype("float32")

    for model_offset, (cohort_name, cohort_mask) in enumerate({
        "original_first_strict": first_cohort_masks["original_first_strict_window"],
        "original_first_robust": first_cohort_masks["original_first_robust_window"],
        "expanded_first_robust": first_cohort_masks["expanded_first_robust_window"],
    }.items()):
        train_mask = cohort_mask & meta_first["split_final"].eq("train").to_numpy()
        y_scaler_first = StandardScaler().fit(y_robust[train_mask].reshape(-1, 1))
        ys_train = y_scaler_first.transform(y_robust[train_mask].reshape(-1, 1)).astype("float32")
        ys_val = y_scaler_first.transform(y_robust[gru_val_mask].reshape(-1, 1)).astype("float32")

        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_SEED + 100 + model_offset)
        seq_in = keras.Input(shape=(SEQ_LEN, Z_first.shape[-1]))
        ctx_in = keras.Input(shape=(3,))
        hs = layers.GRU(32, dropout=.25)(seq_in)
        hs = layers.Dense(16, activation="gelu")(hs)
        hc = layers.Dense(8, activation="gelu")(ctx_in)
        h = layers.Concatenate()([hs, hc])
        h = layers.Dropout(.35)(h)
        h = layers.Dense(16, activation="gelu")(h)
        out = layers.Dense(1, dtype="float32")(h)
        model = keras.Model([seq_in, ctx_in], out)
        model.compile(optimizer=keras.optimizers.AdamW(5e-4, weight_decay=1e-3), loss="mae")
        history = model.fit(
            [Z_first[train_mask], C_first[train_mask]], ys_train,
            validation_data=([Z_first[gru_val_mask], C_first[gru_val_mask]], ys_val),
            epochs=300, batch_size=16, verbose=0,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True),
                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=.5, min_lr=1e-6),
            ],
        )
        pred_all = y_scaler_first.inverse_transform(model.predict([Z_first, C_first], verbose=0)).ravel()
        for eval_name, eval_mask in [("original_first_common", gru_test_mask), ("new_first_robust", gru_new_test_mask)]:
            if eval_mask.sum() == 0:
                continue
            first_only_gru_rows.append({
                "cohort": cohort_name,
                "eval_set": eval_name,
                "n_train": int(train_mask.sum()),
                "n_eval": int(eval_mask.sum()),
                "epochs": len(history.history["loss"]),
                **expanded_metrics(y_robust[eval_mask], pred_all[eval_mask]),
            })
    first_only_gru_df = pd.DataFrame(first_only_gru_rows)
    display(first_only_gru_df)
else:
    print('First-only GRU not run.')


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

recovery_values = [len(meta_exp), len(meta_robust)]
axs[0, 0].bar(["strict", "robust"], recovery_values, color=["0.5", "tab:green"])
axs[0, 0].set_title('Recovered thermal windows')
axs[0, 0].set_ylabel('samples')
axs[0, 0].grid(True, axis="y", alpha=.3)

for cohort_name, sub in blocked_cv_summary_df.groupby("cohort"):
    axs[0, 1].errorbar(sub["model"], sub["MAE_mean"], yerr=sub["MAE_std"].fillna(0), marker="o", label=cohort_name)
axs[0, 1].set_title('First-only temporal cross-validation')
axs[0, 1].set_ylabel("MAE [arcsec]")
axs[0, 1].tick_params(axis="x", rotation=25)
axs[0, 1].legend(); axs[0, 1].grid(True, alpha=.3)

holdout_plot = first_only_final_df[first_only_final_df["eval_set"] == "original_first_common"]
axs[1, 0].bar(holdout_plot["cohort"], holdout_plot["MAE"], color=["tab:blue", "tab:orange"])
axs[1, 0].axhline(SPEC_ARCSEC, color="tab:green", linestyle="--")
axs[1, 0].set_title('Common final first-only holdout')
axs[1, 0].set_ylabel("MAE [arcsec]")
axs[1, 0].tick_params(axis="x", rotation=15)
axs[1, 0].grid(True, axis="y", alpha=.3)

axs[1, 1].bar(sequence_position_df["position_band"], sequence_position_df["median_abs_target"], color="tab:purple")
axs[1, 1].set_title('Median |XEL| by sequence position')
axs[1, 1].set_ylabel("arcsec")
axs[1, 1].grid(True, axis="y", alpha=.3)

plt.tight_layout(); plt.show()

thermal_recovery_df.to_csv("thermal_recovery_summary.csv", index=False)
meta_robust.to_csv("robust_window_quality_audit.csv", index=False)
session_gap_sensitivity_df.to_csv("session_gap_sensitivity.csv", index=False)
sequence_position_df.to_csv("sequence_position_summary.csv", index=False)
cumulative_signature_df.to_csv("cumulative_signature_summary.csv", index=False)
near_pair_stability_df.to_csv("near_pair_stability.csv", index=False)
first_cohort_audit_df.to_csv("first_only_cohort_audit.csv", index=False)
blocked_cv_df.to_csv("first_only_blocked_cv_folds.csv", index=False)
blocked_cv_summary_df.to_csv("first_only_blocked_cv_summary.csv", index=False)
first_only_final_df.to_csv("first_only_final_holdout.csv", index=False)
first_only_bootstrap_df.to_csv("first_only_bootstrap.csv", index=False)
if "first_only_gru_df" in globals():
    first_only_gru_df.to_csv("first_only_gru_results.csv", index=False)

print('New-test exports completed.')
